# SRF Qubit + Cavity Calibration

Calibration notebook for a fixed-frequency transmon coupled to two SRF cavities (Alice and Bob).

**Compatible hardware systems:**
- **OPX1000 / MW-FEM** - digital upconversion, no external mixers
- **OPX+ / Octave** - analog IQ upconversion via Octave

Hardware-specific sections are clearly labelled. Run only the sub-section that matches your system.

## 0. Preamble: run this first every session

In [1]:
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path
from qualibrate_config.core.project.switch import switch_project

config_path = get_qualibrate_config_path()
config = get_qualibrate_config(config_path)
print(f"Current project: {config.project}")

# -- Edit this to match your Qualibrate project name --------------------------
desired_project = "dr3_run11_srf_qubit_2"   # e.g. "dr3_run9_srf_qubit_1"
# -----------------------------------------------------------------------------

if config.project != desired_project:
    switch_project(config_path, desired_project)
    config = get_qualibrate_config(config_path)
    print(f"Switched to project: {config.project}")
else:
    print(f"Project already set to '{desired_project}'")

print(f"Storage location: {config.storage.location}")

Current project: dr3_run11_srf_qubit_2
Project already set to 'dr3_run11_srf_qubit_2'
Storage location: D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage


In [2]:
%matplotlib widget

import sys
from quam_config import Quam, TemporaryCalibrationData


2026-07-16 12:42:39,291 - qm - INFO     - Starting session: cbcaea30-aaff-4007-865d-0b9c7b1bbd08


In [ ]:
import logging
# Remove rotating file handler to avoid Windows lock conflict
qualibrate_logger = logging.getLogger("qualibrate")
for h in qualibrate_logger.handlers[:]:
    if hasattr(h, 'baseFilename'):  # RotatingFileHandler
        qualibrate_logger.removeHandler(h)


## 1. Create QUAM state

Run **once** to build `state.json` and `wiring.json` in `quam_state/`. Skip if the files already exist.

> **Run only the sub-section matching your hardware.**

### 1a. OPX1000 / MW-FEM

Uses a single MW-FEM (slot 1):
- port 1 (out + in 1): shared readout feed-line
- port 2: qubit XY drive
- port 3: f0g1 sideband drive
- port 4 (shareable): Alice + Bob cavity modes (used sequentially)

In [ ]:
import matplotlib.pyplot as plt
from qualang_tools.wirer.wirer.channel_specs import *
from qualang_tools.wirer import Instruments, Connectivity, allocate_wiring, visualize
from quam_builder.builder.qop_connectivity import build_quam_wiring
from quam_builder.builder.superconducting import build_quam
from quam_config import Quam

host_ip      = "192.168.3.50"
port         = None
cluster_name = "Cluster_OPX1000"

##########################################################################
# USER PARAMETERS
# Single MW-FEM (slot 1):
#   port 1 -> readout resonator (out + in 1, shared feed-line)
#   port 2 -> qubit XY drive (auto-allocated)
#   port 3 -> f0g1 sideband drive  (wired manually in populate cell)
#   port 4 -> Alice + Bob cavity   (wired manually in populate cell, shareable)
##########################################################################
qubits = [1]

instruments = Instruments()
instruments.add_mw_fem(controller=1, slots=[1])

# Shared readout feed-line: all qubits multiplex on MW-FEM slot 1 port 1 (FDM)
res_ch   = mw_fem_spec(con=1, slot=1, in_port=1, out_port=1)
# Individual XY drive ports: auto-allocated on MW-FEM slot 1
drive_ch = mw_fem_spec(con=1, slot=1, in_port=None, out_port=None)

connectivity = Connectivity()
connectivity.add_resonator_line(qubits=qubits, constraints=res_ch)       # list -> shared feed-line
connectivity.add_qubit_drive_lines(qubits=qubits, constraints=drive_ch)  # list -> individual ports
# connectivity.add_qubit_flux_lines(qubits=qubits)  # uncomment for tunable qubits
allocate_wiring(connectivity, instruments)

visualize(connectivity.elements, available_channels=instruments.available_channels)
plt.show(block=False)

user_input = input("Do you want to save the updated QUAM? (y/n)")
if user_input.lower() == "y":
    machine = Quam()
    build_quam_wiring(connectivity, host_ip, cluster_name, machine)
    machine = Quam.load()
    build_quam(machine)


Visualization exported to: qm-instrument_config_2026-06-29_21-26-05.html


### 1b. OPX+ / Octave

Uses an OPX+ controller with a single Octave. RF outputs:
- RF1 -> readout resonator
- RF2 -> f0g1 sideband drive
- RF3 -> qubit XY drive
- RF4 -> cavity drive (alice + bob shared)

All Octave RF outputs use the internal LO. Digital trigger mapping:
RF out N -> OPX+ digital output con1/(2N-1), i.e. RF1->1, RF2->3, RF3->5, RF4->7, RF5->9.


In [ ]:
import matplotlib.pyplot as plt
from qualang_tools.wirer.wirer.channel_specs import (
    octave_spec,
    opx_iq_octave_spec,
    opx_dig_spec,
    ChannelSpecOctaveDigital,
)
from qualang_tools.wirer import Instruments, Connectivity, allocate_wiring, visualize
from quam_builder.builder.qop_connectivity import build_quam_wiring
from quam_builder.builder.superconducting import build_quam
from quam_config import Quam

host_ip             = "192.168.6.50"
port                = None
cluster_name        = "Cluster_1"
calibration_db_path = None

##########################################################################
# USER PARAMETERS
# 6-qubit OPX+/Octave setup, fully frequency-multiplexed (FDM) on both
# the readout and XY drive lines.
#
# All 6 resonators share Octave RF1/RFin1.
# All 6 qubit XY drives share OPX+ I/Q ports 5/6, digital trigger 5,
# up-converted via Octave RF3 (Octave digital input 3 to match).
# RF2 is reserved for the f0g1 sideband drive, wired in the populate cell
# (OPX+ ports 3/4, digital trigger 3).
#
# Octave digital trigger convention: RF out N -> OPX+ digital output
# con1/(2N-1), i.e. RF1->1, RF2->3, RF3->5, RF4->7, RF5->9.
##########################################################################
qubits = [0, 1, 2, 3, 4, 5]

instruments = Instruments()
instruments.add_opx_plus(controllers=[1])
instruments.add_octave(indices=1)

# Shared readout: all 6 qubits on Octave RF out 1 / RF in 1 (FDM)
res_ch = octave_spec(index=1, rf_out=1, rf_in=1)
# Shared XY drive: all 6 qubits on OPX+ ports 5/6 (I/Q) + digital trigger 5,
# up-converted via Octave RF3 (FDM). The Octave digital input is pinned to 5
# to match the OPX+ digital trigger.
xy_ch  = (
    opx_iq_octave_spec(out_port_i=5, out_port_q=6, octave_index=1, rf_out=3)
    & opx_dig_spec(out_port=5)
    & ChannelSpecOctaveDigital(con=1, in_port=3)
)

connectivity = Connectivity()

# Resonator line: a single shared spec covering all 6 qubits, allocated once.
connectivity.add_resonator_line(qubits=qubits, triggered=True, constraints=res_ch)
allocate_wiring(connectivity, instruments)

# XY drive lines: add_qubit_drive_lines always creates one spec per qubit, so
# allocate each qubit individually against the SAME pinned xy_ch and free the
# channel afterwards (block_used_channels=False) so every qubit reuses it.
for q in qubits:
    connectivity.add_qubit_drive_lines(qubits=q, triggered=True, constraints=xy_ch)
    allocate_wiring(connectivity, instruments, block_used_channels=False)

# connectivity.add_qubit_flux_lines(qubits=qubits)  # uncomment for tunable qubits

visualize(connectivity.elements, available_channels=instruments.available_channels)
plt.show(block=False)

user_input = input("Save QUAM? (y/n) ").strip().lower()
if user_input == "y":
    machine = Quam()
    build_quam_wiring(connectivity, host_ip, cluster_name, machine)
    machine = Quam.load()
    build_quam(machine, calibration_db_path)

    for octave in machine.octaves.values():
        for rf_out in octave.RF_outputs.values():
            if rf_out.channel is not None:
                rf_out.output_mode = "triggered"

    machine.save()
    print("Done. Add EF and cavity channels to state.json, then populate.")
else:
    print("Skipped.")


Visualization exported to: qm-instrument_config_2026-06-29_21-15-25.html
Skipped.


## 2. Populate QUAM with initial values

Edit the **USER PARAMETERS** section to match chip specs, then run the appropriate cell.

> **Run only the sub-section matching your hardware.**

### 2a. OPX1000 / MW-FEM

In [ ]:
"""
Populate the QUAM state (OPX1000 / MW-FEM) with initial hardware parameters.

Supports any number of qubits sharing one readout feed-line and having
individual XY drive ports. Extend the per-qubit arrays below as needed.

Hardware routing (con1, FEM 1):
    port 1  (shared)  -> readout resonator (out + in 1, FDM)
    port 2            -> qubit XY drive
    port 3            -> f0g1 sideband drive
    port 4  (shared)  -> Alice + Bob cavity modes (shareable, sequential use)
"""

import json
from pprint import pprint
import numpy as np
from qualang_tools.units import unit
from quam.components.pulses import SquarePulse, DragCosinePulse, DragGaussianPulse
from quam_builder.architecture.superconducting.components.pulses import SineSqRampPulse
from quam_builder.architecture.superconducting.components.xy_drive import XYDriveMW
from quam.components.channels import StickyChannelAddon
from quam_builder.architecture.superconducting.components.twpa import TWPA
from quam_builder.architecture.superconducting.cavity.cavity import Cavity
from quam_builder.architecture.superconducting.cavity.cavity_mode import CavityMode
from quam_builder.builder.superconducting.pulses import add_DragGaussian_pulses
from quam_config import Quam, TemporaryCalibrationData
from quam_builder.architecture.superconducting.qubit_pair import CavityTransmonPair
from quam_builder.architecture.superconducting.qubit_pair.cavity_transmon_pair import SidebandTransition

u = unit(coerce_to_integer=True)


def get_band(freq: float) -> int:
    if 50e6 <= freq < 5.5e9:    return 1
    elif 4.5e9 <= freq < 7.5e9: return 2
    elif 6.5e9 <= freq <= 10.5e9: return 3
    else: raise ValueError(f"Frequency {freq} Hz outside MW-FEM range")


def get_full_scale_power_dBm_and_amplitude(desired_power: float, max_amplitude: float = 0.5):
    allowed = [-11, -8, -5, -2, 1, 4, 7, 10, 13, 16]
    resulting = desired_power - 20 * np.log10(max_amplitude)
    if resulting < 0:
        fsp = min(allowed, key=lambda x: abs(x - max(resulting + 3, -11)))
    else:
        fsp = min(allowed, key=lambda x: abs(x - min(resulting + 3, 16)))
    amp = 10 ** ((desired_power - fsp) / 20)
    if not (-11 <= fsp <= 16 and -1 <= amp <= 1):
        raise ValueError(f"Power outside spec: fsp={fsp} dBm, amp={amp:.4f}")
    return fsp, amp


machine = Quam.load()
n_qubits = len(machine.qubits)

##########################################################################
# USER PARAMETERS -- edit to match chip specs
# For N qubits: supply N-element numpy arrays for rr_freq, xy_freq,
# xy_LO, and anharmonicity (one entry per qubit, in machine order).
##########################################################################
CAVITY_ID      = "c1"
QUBIT_FEM_ID   = 1   # FEM slot for resonator + qubit XY (from wiring.json)
CAVITY_FEM_ID  = 1   # same FEM as qubit
F0G1_PORT      = 3
ALICE_PORT     = 4
BOB_PORT       = 4

# -- Readout resonator (shared feed-line, one upconverter_frequency) ----------
rr_freq       = np.array([7.374e9])   # Hz, one per qubit  e.g. [7.50e9, 7.52e9]
rr_LO         = 7.450e9               # Hz, shared upconverter_frequency (FEM 1 port 1)
readout_power = -10                   # dBm

# -- Qubit XY drives (one entry per qubit) ------------------------------------
xy_freq       = np.array([4.049e9])   # Hz, one per qubit
xy_LO         = np.array([4.100e9])   # Hz, per-qubit upconverter_frequency
anharmonicity = np.array([-200e6])    # Hz, per-qubit anharmonicity
drive_power   = -10                   # dBm

# -- Cavity drives (alice + bob share upconverter_frequency) ------------------
alice_freq    = 6.000e9
alice_LO      = 5.900e9
alice_power   = 10
bob_freq      = 6.200e9
bob_power     = 10

# -- f0g1 sideband drive -------------------------------------------------------
alice_f0g1_freq              = 3.25e9
alice_f0g1_LO                = 3.0e9
alice_f0g1_power             = 0
alice_f0g1_saturation_length_ns = 20000
alice_f0g1_pi_length_ns      = 1000   # flat-top duration [ns]; total pulse = pi_length + 2*ramp_len
alice_f0g1_amp               = 0.4

# -- Sideband cooling calibration ----------------------------------------------
max_fock_level = 4        # calibrate transitions f0g1 through f{max-1}gmax
chi_guess_hz   = -2e6     # rough cavity-qubit chi [Hz]; negative sign typical
sideband_ramp_len_ns = 40    # Gaussian ramp duration [ns] each side of the flat top

# -- Timing -------------------------------------------------------------------
tof_ns                      = 224
resonator_depletion_time_ns = 10000
thermalization_time_factor  = 5
sigma_time_factor           = 5

# -- Pulse defaults -----------------------------------------------------------
readout_length_ns            = 8000
saturation_length_ns         = 20000
x180_length_ns               = 1000
gaussian_sigma_ns            = x180_length_ns // 5
drag_alpha                   = 0.0
drag_detuning                = 0.0
selective_x180_length_ns     = 10000
ef_x180_amplitude            = 0.1
displacement_length_ns       = 1000
displacement_sigma_ns        = displacement_length_ns // 5
displacement_initial_amplitude_V = 0.001


# -- TWPA pump -----------------------------------------------------------------
TWPA_ID                  = "twpa1"
TWPA_FEM_ID              = 1    # EDIT: FEM slot for TWPA pump
TWPA_PUMP_PORT           = 5    # EDIT: port on that FEM (ports 1-4 are rr/xy/sideband/cavity)
twpa_pump_freq           = 6.500e9  # Hz -- EDIT: datasheet-guess pump frequency
twpa_pump_LO             = 6.700e9  # Hz -- EDIT: upconverter_frequency for the pump port
twpa_pump_power_dbm      = -15.0    # dBm -- EDIT: initial pump power guess
twpa_pump_full_scale_dbm = 4        # dBm -- EDIT: port full_scale_power_dbm
pump_sticky_duration_ns  = 1000     # ns
# -- T1 estimates -------------------------------------------------------------
T1        = 200e-6
cavity_T1 = 100e-6
##########################################################################

assert len(rr_freq) == n_qubits,      f"rr_freq: expected {n_qubits} entries, got {len(rr_freq)}"
assert len(xy_freq) == n_qubits,      f"xy_freq: expected {n_qubits} entries, got {len(xy_freq)}"
assert len(xy_LO)   == n_qubits,      f"xy_LO: expected {n_qubits} entries, got {len(xy_LO)}"
assert len(anharmonicity) == n_qubits, f"anharmonicity: expected {n_qubits} entries, got {len(anharmonicity)}"
assert np.all(np.abs(rr_freq - rr_LO) < 400e6), "Resonator IF out of range"
assert np.all(np.abs(xy_freq - xy_LO) < 400e6), "XY IF out of range"
assert abs(alice_freq - alice_LO) < 400e6,       "Alice IF out of range"
assert abs(bob_freq - alice_LO)   < 400e6,       "Bob IF out of range (must share LO with Alice)"
assert abs(alice_f0g1_freq - alice_f0g1_LO) < 400e6, "f0g1 IF out of range"

# -- Cavity MW-FEM ports (f0g1, alice, bob) -----------------------------------
for port_id, upconv_freq, power_dbm in [
    (F0G1_PORT,  alice_f0g1_LO, alice_f0g1_power),
    (ALICE_PORT, alice_LO,      alice_power),
    (BOB_PORT,   alice_LO,      bob_power),
]:
    fsp, _ = get_full_scale_power_dBm_and_amplitude(power_dbm)
    port = machine.ports.get_mw_output(
        "con1", CAVITY_FEM_ID, port_id, create=True,
        upconverter_frequency=upconv_freq, band=get_band(upconv_freq),
        full_scale_power_dbm=fsp, delay=0, shareable=True,
    )
    port.upconverter_frequency = upconv_freq
    port.band                  = get_band(upconv_freq)
    port.full_scale_power_dbm  = fsp
    port.shareable             = True

# -- Resonator: shared feed-line, per-qubit IF frequencies --------------------
# max_amplitude is divided by n_qubits so the sum of all readout tones
# stays within the DAC full-scale limit.
rr_fsp, rr_amp = get_full_scale_power_dBm_and_amplitude(
    readout_power, max_amplitude=0.5 / max(1, n_qubits))
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.resonator.f_01           = float(rr_freq[k])
    qubit.resonator.RF_frequency   = float(rr_freq[k])
    qubit.resonator.frequency_bare = float(rr_freq[k])
    qubit.resonator.time_of_flight = tof_ns
    qubit.resonator.smearing       = 0
    qubit.resonator.depletion_time = resonator_depletion_time_ns
    # All qubits share the same port -> same upconverter_frequency
    qubit.resonator.opx_output.upconverter_frequency = rr_LO
    qubit.resonator.opx_output.band                  = get_band(rr_LO)
    qubit.resonator.opx_output.full_scale_power_dbm  = rr_fsp
    qubit.resonator.opx_input.band                   = get_band(rr_LO)
    ro_op = qubit.resonator.operations.get("readout")
    if ro_op is not None:
        ro_op.amplitude = rr_amp
        ro_op.length    = readout_length_ns

# -- Qubit XY: per-qubit frequency and upconverter ----------------------------
xy_fsp, xy_amp = get_full_scale_power_dBm_and_amplitude(drive_power)
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.f_01                                = float(xy_freq[k])
    qubit.xy.RF_frequency                     = float(xy_freq[k])
    qubit.xy.opx_output.upconverter_frequency = float(xy_LO[k])
    qubit.xy.opx_output.band                  = get_band(float(xy_LO[k]))
    qubit.xy.opx_output.full_scale_power_dbm  = xy_fsp
    # Port 2 (XY) is physically coupled to port 3 (f0g1 sideband, shareable=True);
    # QM requires coupled ports to share the same shareable value.
    qubit.xy.opx_output.shareable             = True
    qubit.T1                                  = T1
    qubit.anharmonicity                       = int(anharmonicity[k])
    qubit.grid_location                       = f"{k},0"
    qubit.thermalization_time_factor          = thermalization_time_factor
    qubit.sigma_time_factor                   = sigma_time_factor
    sat_op = qubit.xy.operations.get("saturation")
    if sat_op is not None:
        sat_op.amplitude      = 0.3
        sat_op.length         = saturation_length_ns
        sat_op.digital_marker = "ON"
    add_DragGaussian_pulses(qubit, xy_amp, x180_length_ns, gaussian_sigma_ns,
                            drag_alpha, drag_detuning, int(anharmonicity[k]),
                            digital_marker="ON")

# -- EF and selective pulses --------------------------------------------------
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    xy = qubit.xy
    anharm = int(anharmonicity[k])
    xy.operations["EF_x180"] = DragGaussianPulse(
        length="#../x180/length", amplitude=ef_x180_amplitude,
        sigma="#../x180/sigma", alpha=0.0, anharmonicity=anharm,
        detuning=0.0, subtracted=True, axis_angle=0, digital_marker="ON",
    )
    xy.operations["EF_x90"] = DragGaussianPulse(
        length="#../EF_x180/length", amplitude=ef_x180_amplitude / 2,
        sigma="#../EF_x180/sigma", alpha="#../EF_x180/alpha",
        anharmonicity="#../EF_x180/anharmonicity",
        detuning="#../EF_x180/detuning", subtracted="#../EF_x180/subtracted",
        axis_angle=0, digital_marker="#../EF_x180/digital_marker",
    )
    sel_amp = xy_amp * (x180_length_ns / selective_x180_length_ns)
    xy.operations["selective_x180"] = DragGaussianPulse(
        length=selective_x180_length_ns, amplitude=sel_amp,
        sigma=selective_x180_length_ns // 5, alpha=0.0, anharmonicity=anharm,
        detuning=0.0, subtracted=True, axis_angle=0, digital_marker="ON",
    )
    sel_ef_amp = ef_x180_amplitude * (x180_length_ns / selective_x180_length_ns)
    xy.operations["selective_EF_x180"] = DragGaussianPulse(
        length="#../selective_x180/length", amplitude=sel_ef_amp,
        sigma="#../selective_x180/sigma", alpha=0.0, anharmonicity=anharm,
        detuning=0.0, subtracted=True, axis_angle=0, digital_marker="ON",
    )

# -- Cavity object (alice + bob) ----------------------------------------------
def _make_cavity_drive(port_id: int, freq: float, mode_id: str) -> XYDriveMW:
    return XYDriveMW(
        id=mode_id,
        opx_output=f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{port_id}",
        RF_frequency=freq,
    )

if CAVITY_ID not in machine.cavities:
    alice_mode = CavityMode(id="alice", cavity_mode_drive=_make_cavity_drive(ALICE_PORT, alice_freq, "alice_drive"))
    alice_mode.T1 = cavity_T1
    bob_mode = CavityMode(id="bob", cavity_mode_drive=_make_cavity_drive(BOB_PORT, bob_freq, "bob_drive"))
    bob_mode.T1 = cavity_T1
    machine.cavities[CAVITY_ID] = Cavity(id=CAVITY_ID, alice=alice_mode, bob=bob_mode)
    print(f"  Created cavity '{CAVITY_ID}' with alice and bob modes.")
else:
    for mode_name in ("alice", "bob"):
        mode = getattr(machine.cavities[CAVITY_ID], mode_name, None)
        if mode is not None:
            mode.T1 = cavity_T1

cav_fsp, cav_amp = get_full_scale_power_dBm_and_amplitude(alice_power)
for cav_name, cavity in machine.cavities.items():
    for mode_name, freq, port_id in (("alice", alice_freq, ALICE_PORT), ("bob", bob_freq, BOB_PORT)):
        mode = getattr(cavity, mode_name, None)
        if mode is None or mode.cavity_mode_drive is None:
            continue
        mode.cavity_mode_drive.id           = f"{mode_name}_drive"
        mode.cavity_mode_drive.opx_output   = f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{port_id}"
        mode.cavity_mode_drive.RF_frequency = freq
        mode.cavity_mode_drive.operations["saturation"] = SquarePulse(
            length=readout_length_ns, amplitude=cav_amp, digital_marker="ON")
        mode.cavity_mode_drive.operations["displacement"] = DragGaussianPulse(
            length=displacement_length_ns, amplitude=displacement_initial_amplitude_V,
            sigma=displacement_sigma_ns, alpha=0.0, anharmonicity=0,
            detuning=0.0, axis_angle=0, digital_marker="ON",
        )

for cav_name, cavity in machine.cavities.items():
    for mode_name, factor in (("alice", 3), ("bob", 5)):
        mode = getattr(cavity, mode_name, None)
        if mode is not None:
            mode.thermalization_time_factor = factor

# -- CavityTransmonPair (with f0g1 sideband_drive) ----------------------------
for q_name in machine.qubits:
    for mode_name in ("alice", "bob"):
        pair_key = f"{q_name}_{mode_name}"
        if pair_key not in machine.cavity_transmon_pairs:
            machine.cavity_transmon_pairs[pair_key] = CavityTransmonPair(
                qubit_name=q_name, cavity_mode_name=mode_name)
            print(f"  Created CavityTransmonPair '{pair_key}'")
    for pk in (f"{q_name}_alice", f"{q_name}_bob"):
        p = machine.cavity_transmon_pairs.get(pk)
        if p is not None and not hasattr(p, "parity_time"):
            p.parity_time = None

    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is not None and alice_pair.sideband_drive is None:
        f0g1_drive = XYDriveMW(
            id=f"{q_name}_alice",
            opx_output=f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{F0G1_PORT}",
            RF_frequency=alice_f0g1_freq,
        )
        f0g1_drive.operations["saturation"] = SquarePulse(
            length=alice_f0g1_saturation_length_ns, amplitude=alice_f0g1_amp, digital_marker="ON")
        alice_pair.sideband_drive = f0g1_drive
        print(f"  Created sideband_drive for '{q_name}_alice'.")
    elif alice_pair is not None and alice_pair.sideband_drive is not None:
        alice_pair.sideband_drive.id           = f"{q_name}_alice"
        alice_pair.sideband_drive.opx_output   = f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{F0G1_PORT}"
        alice_pair.sideband_drive.RF_frequency = alice_f0g1_freq

# -- Temporary calibration state ----------------------------------------------
if machine.temp_calibration is None:
    machine.temp_calibration = {}
for k, q_name in enumerate(machine.qubits):
    if q_name not in machine.temp_calibration:
        machine.temp_calibration[q_name] = TemporaryCalibrationData(
            initial_resonator_f01=float(rr_freq[k]),
            initial_resonator_RF_frequency=float(rr_freq[k]),
        )
    else:
        tc = machine.temp_calibration[q_name]
        tc.initial_resonator_f01          = float(rr_freq[k])
        tc.initial_resonator_RF_frequency = float(rr_freq[k])

# -- Multi-level sideband pulses (sine-square ramp_up / square / ramp_down) ----
# Three separate operations replace the old monolithic FlatTopGaussianPulse:
#   sideband_ramp_up   – fixed sine² rising ramp (length = sideband_ramp_len_ns)
#   sideband_square    – constant pulse swept by duration= in QUA (length = initial flat-top guess)
#   sideband_ramp_down – fixed sine² falling ramp, references ramp_up length/amplitude
# pi_flat_top_length_ns in SidebandTransition stores the calibrated flat-portion pi duration.
for q_name in machine.qubits:
    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is None or alice_pair.sideband_drive is None:
        continue
    sd = alice_pair.sideband_drive

    # Remove legacy monolithic flat-top pulse and per-transition operations if present
    for _legacy_op in (
        ["sideband_flat_top"]
        + [f"f{_k}g{_k+1}_pi" for _k in range(max_fock_level)]
    ):
        sd.operations.pop(_legacy_op, None)

    # Three-part pulse: ramp_up, flat (constant, variable duration), ramp_down
    if "sideband_square" not in sd.operations:
        sd.operations["sideband_square"] = SquarePulse(
            length=alice_f0g1_pi_length_ns,
            amplitude=alice_f0g1_amp,
            axis_angle=0.0,
            digital_marker="ON",
        )
    else:
        sd.operations["sideband_square"].length    = alice_f0g1_pi_length_ns
        sd.operations["sideband_square"].amplitude = alice_f0g1_amp

    if "sideband_ramp_up" not in sd.operations:
        sd.operations["sideband_ramp_up"] = SineSqRampPulse(
            length=sideband_ramp_len_ns,
            amplitude="#../sideband_square/amplitude",
            axis_angle="#../sideband_square/axis_angle",
            direction="up",
            digital_marker="ON",
        )
    else:
        sd.operations["sideband_ramp_up"].length = sideband_ramp_len_ns

    if "sideband_ramp_down" not in sd.operations:
        sd.operations["sideband_ramp_down"] = SineSqRampPulse(
            length="#../sideband_ramp_up/length",
            amplitude="#../sideband_square/amplitude",
            axis_angle="#../sideband_square/axis_angle",
            direction="down",
            digital_marker="ON",
        )


    for _k in range(max_fock_level):
        _tkey = f"f{_k}g{_k+1}"
        _rf_guess = alice_f0g1_freq - _k * abs(chi_guess_hz)

        # Initialise typed SidebandTransition (preserves existing calibrated values)
        if _tkey not in alice_pair.transitions:
            alice_pair.transitions[_tkey] = SidebandTransition(
                RF_frequency=_rf_guess,
                pi_flat_top_length_ns=None,
            )
        else:
            tr = alice_pair.transitions[_tkey]
            if tr.RF_frequency is None:
                tr.RF_frequency = _rf_guess

# -- TWPA pump: sticky 'pump' + non-sticky 'pump_' elements ------------------
twpa_fsp, twpa_amp = get_full_scale_power_dBm_and_amplitude(twpa_pump_power_dbm)
twpa_port = machine.ports.get_mw_output(
    "con1", TWPA_FEM_ID, TWPA_PUMP_PORT, create=True,
    upconverter_frequency=twpa_pump_LO, band=get_band(twpa_pump_LO),
    full_scale_power_dbm=twpa_fsp, delay=0, shareable=True,
)
twpa_port.upconverter_frequency = twpa_pump_LO
twpa_port.band                  = get_band(twpa_pump_LO)
twpa_port.full_scale_power_dbm  = twpa_fsp
twpa_port.shareable             = True


def _make_twpa_pump(sticky: bool) -> XYDriveMW:
    drive = XYDriveMW(
        id=f"{TWPA_ID}_pump" + ("" if sticky else "_"),
        opx_output=f"#/ports/mw_outputs/con1/{TWPA_FEM_ID}/{TWPA_PUMP_PORT}",
        RF_frequency="#../pump_frequency",
        sticky=StickyChannelAddon(duration=pump_sticky_duration_ns, digital=False) if sticky else None,
    )
    drive.operations["pump"] = SquarePulse(length=1000, amplitude=twpa_amp, digital_marker="ON")
    return drive


if TWPA_ID not in machine.twpas:
    machine.twpas[TWPA_ID] = TWPA(
        id=TWPA_ID,
        pump=_make_twpa_pump(sticky=True),
        pump_=_make_twpa_pump(sticky=False),
        pump_frequency=twpa_pump_freq,
        pump_amplitude=twpa_amp,
        qubits=list(machine.qubits.keys()),
    )
    print(f"  Created TWPA '{TWPA_ID}' on FEM {TWPA_FEM_ID} port {TWPA_PUMP_PORT}.")
else:
    twpa = machine.twpas[TWPA_ID]
    twpa.pump.RF_frequency  = "#../pump_frequency"
    twpa.pump_.RF_frequency = "#../pump_frequency"
    if twpa.pump_frequency is None:
        twpa.pump_frequency = twpa_pump_freq
    if twpa.pump_amplitude is None:
        twpa.pump_amplitude = twpa_amp


machine.save()
print("QUAM saved.")
with open("qua_config.json", "w+") as f:
    json.dump(machine.generate_config(), f, indent=4)
print("QUA config saved.")

  Created cavity 'c1' with alice and bob modes.
  Created CavityTransmonPair 'q1_alice'
  Created CavityTransmonPair 'q1_bob'
  Created sideband_drive for 'q1_alice'.
  Created TWPA 'twpa1' on FEM 1 port 5.
QUAM saved.
QUA config saved.


### 2b. OPX+ / Octave

In [ ]:
import json
import numpy as np
from pprint import pprint
from qualang_tools.units import unit
from quam.components.pulses import SquarePulse, DragCosinePulse, DragGaussianPulse
from quam.components.octave import OctaveUpConverter
from quam.components.channels import DigitalOutputChannel
from quam.components.ports import OPXPlusAnalogOutputPort, OPXPlusDigitalOutputPort
from quam_builder.architecture.superconducting.components.xy_drive import XYDriveIQ
from quam_builder.architecture.superconducting.cavity.cavity import Cavity
from quam_builder.architecture.superconducting.cavity.cavity_mode import CavityMode
from quam_builder.builder.superconducting.pulses import add_DragGaussian_pulses
from quam_config import Quam
from quam_builder.architecture.superconducting.qubit_pair import CavityTransmonPair

u = unit(coerce_to_integer=True)


def get_octave_gain_and_amplitude(desired_power: float, max_amplitude: float = 0.125):
    """Convert desired output power (dBm) to Octave gain + OPX IF amplitude."""
    octave_gain = round(max(min(desired_power - u.volts2dBm(max_amplitude), 20), -20) * 2) / 2
    amplitude = u.dBm2volts(desired_power - octave_gain)
    if not (-20 <= octave_gain <= 20 and -0.5 <= amplitude < 0.5):
        raise ValueError(f"Power outside spec: gain={octave_gain}, amp={amplitude}")
    return octave_gain, amplitude


machine = Quam.load()
n_qubits = len(machine.qubits)

##########################################################################
# USER PARAMETERS -- edit to match chip specs
# For N qubits: supply N-element numpy arrays for rr_freq, xy_freq, and
# anharmonicity (one entry per qubit, in machine order). rr_LO and xy_LO
# are single shared values (one Octave LO for all qubits on each line).
#
# Hardware routing summary (all Octave LOs internal):
#   Resonator  -> RF_outputs/1 (synth1), shared by all qubits (FDM)
#   Qubit XY   -> RF_outputs/3 (synth3), shared by all qubits (FDM),
#                 OPX+ ports 5/6 I/Q
#   f0g1       -> RF_outputs/2 (synth2), OPX+ ports 3/4 I/Q
#   Cavity     -> RF_outputs/4 (synth4), OPX+ ports 7/8 I/Q
#
# Octave digital trigger convention: RF out N -> OPX+ digital output
# con1/(2N-1), i.e. RF1->1, RF2->3, RF3->5, RF4->7, RF5->9.
##########################################################################
CAVITY_ID = "c1"

# -- Readout resonator (shared Octave RF1, one LO for all 6 qubits, FDM) ------
rr_freq       = np.array([7.231e9, 7.167e9, 7.261e9, 7.384e9, 7.515e9, 7.635e9])  # Hz, one per qubit
rr_LO         = 7.400e9               # Hz, shared Octave RF_outputs/1 LO
readout_power = 20                    # dBm
readout_gain  = 10                    # dB

# -- Qubit XY drives (shared Octave RF3, one LO for all 6 qubits, FDM) --------
xy_freq       = np.array([4.487e9, 4.287e9, 3.792e9, 3.836e9, 3.892e9, 3.803e9])  # Hz, one per qubit
xy_LO         = 4.140e9               # Hz, shared Octave RF_outputs/3 LO
anharmonicity = np.array([-200e6, -200e6, -200e6, -200e6, -200e6, -200e6])  # Hz, one per qubit
drive_power   = -10                   # dBm

# -- Cavity drives (alice + bob share RF4 LO) ---------------------------------
alice_freq    = 6.000e9
alice_LO      = 5.900e9
alice_power   = 20
bob_freq      = 6.200e9
bob_power     = 20

# -- f0g1 sideband drive (RF_outputs/2, internal LO) --------------------------
alice_f0g1_freq              = 3.25e9
alice_f0g1_LO                = 3.0e9
alice_f0g1_gain              = 0
alice_f0g1_saturation_length_ns = 20000
alice_f0g1_pi_length_ns      = 1000
alice_f0g1_sigma_ns          = 200
alice_f0g1_amp               = 0.4

# -- Sideband cooling calibration ----------------------------------------------
max_fock_level = 4        # calibrate transitions f0g1 through f{max-1}gmax
chi_guess_hz   = -2e6     # rough cavity-qubit chi [Hz]; negative sign typical

# -- Pulse defaults -----------------------------------------------------------
readout_length_ns            = 8000
saturation_length_ns         = 20000
x180_length_ns               = 1000
gaussian_sigma_ns            = x180_length_ns // 5
drag_alpha                   = 0.0
drag_detuning                = 0.0
selective_x180_length_ns     = 10000
ef_x180_amplitude            = 0.1
displacement_length_ns       = 1000
displacement_sigma_ns        = displacement_length_ns // 5
displacement_initial_amplitude_V = 0.001

# -- T1 estimates -------------------------------------------------------------
T1                          = 200e-6
cavity_T1                   = 100e-6
resonator_depletion_time_ns = 10000
##########################################################################

assert len(rr_freq) == n_qubits,       f"rr_freq: expected {n_qubits} entries, got {len(rr_freq)}"
assert len(xy_freq) == n_qubits,       f"xy_freq: expected {n_qubits} entries, got {len(xy_freq)}"
assert len(anharmonicity) == n_qubits, f"anharmonicity: expected {n_qubits} entries, got {len(anharmonicity)}"
assert np.all(np.abs(rr_freq - rr_LO) < 400e6), "Resonator IF out of range"
assert np.all(np.abs(xy_freq - xy_LO) < 400e6), "XY IF out of range"
assert abs(alice_freq - alice_LO) < 400e6,       "Alice IF out of range"
assert abs(bob_freq - alice_LO)   < 400e6,       "Bob IF out of range (must share LO with Alice)"
assert abs(alice_f0g1_freq - alice_f0g1_LO) < 400e6, "f0g1 IF out of range"

# -- OPX+ analog/digital port objects ------------------------------------------
_ao = machine.ports.analog_outputs.setdefault("con1", {})
_do = machine.ports.digital_outputs.setdefault("con1", {})

for port_id in (1, 2):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(controller_id="con1", port_id=port_id, delay=0, shareable=False)
if 1 not in _do:
    _do[1] = OPXPlusDigitalOutputPort(controller_id="con1", port_id=1, shareable=False)

# -- Octave RF_outputs/1: shared readout LO -----------------------------------
rr_rf1 = machine.octaves["oct1"].RF_outputs[1]
rr_rf1.LO_frequency = rr_LO
rr_rf1.LO_source    = "internal"
rr_rf1.output_mode  = "triggered"
# Divide max_amplitude by n_qubits so simultaneous readout tones stay in range
rr_gain, rr_amp = get_octave_gain_and_amplitude(readout_power, max_amplitude=0.125 / max(1, n_qubits))
if rr_rf1.gain is None:
    rr_rf1.gain = rr_gain

rr_rfi2 = machine.octaves["oct1"].RF_inputs[2]
rr_rfi2.LO_source    = "internal"
rr_rfi2.LO_frequency = "#/octaves/oct1/RF_outputs/1/LO_frequency"
rr_rfi2.gain_db      = readout_gain

# -- Octave RF_outputs/3: qubit XY drives, FDM across all qubits --------------
# All 6 qubits share OPX+ ports 5/6 + Octave RF3 (one LO, xy_LO). Each
# qubit's RF_frequency sets its own intermediate frequency relative to it.
# Divide max_amplitude by n_qubits so simultaneous drive tones stay in range.
xy_gain, xy_amp = get_octave_gain_and_amplitude(drive_power, max_amplitude=0.125 / max(1, n_qubits))
xy_rf3 = machine.octaves["oct1"].RF_outputs[3]
xy_rf3.LO_frequency = xy_LO
xy_rf3.LO_source    = "internal"
xy_rf3.output_mode  = "triggered"

# -- f0g1 sideband drive (RF_outputs/2) ----------------------------------------
for port_id in (3, 4):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(controller_id="con1", port_id=port_id, delay=0, shareable=True)
    else:
        _ao[port_id].shareable = True
if 3 not in _do:
    _do[3] = OPXPlusDigitalOutputPort(controller_id="con1", port_id=3, shareable=False)

f0g1_rf2 = machine.octaves["oct1"].RF_outputs[2]
f0g1_rf2.LO_frequency = alice_f0g1_LO
f0g1_rf2.LO_source    = "internal"
f0g1_rf2.gain         = alice_f0g1_gain
f0g1_rf2.output_mode  = "triggered"

# -- Cavity drive (RF_outputs/4): alice + bob share ----------------------------
for port_id in (7, 8):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(controller_id="con1", port_id=port_id, delay=0, shareable=True)
    else:
        _ao[port_id].shareable = True
if 7 not in _do:
    _do[7] = OPXPlusDigitalOutputPort(controller_id="con1", port_id=7, shareable=True)
else:
    _do[7].shareable = True

cav_rf4 = machine.octaves["oct1"].RF_outputs[4]
cav_gain, cav_amp = get_octave_gain_and_amplitude(alice_power)
cav_rf4.LO_frequency = alice_LO
cav_rf4.LO_source    = "internal"
cav_rf4.gain         = cav_gain
cav_rf4.output_mode  = "triggered"

rf5 = machine.octaves["oct1"].RF_outputs[5]
rf5.LO_source   = "internal"
rf5.output_mode = "always_off"

machine.octaves["oct1"].loopbacks = []

# -- Resonator: shared feed-line, per-qubit IF frequencies --------------------
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.resonator.f_01         = float(rr_freq[k])
    qubit.resonator.RF_frequency = float(rr_freq[k])
    qubit.resonator.frequency_bare = float(rr_freq[k])
    qubit.resonator.frequency_converter_up.LO_frequency = rr_LO    # shared
    qubit.resonator.frequency_converter_up.gain         = rr_gain
    qubit.resonator.frequency_converter_up.output_mode  = "triggered"
    if qubit.resonator.depletion_time is None:
        qubit.resonator.depletion_time = resonator_depletion_time_ns
    ro_op = qubit.resonator.operations.get("readout")
    if ro_op is not None:
        ro_op.amplitude = rr_amp
        ro_op.length = readout_length_ns

# -- Qubit XY: per-qubit drive frequency, all sharing Octave RF3 (FDM) --------
# Every qubit's xy element points at the SAME OPX+ I/Q ports + Octave RF3
# (allocated in the wiring step); only RF_frequency differs per qubit.
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.f_01                                          = float(xy_freq[k])
    qubit.xy.RF_frequency                               = float(xy_freq[k])
    qubit.xy.frequency_converter_up.LO_frequency        = xy_LO    # shared
    qubit.xy.frequency_converter_up.gain                = xy_gain
    qubit.xy.frequency_converter_up.output_mode         = "triggered"
    if qubit.T1 is None:
        qubit.T1 = T1
    if qubit.anharmonicity is None:
        qubit.anharmonicity = int(anharmonicity[k])
    if qubit.grid_location is None or qubit.grid_location == "":
        qubit.grid_location = f"{k},0"
    sat_op = qubit.xy.operations.get("saturation")
    if sat_op is not None:
        if sat_op.amplitude == 0 or sat_op.amplitude is None:
            sat_op.amplitude = 0.3
        if sat_op.length == 0:
            sat_op.length = saturation_length_ns
        if sat_op.digital_marker is None:
            sat_op.digital_marker = "ON"
    add_DragGaussian_pulses(qubit, xy_amp, x180_length_ns, gaussian_sigma_ns,
                            drag_alpha, drag_detuning, int(anharmonicity[k]),
                            digital_marker="ON")

# -- Cavity object (alice + bob) -----------------------------------------------
if CAVITY_ID not in machine.cavities:
    def _make_cavity_drive():
        drive = XYDriveIQ(
            opx_output_I="#/ports/analog_outputs/con1/7",
            opx_output_Q="#/ports/analog_outputs/con1/8",
            frequency_converter_up="#/octaves/oct1/RF_outputs/4",
            RF_frequency=None,
        )
        drive.digital_outputs["octave_switch_0"] = DigitalOutputChannel(
            opx_output="#/ports/digital_outputs/con1/7", delay=57, buffer=18)
        return drive

    alice_mode = CavityMode(id="alice", cavity_mode_drive=_make_cavity_drive())
    alice_mode.T1 = cavity_T1
    bob_mode = CavityMode(id="bob", cavity_mode_drive=_make_cavity_drive())
    bob_mode.T1 = cavity_T1
    machine.cavities[CAVITY_ID] = Cavity(id=CAVITY_ID, alice=alice_mode, bob=bob_mode)
    cav_rf4.channel = f"#/cavities/{CAVITY_ID}/alice/cavity_mode_drive"
    print(f"  Created cavity '{CAVITY_ID}'.")
else:
    for mode_name in ("alice", "bob"):
        mode = getattr(machine.cavities[CAVITY_ID], mode_name, None)
        if mode is not None:
            mode.T1 = cavity_T1

for cav_name, cavity in machine.cavities.items():
    for mode_name, freq in (("alice", alice_freq), ("bob", bob_freq)):
        mode = getattr(cavity, mode_name, None)
        if mode is None or mode.cavity_mode_drive is None:
            continue
        mode.cavity_mode_drive.RF_frequency = freq
        if "saturation" not in mode.cavity_mode_drive.operations:
            mode.cavity_mode_drive.operations["saturation"] = SquarePulse(
                length=readout_length_ns, amplitude=cav_amp, digital_marker="ON")
        if "displacement" not in mode.cavity_mode_drive.operations:
            mode.cavity_mode_drive.operations["displacement"] = DragGaussianPulse(
                length=displacement_length_ns, amplitude=displacement_initial_amplitude_V,
                sigma=displacement_sigma_ns, alpha=0.0, anharmonicity=0,
                detuning=0.0, axis_angle=0, digital_marker="ON",
            )

# -- CavityTransmonPair (with sideband_drive) ----------------------------------
for q_name in machine.qubits:
    for mode_name in ("alice", "bob"):
        pair_key = f"{q_name}_{mode_name}"
        if pair_key not in machine.cavity_transmon_pairs:
            machine.cavity_transmon_pairs[pair_key] = CavityTransmonPair(
                qubit_name=q_name, cavity_mode_name=mode_name)
            print(f"  Created CavityTransmonPair '{pair_key}'")
    for pk in (f"{q_name}_alice", f"{q_name}_bob"):
        p = machine.cavity_transmon_pairs.get(pk)
        if p is not None and not hasattr(p, "parity_time"):
            p.parity_time = None

    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is not None and alice_pair.sideband_drive is None:
        alice_drive = XYDriveIQ(
            id=f"{q_name}_alice_f0g1",
            opx_output_I="#/ports/analog_outputs/con1/3",
            opx_output_Q="#/ports/analog_outputs/con1/4",
            frequency_converter_up="#/octaves/oct1/RF_outputs/2",
            RF_frequency=alice_f0g1_freq,
        )
        alice_drive.digital_outputs["octave_switch_0"] = DigitalOutputChannel(
            opx_output="#/ports/digital_outputs/con1/3", delay=57, buffer=18)
        alice_drive.operations["saturation"] = SquarePulse(
            length=alice_f0g1_saturation_length_ns, amplitude=alice_f0g1_amp, digital_marker="ON")
        alice_drive.operations["f0g1_pi"] = DragCosinePulse(
            length=alice_f0g1_pi_length_ns, axis_angle=0.0, alpha=0.0,
            anharmonicity=0, amplitude=alice_f0g1_amp, digital_marker="ON")
        alice_pair.sideband_drive = alice_drive
        f0g1_rf2.channel = f"#/cavity_transmon_pairs/{q_name}_alice/sideband_drive"
        print(f"  Created sideband_drive for '{q_name}_alice'.")
    elif alice_pair is not None and alice_pair.sideband_drive is not None:
        alice_pair.sideband_drive.RF_frequency = alice_f0g1_freq

# -- Multi-level sideband pulses for active cooling (f0g1 through f{N-1}gN) ---
for q_name in machine.qubits:
    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is None or alice_pair.sideband_drive is None:
        continue
    sd = alice_pair.sideband_drive
    if alice_pair.extras is None:
        alice_pair.extras = {}
    for _k in range(max_fock_level):
        _op  = f"f{_k}g{_k+1}_pi"
        _fkey = f"f{_k}g{_k+1}_RF_frequency"
        _lkey = f"f{_k}g{_k+1}_pi_length_ns"
        if _op not in sd.operations:
            sd.operations[_op] = DragCosinePulse(
                length=alice_f0g1_pi_length_ns, axis_angle=0.0,
                alpha=0.0, anharmonicity=0, amplitude=alice_f0g1_amp,
                digital_marker="ON",
            )
        alice_pair.extras.setdefault(_fkey, alice_f0g1_freq - _k * abs(chi_guess_hz))
        alice_pair.extras.setdefault(_lkey, alice_f0g1_pi_length_ns)

machine.save()
print("QUAM saved.")
with open("qua_config.json", "w+") as f:
    json.dump(machine.generate_config(), f, indent=4)
print("QUA config saved.")


QUAM saved.
QUA config saved.


## 3. Device setup


### 3a. Close other quantum machines

In [7]:
from qualibrate import QualibrationNode, NodeParameters
from quam_config import Quam

node = QualibrationNode[NodeParameters, Quam](
    name="00_close_other_qms",
    description="Close all other open QMs.",
    parameters=NodeParameters(),
)
node.machine = Quam.load()

@node.run_action()
def close_all_quantum_machines(node: QualibrationNode[NodeParameters, Quam]):
    qmm = node.machine.connect()
    qmm.close_all_qms()
    print("All quantum machines closed.")

Running action close_all_quantum_machines
2026-07-14 18:11:58,602 - qm - INFO     - Performing health check
2026-07-14 18:11:58,602 - qm - INFO     - Cluster healthcheck completed successfully.
All quantum machines closed.
Action close_all_quantum_machines finished


### 3b. Scope verification - all channels

Play each hardware element in an infinite loop to verify signals on the oscilloscope / spectrum analyser.

> **Run only the sub-section matching your hardware.** Run the `scope_job.halt()` cell to stop.

#### OPX1000 / MW-FEM

Elements are uncommented individually; add/remove as needed.

In [ ]:
from qm import QuantumMachinesManager
from qm.qua import *
from quam_config import Quam

machine = Quam.load()
config = machine.generate_config()

qmm = QuantumMachinesManager(host=machine.network.host, cluster_name=machine.network.cluster_name)
qm = qmm.open_qm(config)

# Build element list from all qubits in the machine.
# Enable/disable individual entries by toggling the boolean flag.
ELEMENTS = {}
for q_name, qubit in machine.qubits.items():
    ELEMENTS[f"{q_name}_rr"] = (qubit.resonator.name, True,  "readout")
    ELEMENTS[f"{q_name}_xy"] = (qubit.xy.name,        False, "saturation")  # set True to verify drive

# Cavity / f0g1 elements (uncomment as needed):
# cav = machine.cavities.get("c1")
# if cav is not None:
#     ELEMENTS["alice"] = (cav.alice.cavity_mode_drive.name, True, "saturation")
#     ELEMENTS["bob"]   = (cav.bob.cavity_mode_drive.name,   True, "saturation")
# for pair in machine.cavity_transmon_pairs.values():
#     if getattr(pair, "sideband_drive", None) is not None:
#         ELEMENTS["f0g1"] = (pair.sideband_drive.name, True, "saturation")

active = [(name, pulse) for name, (name, enabled, pulse) in ELEMENTS.items() if enabled]
# Re-build from tuple structure (name -> (elem_name, enabled, pulse))
active = [(elem, pulse) for elem, enabled, pulse in ELEMENTS.values() if enabled]
print("Active elements:")
for elem, pulse in active:
    print(f"  {elem:40s}  op='{pulse}'")

with program() as scope_prog:
    with infinite_loop_():
        for elem_name, pulse_name in active:
            play(pulse_name, elem_name)
            wait(1000 // 4, elem_name)

scope_job = qm.execute(scope_prog)
print("Scope running. Execute the halt cell to stop.")


In [ ]:
scope_job.halt()

#### OPX+ / Octave

Plays resonator, qubit XY, f0g1 sideband, Alice cavity, and Bob cavity simultaneously.

In [ ]:
from qm import QuantumMachinesManager
from qm.qua import *
from quam_config import Quam

machine = Quam.load()
config = machine.generate_config()

qmm = QuantumMachinesManager(host=machine.network.host, cluster_name=machine.network.cluster_name)
qm = qmm.open_qm(config)

# Build element list from all qubits in the machine.
# Enable/disable individual entries by toggling the boolean flag.
ELEMENTS = {}
for q_name, qubit in machine.qubits.items():
    ELEMENTS[f"{q_name}_rr"] = (qubit.resonator.name, True,  "readout")
    ELEMENTS[f"{q_name}_xy"] = (qubit.xy.name,        True,  "saturation")

cav = machine.cavities.get("c1")
if cav is not None:
    if getattr(cav, "alice", None) is not None and cav.alice.cavity_mode_drive is not None:
        ELEMENTS["alice"] = (cav.alice.cavity_mode_drive.name, True, "saturation")
    if getattr(cav, "bob", None) is not None and cav.bob.cavity_mode_drive is not None:
        ELEMENTS["bob"] = (cav.bob.cavity_mode_drive.name, True, "saturation")
for pair in machine.cavity_transmon_pairs.values():
    if getattr(pair, "sideband_drive", None) is not None:
        ELEMENTS["f0g1"] = (pair.sideband_drive.name, True, "f0g1_pi")

active = [(el, op) for el, (el, en, op) in ELEMENTS.items() if en]
active = [(el, op) for el, en, op in ELEMENTS.values() if en]
print("Active elements:")
for el, op in active:
    print(f"  {el:40s}  op='{op}'")

with program() as scope_cw:
    with infinite_loop_():
        align(*[el for el, _ in active])
        for el, op in active:
            play(op, el)

print("Playing in infinite loop -- run the halt cell below to stop.")
scope_job = qm.execute(scope_cw)


2026-06-08 22:46:39,480 - qm - INFO     - Performing health check


INFO:qm.api.frontend_api:Performing health check


2026-06-08 22:46:39,791 - qm - INFO     - Health check passed


INFO:qm.api.frontend_api:Health check passed


2026-06-08 22:46:39,874 - qm - WARNING  - No calibration_db set in octave config, skipping loading calibration data


2026-06-08 22:46:42,400 - qm - ERROR    - CONFIG ERROR in key "mixers.q1_alice_f0g1_mixer_c87" [mixers.corrections.intermediateFrequency.values] : Mixer 'q1_alice_f0g1_mixer_c87' correction 'FrequencyCorrection(frequency=-8.5E8, loFrequency=4.1E9, correction=CorrectionMatrix(v00=1.0, v01=0.0, v10=0.0, v11=1.0))' has an intermediate frequency less than -5.0E8


ERROR:qm.api.frontend_api:CONFIG ERROR in key "mixers.q1_alice_f0g1_mixer_c87" [mixers.corrections.intermediateFrequency.values] : Mixer 'q1_alice_f0g1_mixer_c87' correction 'FrequencyCorrection(frequency=-8.5E8, loFrequency=4.1E9, correction=CorrectionMatrix(v00=1.0, v01=0.0, v10=0.0, v11=1.0))' has an intermediate frequency less than -5.0E8


2026-06-08 22:46:42,400 - qm - ERROR    - CONFIG ERROR in key "elements.q1_alice_f0g1" [element.intermediateFrequencyUpperLimit] : Element 'q1_alice_f0g1' has intermediate frequency less than -5.0E8 (-8.5E8)


ERROR:qm.api.frontend_api:CONFIG ERROR in key "elements.q1_alice_f0g1" [element.intermediateFrequencyUpperLimit] : Element 'q1_alice_f0g1' has intermediate frequency less than -5.0E8 (-8.5E8)


OpenQmException: Can not open QM, see the following errors:
CONFIG ERROR in key "mixers.q1_alice_f0g1_mixer_c87" [mixers.corrections.intermediateFrequency.values] : Mixer 'q1_alice_f0g1_mixer_c87' correction 'FrequencyCorrection(frequency=-8.5E8, loFrequency=4.1E9, correction=CorrectionMatrix(v00=1.0, v01=0.0, v10=0.0, v11=1.0))' has an intermediate frequency less than -5.0E8
CONFIG ERROR in key "elements.q1_alice_f0g1" [element.intermediateFrequencyUpperLimit] : Element 'q1_alice_f0g1' has intermediate frequency less than -5.0E8 (-8.5E8)


In [ ]:
scope_job.halt()

### 3c. OPX1000 only - frequency convention diagnostic

Verifies that `update_frequency` uses the correct band on the MW-FEM. Skip for OPX+ / Octave.

In [ ]:
# ============================================================
# DIAGNOSTIC: test update_frequency convention on OPX1000
# ============================================================
# Run the scope program first (play only, no update_frequency) and
# confirm the output frequency on a spectrum analyzer -> should be at RF.
# Then run THIS cell to call update_frequency with the same IF value
# (df=0 -> same as configured).
# If the scope shows sub-1 GHz AFTER this cell (vs correct RF before),
# then update_frequency is triggering direct synthesis on Band 1,
# and the qubit spectroscopy nodes need to use RF_frequency instead.
# ============================================================

from qm import QuantumMachinesManager
from qm.qua import *
from quam_config import Quam

machine = Quam.load()
config = machine.generate_config()
qubit = machine.qubits["q1"]

qmm = QuantumMachinesManager(
    host=machine.network.host,
    cluster_name=machine.network.cluster_name,
)
qm = qmm.open_qm(config)

print(f"Qubit XY element  : {qubit.xy.name}")
print(f"RF_frequency      : {qubit.xy.RF_frequency / 1e9:.6f} GHz")
print(f"LO_frequency      : {qubit.xy.LO_frequency / 1e9:.6f} GHz")
print(f"intermediate_freq : {qubit.xy.intermediate_frequency / 1e6:.3f} MHz")
print()
print("Scope test A - play() only (no update_frequency):")
print("  Expected scope output: RF =", qubit.xy.RF_frequency / 1e9, "GHz")

with program() as test_A:
    with infinite_loop_():
        play("saturation", qubit.xy.name)
        wait(1000 // 4, qubit.xy.name)

job_A = qm.execute(test_A)
input("Check scope -> should show RF frequency. Press Enter to stop and run test B...")
job_A.halt()

print()
print("Scope test B - update_frequency(intermediate_frequency) then play():")
print("  Expected: same RF frequency (df=0, so IF unchanged)")
print("  If scope shows sub-1 GHz here -> update_frequency uses direct output, not IF")

with program() as test_B:
    with infinite_loop_():
        # Same IF as configured (df=0), should not change output
        qubit.xy.update_frequency(qubit.xy.intermediate_frequency)
        play("saturation", qubit.xy.name)
        wait(1000 // 4, qubit.xy.name)

job_B = qm.execute(test_B)
input("Check scope -> compare with test A. Press Enter to stop...")
job_B.halt()

print()
print("Scope test C - update_frequency(RF_frequency) then play():")
print("  If OPX1000 expects RF (not IF) in update_frequency, this should give correct output")

with program() as test_C:
    with infinite_loop_():
        qubit.xy.update_frequency(qubit.xy.RF_frequency)
        play("saturation", qubit.xy.name)
        wait(1000 // 4, qubit.xy.name)

job_C = qm.execute(test_C)
input("Check scope -> if this gives correct RF, the fix is to use RF_frequency in update_frequency. Press Enter to stop...")
job_C.halt()
print("Done.")


### 3d. Mixer calibration

> **OPX+ / Octave only.** Skip for OPX1000 - MW-FEM uses direct digital synthesis and has no analog IQ mixers.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

mixer_cal = library.nodes["01a_mixer_calibration"].copy(name="mixer_calibration")
mixer_cal.parameters.qubits = ["q1"]
mixer_cal.run()

Running action execute_qua_program
2026-06-09 22:53:13,761 - qm - INFO     - Performing health check
2026-06-09 22:53:14,058 - qm - INFO     - Health check passed
2026-06-09 22:53:15,745 - qm - INFO     - Opening QM
2026-06-09 22:53:15,745 - qm - INFO     - Calibrating q1.resonator
2026-06-09 22:53:18,107 - qm - INFO     - Compiling program
2026-06-09 22:53:23,765 - qm - INFO     - Calibrating q1.xy
2026-06-09 22:53:25,776 - qm - INFO     - Compiling program
2026-06-09 22:53:31,349 - qm - WARNING  - At least one of the correction values are out of range. values should be between -2 and 2 - 2 ** (-16), got [-1.830096686840894, 0.6952398362900426, 4.052129711781794, -10.666519340654789]. Not setting the correction matrix.
2026-06-09 22:53:33,406 - qm - INFO     - Compiling program
2026-06-09 22:53:40,937 - qm - INFO     - Compiling program
2026-06-09 22:53:48,594 - qm - INFO     - Compiling program
2026-06-09 22:53:54,166 - qm - INFO     - Closing QM
Action execute_qua_program finished
R

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\01a_mixer_calibration.py:123: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action save_results


save failed: No database connection configured for project 'noise_study'


Action save_results finished


NodeRunSummary(name='mixer_calibration', description='\n    A simple program to calibrate Octave mixers for all qubits and resonators\n', created_at=datetime.datetime(2026, 6, 9, 22, 53, 13, 639625, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 6, 9, 22, 54, 0, 271776, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), parameters=Parameters(multiplexed=False, use_state_discrimination=False, reset_type='thermal', qubits=['q1'], calibrate_resonator=True, calibrate_drive=True, calibrate_cavity_drive=True, calibrate_sideband_drive=True, simulate=False, simulation_duration_ns=50000, use_waveform_report=True, timeout=120, load_data_id=None), outcomes={'q1': <Outcome.SUCCESSFUL: 'successful'>, 'alice': <Outcome.SUCCESSFUL: 'successful'>, 'bob': <Outcome.SUCCESSFUL: 'successful'>, 'q1_alice': <Outcome.SUCCESSFUL: 'successful'>}, error=None, initial_targets=['q1'],

### 3e. Time of flight

In [ ]:
from quam_config import Quam
machine = Quam.load()

# Adjust TOF for all qubits sharing the same readout line.
# All qubits on one feed-line typically have the same TOF.
tof_ns = 40   # ns  edit to match measured TOF
for q_name, qubit in machine.qubits.items():
    qubit.resonator.time_of_flight = tof_ns
    print(f"{q_name}: TOF = {qubit.resonator.time_of_flight} ns")

machine.save()


q1: TOF = 40 ns


In [ ]:
from qualibrate import QualibrationLibrary
from quam_config import Quam
from quam_builder.architecture.superconducting.components.readout_resonator import ReadoutResonatorMW

machine = Quam.load()
library = QualibrationLibrary.get_active_library()

qubit = next(iter(machine.qubits.values()))
if isinstance(qubit.resonator, ReadoutResonatorMW):
    tof_node = library.nodes["01b_time_of_flight_mw_fem"].copy(name="time_of_flight")
    tof_node.parameters.readout_amplitude_in_dBm = 0
else:
    tof_node = library.nodes["01a_time_of_flight"].copy(name="time_of_flight")
    tof_node.parameters.readout_amplitude_in_v = 0.0001

tof_node.parameters.qubits = ["q1"]
tof_node.run()

Getting calibration path from config
c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
An error occurred on scanning graph file 80_calibration_graph_bringup_flux_tunable_transmon.py
Traceback (most recent call last):
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\qualibration_graph.py", line 422, in scan_folder_for_instances
    cls.scan_graph_file(file, graphs)
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\qualibration_graph.py", line 452, in scan_graph_file
    _module = import_from_path(get_module_name(file), file)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packag

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 10:48:04,354 - qm - INFO     - Performing health check
2026-06-30 10:48:04,364 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 10:48:04,775 - qm - INFO     - Opened quantum machine with id: QM-d9168cf9-8dde-4653-b22b-c309fc9ec963
2026-06-30 10:48:04,785 - qm - INFO     - Opening QM
2026-06-30 10:48:04,785 - qm - INFO     - Clearing queue
2026-06-30 10:48:04,885 - qm - INFO     - Adding program to queue.
2026-06-30 10:48:08,257 - qm - INFO     - Program added to queue. Job id: 651f835a-900f-4c04-b8b9-39deb2c80839
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 0.32s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 0.47s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 0.61s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\01b_time_of_flight_mw_fem.py:207: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='time_of_flight', description='\n        TIME OF FLIGHT - MW FEM\nThis sequence involves sending a readout pulse and capturing the raw ADC traces.\nThe data undergoes post-processing to calibrate three distinct parameters:\n    - Time of Flight: This represents the internal processing time and the propagation\n      delay of the readout pulse. Its value can be adjusted in the configuration under\n      "time_of_flight". This value is utilized to offset the acquisition window relative\n      to when the readout pulse is dispatched.\n\n    - Analog Inputs Gain: If a signal is constrained by digitization or if it saturates\n      the ADC, the variable gain of the OPX analog input, ranging from -12 dB to 20 dB,\n      can be modified to fit the signal within the ADC range of +/-0.5V.\n      \nPrerequisites:\n    - Having initialized the Quam (quam_config/populate_quam_state_*.py).\n\nState update:\n    - The time of flight: qubit.resonator.time_of_flight\n', created_at=

### 3f. TWPA calibration

Calibrates the TWPA pump (power, frequency) and signal saturation power,
using the existing qubit's `resonator` channel as the VNA probe and the
TWPA's non-sticky `pump_` element to play the pump tone. Gain is always
reported in dB relative to pump-off transmission.

Run the three steps below in order: power sweep -> frequency sweep ->
signal saturation. For a fully automated run, see section 10a.

#### 3f-i. TWPA pump power sweep

Sweeps the TWPA pump power (~10 dB span) at a fixed datasheet-guess
frequency to find the gain-vs-power curve and saturation point.

**State update**: `twpa.pump_amplitude`, `twpa.max_avg_gain`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

twpa_power_sweep = library.nodes["01_twpa_pump_power_sweep"].copy(name="twpa_pump_power_sweep")
twpa_power_sweep.parameters.qubits = ["q1"]
twpa_power_sweep.parameters.twpa_id = "twpa1"
twpa_power_sweep.parameters.signal_frequency_span_in_mhz = 500.0
twpa_power_sweep.parameters.signal_frequency_step_in_mhz = 0.5
twpa_power_sweep.parameters.signal_power_dbm = None  # None -> keep currently configured readout amplitude
twpa_power_sweep.parameters.pump_power_min_dbm = -25.0
twpa_power_sweep.parameters.pump_power_max_dbm = -10.0
twpa_power_sweep.parameters.pump_power_step_db = 0.25
twpa_power_sweep.parameters.num_shots = 10
twpa_power_sweep.run()

#### 3f-ii. TWPA pump frequency sweep

Sweeps the TWPA pump frequency (~200 MHz span) at the best power found in
3f-i, mapping gain vs (pump frequency, signal frequency) to pick the
operating point.

**State update**: `twpa.pump_frequency`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

twpa_freq_sweep = library.nodes["02_twpa_pump_frequency_sweep"].copy(name="twpa_pump_frequency_sweep")
twpa_freq_sweep.parameters.qubits = ["q1"]
twpa_freq_sweep.parameters.twpa_id = "twpa1"
twpa_freq_sweep.parameters.signal_frequency_span_in_mhz = 500.0
twpa_freq_sweep.parameters.signal_frequency_step_in_mhz = 0.5
twpa_freq_sweep.parameters.signal_power_dbm = None  # None -> keep currently configured readout amplitude
twpa_freq_sweep.parameters.pump_frequency_span_in_mhz = 200.0
twpa_freq_sweep.parameters.pump_frequency_step_in_mhz = 5.0
twpa_freq_sweep.parameters.num_shots = 10
twpa_freq_sweep.run()

#### 3f-iii. TWPA signal saturation power sweep

At the calibrated pump setting, sweeps the signal (readout) power to find
the TWPA's input/output saturation power (1 dB compression point).

**State update**: `twpa.p_saturation`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

twpa_saturation = library.nodes["03_twpa_signal_saturation_power_sweep"].copy(name="twpa_signal_saturation_power_sweep")
twpa_saturation.parameters.qubits = ["q1"]
twpa_saturation.parameters.twpa_id = "twpa1"
twpa_saturation.parameters.signal_power_min_dbm = -20.0
twpa_saturation.parameters.signal_power_max_dbm = 0.0
twpa_saturation.parameters.num_power_points = 16
twpa_saturation.parameters.num_shots = 50
twpa_saturation.run()

## Manual Calibration

Cells below run individual calibration nodes manually.
Use these for initial device characterisation, debugging, or to re-run a specific step.

For a fully automated run from scratch, see the **Automated Calibration Graphs** section (sec. 10).

### 4. Readout resonator

#### 4a. Wide resonator spectroscopy

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

broad_spec = library.nodes["02d_broad_resonator_spectroscopy"].copy(name="broad_res_spec")
broad_spec.parameters.qubits = ["q1"]
broad_spec.parameters.frequency_span_in_mhz = 600.0
broad_spec.parameters.frequency_step_in_mhz = 0.1
broad_spec.parameters.num_shots = 50
broad_spec.parameters.peak_prominence = 10.0
broad_spec.parameters.peak_width = (1, 10.0)
broad_spec.parameters.blacklist_exclusion_radius_mhz = 10.0
broad_spec.parameters.readout_power_dbm = -30.0
broad_spec.parameters.max_amp = 0.3
broad_spec.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 17:02:09,666 - qm - INFO     - Performing health check
2026-07-02 17:02:09,676 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 17:02:09,836 - qm - INFO     - Opened quantum machine with id: QM-de5ae072-36b2-4fc8-9d85-0ed2cc817803
2026-07-02 17:02:09,846 - qm - INFO     - Opening QM
2026-07-02 17:02:09,846 - qm - INFO     - Clearing queue
2026-07-02 17:02:09,846 - qm - INFO     - Adding program to queue.
2026-07-02 17:02:10,096 - qm - INFO     - Program added to queue. Job id: e9baebd6-dfad-4278-ac87-96bc9860cbf6
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 8.66s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 8.70s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 8.75s
Progress: [###################

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02d_broad_resonator_spectroscopy.py:215: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='broad_res_spec', description="\n        1D BROAD-BAND RESONATOR SPECTROSCOPY\nThis sequence involves measuring the resonator by sending a readout pulse and demodulating the signals to extract the\n'I' and 'Q' quadratures across varying readout intermediate frequencies for all the active qubits.\nThe data is then post-processed to determine the resonator resonance frequency.\nThis frequency is used to update the readout frequency in the state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the readout line (node 01a_mixer_calibration.py).\n    - Having calibrated the time of flight, offsets, and gains (node 01a_time_of_flight.py).\n    - Having initialized the QUAM state parameters for the readout pulse amplitude and duration, and the resonators depletion time.\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency

#### 4b. Resonator spectroscopy (fine)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

res_spec = library.nodes["02a_resonator_spectroscopy"].copy(name="resonator_spec")
res_spec.parameters.qubits = ["q1"]
res_spec.parameters.frequency_span_in_mhz = 20.0
res_spec.parameters.frequency_step_in_mhz = 0.01
res_spec.parameters.readout_power_dbm = -10
res_spec.parameters.num_shots = 200
res_spec.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 14:31:10,178 - qm - INFO     - Performing health check
2026-06-30 14:31:10,188 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 14:31:10,409 - qm - INFO     - Opened quantum machine with id: QM-5bfd60a7-541e-4414-8230-4849b4b4a395
2026-06-30 14:31:10,409 - qm - INFO     - Opening QM
2026-06-30 14:31:10,419 - qm - INFO     - Clearing queue
2026-06-30 14:31:10,429 - qm - INFO     - Adding program to queue.
2026-06-30 14:31:10,610 - qm - INFO     - Program added to queue. Job id: b6cf73d1-0f2e-4121-950c-ebd463a2d935
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 7.10s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 7.15s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 7.21s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:219: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='resonator_spec', description="\n        1D RESONATOR SPECTROSCOPY\nThis sequence involves measuring the resonator by sending a readout pulse and demodulating the signals to extract the\n'I' and 'Q' quadratures across varying readout intermediate frequencies for all the active qubits.\nThe data is then post-processed to determine the resonator resonance frequency.\nThis frequency is used to update the readout frequency in the state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the readout line (node 01a_mixer_calibration.py).\n    - Having calibrated the time of flight, offsets, and gains (node 01a_time_of_flight.py).\n    - Having initialized the QUAM state parameters for the readout pulse amplitude and duration, and the resonators depletion time.\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency\n", create

#### 4c. Resonator punch-out (optimal readout power)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

punch_out = library.nodes["02e_resonator_punch_out"].copy(name="resonator_punch_out")
punch_out.parameters.qubits = ["q1"]
punch_out.parameters.frequency_span_in_mhz = 150.0
punch_out.parameters.frequency_step_in_mhz = 1
punch_out.parameters.min_power_dbm = -40
punch_out.parameters.max_power_dbm = 0
punch_out.parameters.num_power_points = 10
punch_out.parameters.max_amp = 0.1
punch_out.parameters.num_shots = 100
punch_out.parameters.frequency_shift_threshold_in_hz = 0.1e6
punch_out.parameters.use_adaptive_span = False
punch_out.parameters.sweep_left_offset_mhz = 1.0
punch_out.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 14:32:01,090 - qm - INFO     - Performing health check
2026-06-30 14:32:01,090 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 14:32:01,320 - qm - INFO     - Opened quantum machine with id: QM-54aa36a2-2836-43b8-a2a1-0a490dc908fb
2026-06-30 14:32:01,320 - qm - INFO     - Opening QM
2026-06-30 14:32:01,320 - qm - INFO     - Clearing queue
2026-06-30 14:32:01,331 - qm - INFO     - Adding program to queue.
2026-06-30 14:32:01,582 - qm - INFO     - Program added to queue. Job id: c8f77923-97c9-4387-ba54-4ebbd9f8cd2b
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 15.85s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 15.91s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 15.95s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02e_resonator_punch_out.py:333: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='resonator_punch_out', description="\n        RESONATOR PUNCH-OUT SPECTROSCOPY\nThis sequence characterizes the resonator response as a function of readout power\nin order to detect power-induced shifts of the resonator frequency (punch-out).\nA readout pulse is applied and the demodulated 'I' and 'Q' quadratures are acquired\nfor all resonators simultaneously while sweeping the readout frequency at a small\nnumber of readout power levels.\n\nFor each power level, the resonator frequency is extracted directly from the\nmeasured response. By comparing the resonator frequency at low and high readout\npower, the presence of a power-induced frequency shift is detected. Based on this\nanalysis, an optimal readout power is selected that avoids resonator punch-out while\nmaintaining sufficient signal strength.\n\nPrerequisites:\n    - Having calibrated the resonator frequency at low power\n      (e.g., node 02a_resonator_spectroscopy.py).\n    - Having specified the desire

#### 4c bis. Resonator spectroscopy vs power

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

res_spec_vs_power = library.nodes["02b_resonator_spectroscopy_vs_power"].copy(name="resonator_spectroscopy_vs_power")
res_spec_vs_power.parameters.qubits = ["q0"]
res_spec_vs_power.parameters.max_power_dbm = 0
res_spec_vs_power.parameters.min_power_dbm = -30
res_spec_vs_power.parameters.num_power_points = 10
res_spec_vs_power.parameters.moving_average_filter_window_num_points = 1
res_spec_vs_power.parameters.derivative_smoothing_window_num_points = 1
res_spec_vs_power.parameters.frequency_span_in_mhz = 2
res_spec_vs_power.parameters.frequency_step_in_mhz = 0.01
res_spec_vs_power.parameters.num_shots = 200
res_spec_vs_power.run()

Running action create_qua_program
Setting the Octave gain to 10.0 dB
Setting the readout amplitude to 0.1 V
Action create_qua_program finished
Running action execute_qua_program
2026-06-22 14:14:56,357 - qm - INFO     - Performing health check
2026-06-22 14:14:56,654 - qm - INFO     - Health check passed
2026-06-22 14:14:58,732 - qm - INFO     - Opening QM
2026-06-22 14:14:58,737 - qm - INFO     - Sending program to QOP for compilation
2026-06-22 14:14:58,939 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 3.57s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 3.60s
2026-06-22 14:15:02,759 - qm - INFO     - Closing QM
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Setting the Octave gain to -20 dB
Setting the readout amplitude

c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibration_libs\analysis\feature_detection.py:109: FutureWarning: Reductions are applied along the rolling dimension(s) '['detuning']'. Passing the 'dim' kwarg to reduction operations has no effect.
  rolling = da.rolling({dim: 10}, center=True).mean(dim=dim)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02b_resonator_spectroscopy_vs_power.py:228: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'automatic_calibration'


Action save_results finished


NodeRunSummary(name='resonator_spectroscopy_vs_power', description="\n        RESONATOR SPECTROSCOPY VERSUS READOUT POWER\nThis sequence involves measuring the resonator by sending a readout pulse and\ndemodulating the signals to extract the 'I' and 'Q' quadratures for all resonators\nsimultaneously. This is done across various readout frequencies and amplitudes.\nBased on the results, one can determine if a qubit is coupled to the resonator by\nnoting the resonator frequency splitting. This information can then be used to adjust\nthe readout amplitude, choosing a readout amplitude value just before the observed\nfrequency splitting.\n\nPrerequisites:\n    - Having calibrated the resonator frequency (node 02a_resonator_spectroscopy.py).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency at the optimal readout power: qubit.resonator.f_01 & qubit.resonator.RF_frequency\n    - The readout power: qubit.resonator.se

#### 4d. Resonator spectroscopy at calibrated power

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

res_spec_lp = library.nodes["02a_resonator_spectroscopy"].copy(name="resonator_spec_low_power")
res_spec_lp.parameters.qubits = ["q1"]
res_spec_lp.parameters.frequency_span_in_mhz = 5.0
res_spec_lp.parameters.frequency_step_in_mhz = 0.01
res_spec_lp.parameters.num_shots = 300
res_spec_lp.parameters.readout_power_dbm = -20 #-30
res_spec_lp.parameters.max_amp = 0.1
res_spec_lp.parameters.save_readout_amplitude = True
res_spec_lp.parameters.run_circle_fit = True
res_spec_lp.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 14:33:30,795 - qm - INFO     - Performing health check
2026-06-30 14:33:30,804 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 14:33:31,044 - qm - INFO     - Opened quantum machine with id: QM-d226c455-5915-4e9e-a4d3-c08599bc4d1f
2026-06-30 14:33:31,044 - qm - INFO     - Opening QM
2026-06-30 14:33:31,044 - qm - INFO     - Clearing queue
2026-06-30 14:33:31,054 - qm - INFO     - Adding program to queue.
2026-06-30 14:33:31,256 - qm - INFO     - Program added to queue. Job id: 0413e67f-9552-4491-88b7-57fcee3c1bc7
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 2.67s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 2.71s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 2.77s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:219: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:233: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='resonator_spec_low_power', description="\n        1D RESONATOR SPECTROSCOPY\nThis sequence involves measuring the resonator by sending a readout pulse and demodulating the signals to extract the\n'I' and 'Q' quadratures across varying readout intermediate frequencies for all the active qubits.\nThe data is then post-processed to determine the resonator resonance frequency.\nThis frequency is used to update the readout frequency in the state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the readout line (node 01a_mixer_calibration.py).\n    - Having calibrated the time of flight, offsets, and gains (node 01a_time_of_flight.py).\n    - Having initialized the QUAM state parameters for the readout pulse amplitude and duration, and the resonators depletion time.\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency\

#### 4e. Readout depletion measurement

Measures how long the resonator takes to deplete photons after a readout pulse.
Sweeps the wait time `tau` between a first readout and a Ramsey sequence on the qubit.
Residual photons AC-Stark shift the qubit during the Ramsey idle time; fitting the
exponential decay gives the resonator depletion time constant.
Updates `qubit.resonator.depletion_time` to 3x the fitted time constant.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_depletion = library.nodes["08c_readout_depletion"].copy(name="ro_depletion")
ro_depletion.parameters.qubits = ["q1"]
ro_depletion.parameters.min_wait_time_in_ns = 1000
ro_depletion.parameters.max_wait_time_in_ns = 50_000
ro_depletion.parameters.wait_time_num_points = 10
ro_depletion.parameters.log_or_linear_sweep = "linear"
ro_depletion.parameters.ramsey_idle_time_in_ns = 1000
ro_depletion.parameters.num_shots = 400
ro_depletion.run()

### 5. Transmon ge calibration

#### 5a. Qubit spectroscopy vs power


In [12]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

qubit_spec_vs_power = library.nodes["03c_qubit_spectroscopy_vs_power"].copy(name="qubit_spec_vs_power")
qubit_spec_vs_power.parameters.qubits = ["q1"]
qubit_spec_vs_power.parameters.frequency_span_in_mhz = 200.0
qubit_spec_vs_power.parameters.frequency_step_in_mhz = 1
qubit_spec_vs_power.parameters.min_power_dbm = -60.0
qubit_spec_vs_power.parameters.max_power_dbm = -10.0
qubit_spec_vs_power.parameters.num_power_points = 10
qubit_spec_vs_power.parameters.max_amplitude_opx = 0.24
qubit_spec_vs_power.parameters.min_amplitude_opx = 0.01
qubit_spec_vs_power.parameters.operation = "saturation"
qubit_spec_vs_power.parameters.operation_len_in_ns = 2_000
qubit_spec_vs_power.parameters.linewidth_threshold_hz = 5e6  # select power just below this linewidth
qubit_spec_vs_power.parameters.power_buffer_db = -0.0        # safety margin below threshold power
qubit_spec_vs_power.parameters.num_shots = 200
qubit_spec_vs_power.parameters.use_adaptive_span = False
qubit_spec_vs_power.parameters.signal_source = "I_rot"
# x180/saturation power: P_threshold + 20*log10(operation_len_in_ns / T_pi_target)
# where T_pi_target = rabi_sweep_max_duration_ns / (2 * rabi_target_periods)
qubit_spec_vs_power.parameters.rabi_target_periods = 1       # desired periods in time Rabi sweep
qubit_spec_vs_power.parameters.rabi_sweep_max_duration_ns = 300.0  # upper bound of sweep [ns]
qubit_spec_vs_power.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:04:18,693 - qm - INFO     - Performing health check
2026-07-12 17:04:18,693 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:04:18,873 - qm - INFO     - Opened quantum machine with id: QM-28009854-2398-4ba4-a0ac-1c398878d1e5
2026-07-12 17:04:18,873 - qm - INFO     - Opening QM
2026-07-12 17:04:18,873 - qm - INFO     - Clearing queue
2026-07-12 17:04:18,883 - qm - INFO     - Adding program to queue.
2026-07-12 17:04:19,224 - qm - INFO     - Program added to queue. Job id: 1608ecb6-7bc7-42b3-8290-849e67c68df6
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 24.47s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 24.50s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 24.54s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03c_qubit_spectroscopy_vs_power.py:336: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03c_qubit_spectroscopy_vs_power.py:342: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[q1] Detected qubit frequency: 4.698126 GHz


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03c_qubit_spectroscopy_vs_power.py:349: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='qubit_spec_vs_power', description='\n        QUBIT SPECTROSCOPY VS DRIVE POWER\nThis sequence involves probing the qubit transition by applying an XY drive while sweeping the drive power and\nintermediate frequency around the expected qubit transition for all active qubits.\nThe qubit response is measured via the readout resonator, and the demodulated I/Q signals are post-processed to extract\nthe qubit spectroscopy signal as a function of frequency and drive power.\n\nThe resulting 2D spectroscopy map is analyzed to identify the qubit transition frequency, assess power broadening\nand saturation effects, and select an appropriate drive power for subsequent calibrations.\nA rough estimate of the qubit frequency at the selected drive power is extracted and used to update the qubit state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the XY control line (node 01a_mixer_calibration.py).\n    - Having calibrated the readout chain, includin

#### 5b. Qubit spectroscopy

> **SRF note**: The qubit appears as a **peak**. Set `find_dip=False`.

In [4]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

qubit_spec = library.nodes["03a_qubit_spectroscopy"].copy(name="qubit_spec")
qubit_spec.parameters.qubits = ["q1"]
qubit_spec.parameters.find_dip = False              # qubit appears as peak in reflection readout
qubit_spec.parameters.frequency_span_in_mhz = 200.0
qubit_spec.parameters.frequency_step_in_mhz = 1
qubit_spec.parameters.operation = "saturation"
qubit_spec.parameters.operation_len_in_ns = 20_000
qubit_spec.parameters.operation_amplitude_factor = .1
qubit_spec.parameters.num_shots = 300
qubit_spec.parameters.update_iw_angle = False
qubit_spec.parameters.signal_source = "I"
qubit_spec.run()

Getting calibration path from config
c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 16:52:10,951 - qm - INFO     - Performing health check
2026-07-12 16:52:10,961 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 16:52:11,150 - qm - INFO     - Opened quantum machine with id: QM-488465ac-f540-4f15-814a-3343b3faa8dd
2026-07-12 16:52:11,150 - qm - INFO     - Opening QM
2026-07-12 16:52:11,150 - qm - INFO     - Clearing queue
2026-07-12 16:52:11,160 - qm - INFO     - Adding program to queue.
2026-07-12 16:52:11,481 - qm - INFO     - Program added to queue. Job id: d6ef66e8-6c7a-42da-98d7-fa3269355b89
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 2.85s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 2.89s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 2.93s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03a_qubit_spectroscopy.py:224: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='qubit_spec', description='\n        QUBIT SPECTROSCOPY\nThis sequence involves sending a saturation pulse to the qubit, placing it in a mixed state,\nand then measuring the state of the resonator across various qubit drive frequencies.\nIn order to facilitate the qubit search, the qubit pulse duration and amplitude can be changed manually\nfrom the node parameters.\n\nThe data is post-processed to determine the qubit resonance frequency and the width of the peak.\n\nNote that it can happen that the qubit is excited by the image sideband or LO leakage instead of the desired sideband.\nThis is why calibrating the qubit mixer is highly recommended when using external mixers or the Octave.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The qubit 0->1 

#### 5c. Time Rabi

Find the pi-pulse duration by sweeping the qubit pulse length at fixed amplitude.

In [15]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

time_rabi = library.nodes["04c_time_rabi"].copy(name="time_rabi")
time_rabi.parameters.qubits = ["q1"]
time_rabi.parameters.min_duration_ns = 16
time_rabi.parameters.max_duration_ns = 300
time_rabi.parameters.duration_step_ns = 4
time_rabi.parameters.num_shots = 200
time_rabi.parameters.operation = "saturation"
time_rabi.parameters.operation_amplitude_factor = 1.0
time_rabi.parameters.drive_power_dbm = 10 #-20  # optional: override XY power
time_rabi.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:08:22,325 - qm - INFO     - Performing health check
2026-07-12 17:08:22,325 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:08:22,465 - qm - INFO     - Opened quantum machine with id: QM-6a9efeb1-7ed6-41b2-bd83-054ca26b8d3f
2026-07-12 17:08:22,465 - qm - INFO     - Opening QM
2026-07-12 17:08:22,465 - qm - INFO     - Clearing queue
2026-07-12 17:08:22,475 - qm - INFO     - Adding program to queue.
2026-07-12 17:08:22,875 - qm - INFO     - Program added to queue. Job id: dc3fdcee-219a-4916-9fde-2c61dedf5e19
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 11.46s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 11.50s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 11.54s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04c_time_rabi.py:213: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='time_rabi', description='\n        TIME RABI\nThis sequence plays a qubit drive pulse with variable duration and measures the resonator\nfor different pulse durations.  The result is a Rabi oscillation in the I quadrature from\nwhich the π-pulse duration is extracted.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the qubit drive (node 01a).\n    - Having calibrated the readout (time of flight, offsets, gains).\n    - Having found the qubit frequency (node 03a_qubit_spectroscopy or 03c_qubit_spectroscopy_vs_power).\n\nState update:\n    - The qubit pulse duration for the selected operation:\n      qubit.xy.operations[operation].length  (in nanoseconds)\n    - If drive_power_dbm is set and the fit succeeds, the amplitude override\n      is kept in the state (not reverted). If the fit fails it is reverted.\n', created_at=datetime.datetime(2026, 7, 12, 17, 8, 22, 195084, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 

#### 5d. Power Rabi

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

power_rabi = library.nodes["04b_power_rabi"].copy(name="power_rabi")
power_rabi.parameters.qubits = ["q1"]
power_rabi.parameters.min_amp_factor = 0.001
power_rabi.parameters.max_amp_factor = 1.99
power_rabi.parameters.amp_factor_step = 0.010
power_rabi.parameters.operation = "x180"
power_rabi.parameters.num_shots = 100
power_rabi.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 16:53:17,206 - qm - INFO     - Performing health check
2026-07-12 16:53:17,206 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 16:53:17,387 - qm - INFO     - Opened quantum machine with id: QM-6c61e51d-6dbc-4dde-a4c8-a081b5166ee9
2026-07-12 16:53:17,387 - qm - INFO     - Opening QM
2026-07-12 16:53:17,387 - qm - INFO     - Clearing queue
2026-07-12 16:53:17,387 - qm - INFO     - Adding program to queue.
2026-07-12 16:53:17,847 - qm - INFO     - Program added to queue. Job id: 77e35c7c-0002-4fc4-b4c1-3a6ba6387aa8
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 15.90s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 15.94s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 15.98s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04b_power_rabi.py:235: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='power_rabi', description='\n        POWER RABI WITH ERROR AMPLIFICATION\nThis sequence involves repeatedly executing the qubit pulse (such as x180) \'N\' times and\nmeasuring the state of the resonator across different qubit pulse amplitudes and number of pulses.\nBy doing so, the effect of amplitude inaccuracies is amplified, enabling a more precise measurement of the pi pulse\namplitude. The results are then analyzed to determine the qubit pulse amplitude suitable for the selected duration.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the qubit frequency (node 03a_qubit_spectroscopy.py).\n    - Having set the qubit gates duration (qubit.xy.operations["x180"].length).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The qubit pulse amplitude corresponding to the specified operation (x180, x90...)\n    (qubit.xy.operations[operation].amplitude)

#### 5e. Ramsey (T2*)

In [22]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ramsey = library.nodes["06a_ramsey"].copy(name="ramsey")
ramsey.parameters.qubits = ["q1"]
ramsey.parameters.num_shots = 100
ramsey.parameters.x180_operation = "x180"
ramsey.parameters.frequency_detuning_in_mhz = 2.0
ramsey.parameters.max_wait_time_in_ns = 5_000
ramsey.parameters.wait_time_num_points = 100
ramsey.parameters.log_or_linear_sweep = "linear"
ramsey.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:20:58,006 - qm - INFO     - Performing health check
2026-07-12 17:20:58,006 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:20:58,196 - qm - INFO     - Opened quantum machine with id: QM-97b5c5cf-ca87-4937-b7a2-7e8768cc6126
2026-07-12 17:20:58,196 - qm - INFO     - Opening QM
2026-07-12 17:20:58,196 - qm - INFO     - Clearing queue
2026-07-12 17:20:58,196 - qm - INFO     - Adding program to queue.
2026-07-12 17:20:58,627 - qm - INFO     - Program added to queue. Job id: c9686892-4741-4957-9508-9a33b22d0a5c
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 7.59s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 7.64s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 7.68s
Progress: [#############

C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06a_ramsey.py:261: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='ramsey', description='\n        RAMSEY WITH VIRTUAL Z ROTATIONS\nThe program consists in playing a Ramsey sequence (x90 - idle_time - x90/y90 - measurement) for different idle times.\nInstead of detuning the qubit gates, the frame of the second x90 pulse is rotated (de-phased) to mimic an accumulated\nphase acquired for a given detuning after the idle time.\nThis method has the advantage of playing gates on resonance as opposed to the detuned Ramsey.\n\nFrom the results, one can fit the Ramsey oscillations and precisely measure the qubit resonance frequency and T2*.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desired flux poi

#### 5f. T1 (ge)

In [16]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

T1_ge = library.nodes["05_T1"].copy(name="T1_ge")
T1_ge.parameters.qubits = ["q1"]
T1_ge.parameters.num_shots = 500
T1_ge.parameters.min_wait_time_in_ns = 16
T1_ge.parameters.max_wait_time_in_ns = 450_000
T1_ge.parameters.wait_time_num_points = 100
T1_ge.parameters.log_or_linear_sweep = "linear"
T1_ge.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:10:25,619 - qm - INFO     - Performing health check
2026-07-12 17:10:25,619 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:10:25,809 - qm - INFO     - Opened quantum machine with id: QM-64e3e5b6-78cf-48b4-b6c4-98ce7a822a39
2026-07-12 17:10:25,809 - qm - INFO     - Opening QM
2026-07-12 17:10:25,809 - qm - INFO     - Clearing queue
2026-07-12 17:10:25,819 - qm - INFO     - Adding program to queue.
2026-07-12 17:10:26,119 - qm - INFO     - Program added to queue. Job id: 4795dd80-358e-446f-a366-8b7e86a02a60
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 51.91s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 51.96s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 51.99s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\05_T1.py:218: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='T1_ge', description='\n        T1 MEASUREMENT\nThe sequence consists in putting the qubit in the excited stated by playing the x180 pulse and measuring the resonator\nafter a varying time. The qubit T1 is extracted by fitting the exponential decay of the measured quadratures/state.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The T1 relaxation time: qubit.T1\n', created_at=datetime.datetime(2026, 7, 12, 17, 10, 25, 399372, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(202

#### 5g. T1 Monitor (ge)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

T1_monitor = library.nodes["29_T1_monitor"].copy(name="T1_monitor_ge")
T1_monitor.parameters.qubits = ["q1"]
T1_monitor.parameters.n_iter = 5*50*8
T1_monitor.parameters.num_shots = 200
T1_monitor.parameters.min_wait_time_in_ns = 16
T1_monitor.parameters.max_wait_time_in_ns = 300_000
T1_monitor.parameters.wait_time_num_points = 71
T1_monitor.parameters.log_or_linear_sweep = "linear"
T1_monitor.run()

2026-07-02 11:56:20,567 - qualibrate - INFO - Creating node 31_T1_monitor
2026-07-02 11:56:20,626 - qualibrate - INFO - Copying node with name 31_T1_monitor with parameters name = 'T1_monitor_ge', node_parameters = {}
2026-07-02 11:56:20,636 - qualibrate - INFO - Creating node 31_T1_monitor
2026-07-02 11:56:20,716 - qualibrate - INFO - Run node T1_monitor_ge with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 11:56:20,906 - qm - INFO     - Performing health check
2026-07-02 11:56:20,916 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 11:56:21,076 - qm - INFO     - Opened quantum machine with id: QM-1b2fefc6-898a-47a2-a766-23bd91184d05
2026-07-02 11:56:21,076 - qm - INFO     - Opening QM
2026-07-02 11:56:21,076 - qm - INFO     - Clearing queue
2026-07-02 11:56:21,086 - qm - INFO     - Adding program to queue.
2026-07-02 11:56:21,397 - qm - INFO     - Program added to queue. Job id: 0d94da4d-8aff-491f-82dd-70ab604673d2
2026-07-02 11:56:28,586 - qm - INFO     - Closing QM


2026-07-02 11:56:28,606 - qualibrate - INFO - Node T1_monitor_ge - Iter 1/2000  |  t = 0.1 min  |  q1: T1 = 127.3 µs


2026-07-02 11:56:28,776 - qm - INFO     - Opened quantum machine with id: QM-1a591c97-824d-4fc4-ad97-8cd56e70af3c
2026-07-02 11:56:28,776 - qm - INFO     - Opening QM
2026-07-02 11:56:28,786 - qm - INFO     - Clearing queue
2026-07-02 11:56:28,786 - qm - INFO     - Adding program to queue.
2026-07-02 11:56:29,077 - qm - INFO     - Program added to queue. Job id: cfcd546f-ff01-4c50-995a-009bcc32d117
2026-07-02 11:56:33,585 - qm - INFO     - Closing QM


2026-07-02 11:56:34,157 - qualibrate - INFO - Node T1_monitor_ge - Iter 2/2000  |  t = 0.2 min  |  q1: T1 = 126.5 µs


2026-07-02 11:56:34,277 - qm - INFO     - Opened quantum machine with id: QM-4ad6ff5a-b02f-4138-95df-778f73f3314e
2026-07-02 11:56:34,277 - qm - INFO     - Opening QM
2026-07-02 11:56:34,287 - qm - INFO     - Clearing queue
2026-07-02 11:56:34,297 - qm - INFO     - Adding program to queue.
2026-07-02 11:56:34,579 - qm - INFO     - Program added to queue. Job id: d385f944-2596-474f-a819-ad5f55b66fc4
2026-07-02 11:56:41,816 - qm - INFO     - Closing QM


2026-07-02 11:56:41,836 - qualibrate - INFO - Node T1_monitor_ge - Iter 3/2000  |  t = 0.3 min  |  q1: T1 = 203.5 µs


2026-07-02 11:56:41,986 - qm - INFO     - Opened quantum machine with id: QM-cdab8118-078c-4bf4-a436-6192e59f0ab0
2026-07-02 11:56:41,986 - qm - INFO     - Opening QM
2026-07-02 11:56:41,986 - qm - INFO     - Clearing queue
2026-07-02 11:56:41,996 - qm - INFO     - Adding program to queue.
2026-07-02 11:56:42,266 - qm - INFO     - Program added to queue. Job id: 648d0bd9-0bb9-4d14-9c15-207d9717d4f3


#### 5e. Spin echo (T2)

In [19]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

echo = library.nodes["06b_echo"].copy(name="T2_echo")
echo.parameters.qubits = ["q1"]
echo.parameters.num_shots = 200
echo.parameters.max_wait_time_in_ns = 50_000
echo.parameters.min_wait_time_in_ns = 100
echo.parameters.wait_time_num_points = 100
echo.parameters.log_or_linear_sweep = "linear"
echo.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:17:30,973 - qm - INFO     - Performing health check
2026-07-12 17:17:30,973 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:17:31,134 - qm - INFO     - Opened quantum machine with id: QM-73abd46a-9d88-4dd9-a66e-3258a2c46209
2026-07-12 17:17:31,134 - qm - INFO     - Opening QM
2026-07-12 17:17:31,134 - qm - INFO     - Clearing queue
2026-07-12 17:17:31,143 - qm - INFO     - Adding program to queue.
2026-07-12 17:17:31,827 - qm - INFO     - Program added to queue. Job id: 92c4793d-c8a7-4cdb-86b0-c6da35578a46
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 8.60s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 8.64s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 8.68s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06b_echo.py:208: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='T2_echo', description='\n        T2 echo MEASUREMENT\nThe sequence consists in playing an echo sequence (x90 - idle_time - x180 - idle_time - -x90 - measurement) for \ndifferent idle times.\nThe qubit T2 echo is extracted by fitting the exponential decay of the measured quadratures/state.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the qubit frequency precisely (node 06a_ramsey.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nNext steps before going to the next node:\n    - Update the qubit T2 echo: qubit.T2echo.\n', created_at=datetime.datetime(2026, 7, 12, 17, 17, 30, 722879, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 7, 12, 17, 17, 41, 205784, tzinfo=datetime.timezone(datetime.timedelt

#### 5f. IQ blobs

In [18]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

iq_blobs = library.nodes["07_iq_blobs"].copy(name="iq_blobs")
iq_blobs.parameters.qubits = ["q1"]
iq_blobs.parameters.num_shots = 4000
# iq_blobs.parameters.ge_pi_pulse = "x180_selective"
iq_blobs.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:17:23,329 - qm - INFO     - Performing health check
2026-07-12 17:17:23,339 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:17:23,519 - qm - INFO     - Opened quantum machine with id: QM-6163d6d0-e28e-4d37-8248-600fa63b3d9d
2026-07-12 17:17:23,519 - qm - INFO     - Opening QM
2026-07-12 17:17:23,519 - qm - INFO     - Clearing queue
2026-07-12 17:17:23,534 - qm - INFO     - Adding program to queue.
2026-07-12 17:17:24,011 - qm - INFO     - Program added to queue. Job id: 1168d9ee-f9f4-49f7-920d-4c3f39b31dba
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.06s
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.12s
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.17s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\07_iq_blobs.py:236: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='iq_blobs', description='\n        IQ BLOBS\nThis sequence involves measuring the state of the resonator \'N\' times, first after thermalization (with the qubit in\nthe |g> state) and then after applying a x180 (pi) pulse to the qubit (bringing the qubit to the |e> state).\nThe resulting IQ blobs are displayed, and the data is processed to determine:\n    - The rotation angle required for the integration weights, ensuring that the\n      separation between |g> and |e> states aligns with the \'I\' quadrature.\n    - The threshold along the \'I\' quadrature for effective qubit state discrimination (at the center between the two blobs).\n    - The repeat-until-success threshold along the \'I\' quadrature for effective active reset (at the center of the |g> blob).\n    - The readout confusion matrix, which is also influenced by the x180 pulse fidelity.\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated

#### Readout optimization

##### 5g. Readout frequency optimization

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_freq_opt = library.nodes["08a_readout_frequency_optimization"].copy(name="readout_freq_opt")
ro_freq_opt.parameters.qubits = ["q1"]
ro_freq_opt.parameters.frequency_span_in_mhz = 20.0
ro_freq_opt.parameters.frequency_step_in_mhz = 0.05
ro_freq_opt.parameters.num_shots = 200
ro_freq_opt.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 12:00:56,282 - qm - INFO     - Performing health check
2026-07-02 12:00:56,292 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 12:00:56,472 - qm - INFO     - Opened quantum machine with id: QM-886af46c-c130-4d85-9268-9117bcf2fc29
2026-07-02 12:00:56,472 - qm - INFO     - Opening QM
2026-07-02 12:00:56,472 - qm - INFO     - Clearing queue
2026-07-02 12:00:56,482 - qm - INFO     - Adding program to queue.
2026-07-02 12:00:56,892 - qm - INFO     - Program added to queue. Job id: 57799ede-015c-45a7-a069-d1f887cbc392
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 60.62s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 60.68s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 60.75s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08a_readout_frequency_optimization.py:227: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='readout_freq_opt', description="\n        READOUT OPTIMISATION: FREQUENCY\nThe sequence consists in measuring the state of the resonator after thermalization (qubit in |g>) and after\nplaying a pi pulse to the qubit (qubit in |e>) successively while sweeping the readout frequency.\nThe 'I' & 'Q' quadratures when the qubit is in |g> and |e> are extracted to derive the readout fidelity.\nThe optimal readout frequency is chosen as to maximize the state discrimination Signal-to-Noise Ratio (SNR).\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency\n    - The dispersive shift: qubit.chi\n", created_at=datetime.datetime(2026, 7, 2, 12, 0, 56, 122193, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), '

##### 5h. Readout length optimization

Finds the optimal readout pulse duration by maximising g/e discrimination fidelity.
Uses accumulated demodulation: IQ is accumulated in 16 ns chunks within a single pulse,
yielding fidelity vs cumulative readout length in one experiment.
Updates `qubit.resonator.operations["readout"].length`.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_length_opt = library.nodes["08d_readout_length_optimization"].copy(name="ro_length_opt")
ro_length_opt.parameters.qubits = ["q1"]
ro_length_opt.parameters.num_shots = 2000
ro_length_opt.parameters.max_readout_length_in_ns = 60_000
ro_length_opt.parameters.division_length_in_ns = 160
ro_length_opt.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 12:04:14,843 - qm - INFO     - Performing health check
2026-07-02 12:04:14,847 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 12:04:15,044 - qm - INFO     - Opened quantum machine with id: QM-458fab88-8c5e-4864-9a37-fa75e16086f4
2026-07-02 12:04:15,044 - qm - INFO     - Opening QM
2026-07-02 12:04:15,044 - qm - INFO     - Clearing queue
2026-07-02 12:04:15,054 - qm - INFO     - Adding program to queue.
2026-07-02 12:04:20,646 - qm - INFO     - Program added to queue. Job id: 62dcfa5b-7a5b-451f-b95a-54a79b125977
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 4.86s
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 5.00s
2026-07-02 12:04:25,866 - qm - INFO     - Closing QM
Action execute_qua_program finished
Running action analyse_data
Action an

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08d_readout_length_optimization.py:272: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='ro_length_opt', description='\n        READOUT LENGTH OPTIMIZATION\n\nFinds the optimal readout pulse duration by maximising the g/e state discrimination fidelity.\n\nUses accumulated demodulation: within a single readout pulse the IQ signal is accumulated\nin chunks of `division_length_in_cc` clock cycles (= 4 ns each). For each shot both the\nground state (after thermalization) and the excited state (after x180) are measured. The\ntwo-state discriminator is applied at each cumulative length to compute fidelity vs time.\nThe readout pulse length is then updated to the length that gives the highest fidelity.\n\nNote: integration weight names ("rotated_cos", "rotated_sin", "rotated_minus_sin") can be\nadjusted via parameters if your readout operation uses different names.\n\nPrerequisites:\n    - Calibrated readout frequency and power (nodes 08a, 08b).\n    - Calibrated x180 pulse (node 04b or 04c).\n    - Calibrated IQ rotation angle (node 07_iq_blobs).\n\nState up

##### 5h. Readout power optimization

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_pwr_opt = library.nodes["08b_readout_power_optimization"].copy(name="readout_power_opt")
ro_pwr_opt.parameters.qubits = ["q1"]
ro_pwr_opt.parameters.num_shots = 2000
ro_pwr_opt.parameters.start_amp = 0.01
ro_pwr_opt.parameters.end_amp = 1.5
ro_pwr_opt.parameters.num_amps = 10
ro_pwr_opt.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 12:04:27,739 - qm - INFO     - Performing health check
2026-07-02 12:04:27,739 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 12:04:27,970 - qm - INFO     - Opened quantum machine with id: QM-db64928d-9f3c-44a9-b056-0c8214bcee05
2026-07-02 12:04:27,970 - qm - INFO     - Opening QM
2026-07-02 12:04:27,970 - qm - INFO     - Clearing queue
2026-07-02 12:04:27,979 - qm - INFO     - Adding program to queue.
2026-07-02 12:04:28,450 - qm - INFO     - Program added to queue. Job id: d765d0c0-b2c5-456e-ae99-f800359f5dcc
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.08s
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.17s
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.25s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08b_readout_power_optimization.py:220: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='readout_power_opt', description='\n        READOUT POWER OPTIMIZATION\nThe sequence consists in measuring the state of the resonator after thermalization (qubit in |g>) and after\nplaying a pi pulse to the qubit (qubit in |e>) successively while sweeping the readout amplitude.\nThe \'I\' & \'Q\' quadratures when the qubit is in |g> and |e> are extracted to derive the readout fidelity.\nThe optimal readout amplitude is chosen as to maximize the readout fidelity.\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n\nState update:\n    - The readout amplitude: qubit.resonator.operations["readout"].amplitude\n    - The integration weight angle: qubit.resonator.operations["readout"].integration_weights_angle\n    - the ge discrimination threshold: qubit.resonator.operations["readout"].threshold\n    - the Repeat Un

##### 5h. Readout frequency × amplitude optimization (2D)

Jointly sweeps the readout frequency and amplitude to find the point of maximum single-shot readout fidelity.
Combines `08a_readout_frequency_optimization` and `08b_readout_power_optimization` into a single 2D scan.
Updates `qubit.resonator.f_01`/`RF_frequency`, `qubit.resonator.operations["readout"].amplitude`, the
discrimination threshold/angle, and the confusion matrix.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_freq_amp_opt = library.nodes["08e_readout_frequency_amplitude_optimization"].copy(name="readout_freq_amp_opt")
ro_freq_amp_opt.parameters.qubits = ["q1"]
ro_freq_amp_opt.parameters.num_shots = 1000
ro_freq_amp_opt.parameters.frequency_span_in_mhz = 6.0
ro_freq_amp_opt.parameters.frequency_num_points = 30
ro_freq_amp_opt.parameters.min_amp_factor = 0.05
ro_freq_amp_opt.parameters.max_amp_factor = 1.5
ro_freq_amp_opt.parameters.num_amps = 5
ro_freq_amp_opt.parameters.outliers_threshold = 0.98
ro_freq_amp_opt.run()

Failed to run node readout_freq_amp_opt
Traceback (most recent call last):
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\qualibration_node.py", line 895, in run
    self.run_node_file(self.filepath)
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\qualibration_node.py", line 944, in run_node_file
    _module = import_from_path(get_module_name(node_filepath), node_filepath)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\utils\read_files.py", line 20, in import_from_path
    return import_from_path_importlib(module_name, file_path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\utils\read_files.py", line 33, in import_from_path_importlib
    spec.loader.exec_module(module)
  File "<frozen importlib._bo

Running action create_qua_program


Exception: The spacing of the input array must be even in linear or logarithmic scales. Please use `for_each_()` for arbitrary scans.

#### 5i. DRAG calibration

In [ ]:
# -- DRAG calibration (10b) --------------------------------------------------
from qualibrate import QualibrationLibrary

library = QualibrationLibrary.get_active_library()
drag_calib = library.nodes["10b_drag_calibration_180_minus_180"].copy(name="drag_calibration")

drag_calib.parameters.qubits = ["q1"]
drag_calib.parameters.operation = "x180"
drag_calib.parameters.num_shots = 100
drag_calib.parameters.min_amp_factor = 0.0
drag_calib.parameters.max_amp_factor = 1.01
drag_calib.parameters.amp_factor_step = 0.01
drag_calib.parameters.max_number_pulses_per_sweep = 25
drag_calib.parameters.smooth_sigma = 1.2
drag_calib.parameters.use_state_discrimination = True
drag_calib.parameters.alpha_setpoint = 1.0   # ← fixes the collapsed x-axis

drag_calib.run()


#### Selective Power Rabi

Calibrates the amplitude of a narrow-bandwidth `selective_x180` **DragCosinePulse**.
The pulse length controls frequency selectivity: bandwidth  1/T (e.g. 10 us -> ~100 kHz).

Run the **populate** cell first to write the DragCosinePulse to `state.json`, then run the **rabi** cell to calibrate its amplitude.

**State update**: `operations["selective_x180"].amplitude`, `operations["selective_x180"].length`.

In [ ]:
# Add selective_x180 (DragGaussianPulse) to q1.xy if not already present.
# Parameters are derived from the standard x180 pulse:
#   - length    : set by selective_length_ns below
#   - amplitude : inversely proportional to length  (area = amplitude * length = const)
#   - sigma     : always length / 5
#
# Only inserts missing keys - all other state values are left unchanged.
from quam_config import Quam
from quam.components.pulses import DragGaussianPulse

machine = Quam.load()
xy = machine.qubits["q1"].xy

# Parameters
selective_length_ns = 2000   # adjust as needed

# Read the reference x180 values
x180 = xy.operations["x180"]
x180_length    = x180.length      # ns
x180_amplitude = x180.amplitude   # V

# Scale amplitude inversely with length (constant pulse area -> same rotation angle)
selective_amplitude = x180_amplitude * (x180_length / selective_length_ns)
selective_sigma     = selective_length_ns / 5

print(f"x180 reference : length={x180_length} ns, amplitude={x180_amplitude:.6f} V")
print(f"selective_x180 : length={selective_length_ns} ns, amplitude={selective_amplitude:.6f} V, sigma={selective_sigma:.0f} ns")

if "selective_x180" not in xy.operations:
    xy.operations["selective_x180"] = DragGaussianPulse(
        length=selective_length_ns,
        amplitude=selective_amplitude,
        sigma=selective_sigma,
        alpha=0.0,
        anharmonicity=x180.anharmonicity,
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    print("selective_x180 added")
else:
    print("selective_x180 already exists - skipped (delete it first to re-add)")
    sel = xy.operations["selective_x180"]
    print(f"  current: length={sel.length} ns, amplitude={sel.amplitude:.6f} V, sigma={sel.sigma} ns")

machine.save()
print("State saved.")


In [23]:
from qualibrate import QualibrationLibrary
# from calibration_utils.dd_protected_selective.parameters import DDProtectedGateParameters
library = QualibrationLibrary.get_active_library()

selective_rabi = library.nodes["04b_power_rabi"].copy(name="selective_power_rabi")
selective_rabi.parameters.qubits = ["q1"]
selective_rabi.parameters.operation = "selective_x180"
# selective_rabi.parameters.operation_length_in_ns = 4_000  # 10 us -> ~100 kHz bandwidth
selective_rabi.parameters.min_amp_factor = 0.001
selective_rabi.parameters.max_amp_factor = 1.99
selective_rabi.parameters.amp_factor_step = 0.01
selective_rabi.parameters.num_shots = 200
# selective_rabi.parameters.use_dd_protection = False
# selective_rabi.parameters.dd_gate_params = dd_params
selective_rabi.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:24:17,346 - qm - INFO     - Performing health check
2026-07-12 17:24:17,356 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:24:17,506 - qm - INFO     - Opened quantum machine with id: QM-e9417938-e4f5-43f4-ba4b-9c5c6ce77dd9
2026-07-12 17:24:17,506 - qm - INFO     - Opening QM
2026-07-12 17:24:17,506 - qm - INFO     - Clearing queue
2026-07-12 17:24:17,516 - qm - INFO     - Adding program to queue.
2026-07-12 17:24:17,906 - qm - INFO     - Program added to queue. Job id: 45456436-9d8e-4e5f-973f-d111d177f481
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 15.35s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 15.39s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 15.44s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04b_power_rabi.py:235: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='selective_power_rabi', description='\n        POWER RABI WITH ERROR AMPLIFICATION\nThis sequence involves repeatedly executing the qubit pulse (such as x180) \'N\' times and\nmeasuring the state of the resonator across different qubit pulse amplitudes and number of pulses.\nBy doing so, the effect of amplitude inaccuracies is amplified, enabling a more precise measurement of the pi pulse\namplitude. The results are then analyzed to determine the qubit pulse amplitude suitable for the selected duration.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the qubit frequency (node 03a_qubit_spectroscopy.py).\n    - Having set the qubit gates duration (qubit.xy.operations["x180"].length).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The qubit pulse amplitude corresponding to the specified operation (x180, x90...)\n    (qubit.xy.operations[operation].

#### Selective Ramsey (T2* with selective pi-pulse)

Runs a Ramsey experiment using the `selective_x180` pulse (amplitude alpha ~ 0.5 for x90).
Because the selective pulse probes a narrow frequency window, this gives a cleaner
frequency calibration when cavity photons are present (PNRS context).

State update: updates `selective_x180.detuning` with the frequency correction instead
of the qubit's global `f_01` / `T2ramsey`, preserving the fast-pulse calibration.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

sel_ramsey = library.nodes["06a_ramsey"].copy(name="selective_ramsey")
sel_ramsey.parameters.qubits = ["q1"]
sel_ramsey.parameters.num_shots = 200
sel_ramsey.parameters.x180_operation = "selective_x180"
sel_ramsey.parameters.frequency_detuning_in_mhz = 1.0
sel_ramsey.parameters.max_wait_time_in_ns = 5_000
sel_ramsey.parameters.wait_time_num_points = 100
sel_ramsey.parameters.log_or_linear_sweep = "linear"
sel_ramsey.parameters.selective_state_update = True
sel_ramsey.parameters.correct_with_pulse_detuning = False
sel_ramsey.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-05 21:12:21,295 - qm - INFO     - Performing health check
2026-07-05 21:12:21,295 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-05 21:12:21,476 - qm - INFO     - Opened quantum machine with id: QM-d0d2215d-e92e-4492-9796-d6eb2a65e23e
2026-07-05 21:12:21,476 - qm - INFO     - Opening QM
2026-07-05 21:12:21,485 - qm - INFO     - Clearing queue
2026-07-05 21:12:21,493 - qm - INFO     - Adding program to queue.
2026-07-05 21:12:22,035 - qm - INFO     - Program added to queue. Job id: 20a568a7-a055-4c4a-bb76-aefd86e22a55
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 32.75s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 32.79s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 32.83s
Progress: [##########

C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06a_ramsey.py:261: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='selective_ramsey', description='\n        RAMSEY WITH VIRTUAL Z ROTATIONS\nThe program consists in playing a Ramsey sequence (x90 - idle_time - x90/y90 - measurement) for different idle times.\nInstead of detuning the qubit gates, the frame of the second x90 pulse is rotated (de-phased) to mimic an accumulated\nphase acquired for a given detuning after the idle time.\nThis method has the advantage of playing gates on resonance as opposed to the detuned Ramsey.\n\nFrom the results, one can fit the Ramsey oscillations and precisely measure the qubit resonance frequency and T2*.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desire

#### Dispersive shift (chi)

Measures chi = f_resonator(|e>) - f_resonator(|g>) and sets the optimal readout frequency at the mid-point for maximum contrast.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

disp_shift = library.nodes["20_dispersive_shift"].copy(name="dispersive_shift")
disp_shift.parameters.qubits = ["q1"]
disp_shift.parameters.num_shots = 200
disp_shift.parameters.frequency_span_in_mhz = 30.0    # total span of the frequency sweep [MHz]
disp_shift.parameters.frequency_step_in_mhz = 0.05   # frequency step size [MHz]
disp_shift.parameters.min_dip_contrast = 0.05         # minimum contrast to declare a dip found
disp_shift.parameters.lo_leakage_exclusion_mhz = 10.0 # frequency window around LO to exclude [MHz]
disp_shift.parameters.fit_on_phase = True
disp_shift.run()

### 6. Transmon ef calibration

#### 6a. Qubit spectroscopy ef

> Reflection readout -> set `find_dip=True` here too.

In [ ]:
# Add EF pulse operations to q1.xy if not already present.
# Only inserts the missing keys - all other state values are left unchanged.
from quam_config import Quam
from quam.components.pulses import DragGaussianPulse

machine = Quam.load()
xy = machine.qubits["q1"].xy

if "EF_x180" not in xy.operations:
    xy.operations["EF_x180"] = DragGaussianPulse(
        length=40,
        amplitude=0.1,
        sigma=8,
        alpha=0.0,
        anharmonicity=-200e6,
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    print("EF_x180 added")
else:
    print("EF_x180 already exists - skipped")

if "EF_x90" not in xy.operations:
    xy.operations["EF_x90"] = DragGaussianPulse(
        length="#../EF_x180/length",
        amplitude=0.05,
        sigma="#../EF_x180/sigma",
        alpha="#../EF_x180/alpha",
        anharmonicity="#../EF_x180/anharmonicity",
        detuning="#../EF_x180/detuning",
        subtracted="#../EF_x180/subtracted",
        axis_angle=0,
        digital_marker="#../EF_x180/digital_marker",
    )
    print("EF_x90 added")
else:
    print("EF_x90 already exists - skipped")

machine.save()
print("State saved.")

EF_x180 already exists — skipped
EF_x90 already exists — skipped
State saved.


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

qubit_spec_ef = library.nodes["12_qubit_spectroscopy_EF"].copy(name="qubit_spec_ef")
qubit_spec_ef.parameters.qubits = ["q1"]
qubit_spec_ef.parameters.find_dip = False
qubit_spec_ef.parameters.frequency_span_in_mhz = 200.0
qubit_spec_ef.parameters.frequency_step_in_mhz = 1
qubit_spec_ef.parameters.operation_len_in_ns = 2_000
qubit_spec_ef.parameters.operation = "saturation"
qubit_spec_ef.parameters.operation_amplitude_factor = 1.0
qubit_spec_ef.parameters.signal_source = "I"
qubit_spec_ef.parameters.num_shots = 100
qubit_spec_ef.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-04 01:02:58,293 - qm - INFO     - Performing health check
2026-07-04 01:02:58,303 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-04 01:02:58,465 - qm - INFO     - Opened quantum machine with id: QM-5cf5e853-c40d-4e93-ba50-33cabe7da200
2026-07-04 01:02:58,465 - qm - INFO     - Opening QM
2026-07-04 01:02:58,465 - qm - INFO     - Clearing queue
2026-07-04 01:02:58,474 - qm - INFO     - Adding program to queue.
2026-07-04 01:02:59,056 - qm - INFO     - Program added to queue. Job id: 436e80b2-73ed-423a-b432-8f71f594cdb7
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 31.48s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 31.53s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 31.58s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_Qubit_Spectroscopy_E_to_F.py:225: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='qubit_spec_ef', description='\n        QUBIT SPECTROSCOPY E TO F\nThis sequence involves preparing the excited state then sending a saturation pulse to the qubit around its e->f transition,\nand then measuring the state of the resonator across various qubit drive frequencies.\nIn order to facilitate the qubit search, the qubit pulse duration and amplitude can be changed manually\nfrom the node parameters.\n\nThe data is post-processed to determine the qubit second transition resonance frequency.\n\nNote that it can happen that the qubit is excited by the image sideband or LO leakage instead of the desired sideband.\nThis is why calibrating the qubit mixer is highly recommended when using external mixers or the Octave.\n\nPrerequisites:\n    - Having calibrated single qubit gates.\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n\nState update:\n    - The qubit e->f frequency: qubit.anharmonicity.\n', created_at=datetime.datetime(2026, 7

#### 6b. Time Rabi ef
Find the EF pi-pulse duration by sweeping the EF drive pulse length.
A ge x180 prepares |e> before the EF drive, and a final ge x180 improves readout fidelity.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

time_rabi_ef = library.nodes["04d_time_rabi_ef"].copy(name="time_rabi_ef")
time_rabi_ef.parameters.qubits = ["q1"]
time_rabi_ef.parameters.min_duration_ns = 16
time_rabi_ef.parameters.max_duration_ns = 200
time_rabi_ef.parameters.duration_step_ns = 4
time_rabi_ef.parameters.num_shots = 200
time_rabi_ef.parameters.operation_amplitude_factor = 1.0
time_rabi_ef.parameters.ef_x180_operation = "EF_x180"
time_rabi_ef.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 15:14:21,374 - qm - INFO     - Performing health check
2026-06-30 15:14:21,384 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 15:14:21,694 - qm - INFO     - Opened quantum machine with id: QM-1338a913-12ae-4e1f-8656-1fc6f0388af6
2026-06-30 15:14:21,704 - qm - INFO     - Opening QM
2026-06-30 15:14:21,704 - qm - INFO     - Clearing queue
2026-06-30 15:14:21,714 - qm - INFO     - Adding program to queue.
2026-06-30 15:14:22,657 - qm - INFO     - Program added to queue. Job id: 1fc4a839-e18c-4208-bc81-2c091c9b7d1a
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 5.92s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 5.97s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 6.02s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04d_time_rabi_ef.py:208: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
Failed to run node time_rabi_ef
Traceback (most recent call last):
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\qualibration_node.py", line 895, in run
    self.run_node_file(self.filepath)
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\qualibration_node.py", line 944, in run_node_file
    _module = import_from_path(get_module_name(node_filepath), node_filepath)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibrate\core\utils\read_files.py", line 20, in import_from_path
    return import_from_path_importlib(module_name, file_path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File

Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state


ValueError: Cannot set attribute length to 92 because it is a reference. To overwrite the reference, set the attribute to None first.
Object: DragGaussianPulse(length=152, id=None, digital_marker='ON', axis_angle=0, amplitude=0.20647407089326122, sigma=30, alpha=0.0, anharmonicity=-200000000, detuning=0.0, subtracted=True)
Original value: #../x180/length

#### 6c. Power Rabi ef

In [25]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

power_rabi_ef = library.nodes["13_power_rabi_ef"].copy(name="power_rabi_ef")
power_rabi_ef.parameters.qubits = ["q1"]
power_rabi_ef.parameters.min_amp_factor = 0.001
power_rabi_ef.parameters.max_amp_factor = 1.9
power_rabi_ef.parameters.amp_factor_step = 0.01
power_rabi_ef.parameters.num_shots = 200
power_rabi_ef.parameters.use_state_discrimination = False
power_rabi_ef.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-12 17:30:18,958 - qm - INFO     - Performing health check
2026-07-12 17:30:18,958 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-12 17:30:19,108 - qm - INFO     - Opened quantum machine with id: QM-9a4487b7-fcd9-4d46-b11f-21d972d7b291
2026-07-12 17:30:19,108 - qm - INFO     - Opening QM
2026-07-12 17:30:19,108 - qm - INFO     - Clearing queue
2026-07-12 17:30:19,123 - qm - INFO     - Adding program to queue.
2026-07-12 17:30:19,549 - qm - INFO     - Program added to queue. Job id: 0d2eba57-e7b9-402b-b072-da152417e7a0
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 27.98s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 28.01s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 28.05s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\13_power_rabi_ef.py:246: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='power_rabi_ef', description='\n        EF POWER RABI CALIBRATION\nThis node calibrates the pi pulse operation between the |e> and |f> states of a superconducting\nqubit by populating the |e> state with a previously calibrated pi pulse and applying a varying\namplitude detuned pulse at the |e> -> |f> transition frequency.\nPrerequisites:\n        - Having calibrated a pi pulse operation between the |g> and |e> states of the qubit (x180).\n            (04_power_rabi.py)\n        - Having calibrated the readout resonator dispersive shift (chi).\n            (08a_readout_frequency_optimization.py)\n        - Having calibrated the qubit anharmonicity.\n\nState update:\n        - The qubit pulse amplitude corresponding to the EF_x180 operation\n            (qubit.xy.operations["EF_x180"].amplitude).\n', created_at=datetime.datetime(2026, 7, 12, 17, 30, 18, 798394, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at

#### 6d. Ramsey ef
Refines the EF transition frequency (corrects `q.anharmonicity`) and measures EF T2* via a virtual-Z Ramsey sequence on the ef transition.

In [9]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ramsey_ef = library.nodes["06b_ramsey_ef"].copy(name="ramsey_ef")
ramsey_ef.parameters.qubits = ["q1"]
ramsey_ef.parameters.frequency_detuning_in_mhz = 2.0
ramsey_ef.parameters.min_wait_time_in_ns = 16
ramsey_ef.parameters.max_wait_time_in_ns = 5_000
ramsey_ef.parameters.wait_time_num_points = 100
ramsey_ef.parameters.log_or_linear_sweep = "linear"
ramsey_ef.parameters.num_shots = 200
ramsey_ef.parameters.ef_x180_operation = "EF_x180"
ramsey_ef.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 01:05:58,625 - qm - INFO     - Performing health check
2026-07-14 01:05:58,625 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 01:05:58,825 - qm - INFO     - Opened quantum machine with id: QM-f31b063e-0d74-413e-8073-e41039d9d21d
2026-07-14 01:05:58,825 - qm - INFO     - Opening QM
2026-07-14 01:05:58,825 - qm - INFO     - Clearing queue
2026-07-14 01:05:58,835 - qm - INFO     - Adding program to queue.
2026-07-14 01:05:59,285 - qm - INFO     - Program added to queue. Job id: 6dc81ca0-b528-4a99-930a-2f9ad9f58b4e
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.64s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.67s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.72s
Progress: [##########

C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06b_ramsey_ef.py:231: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'



Running action update_state
Action update_state finished
Running action save_results
Action save_results finished


NodeRunSummary(name='ramsey_ef', description='\n        EF RAMSEY WITH VIRTUAL Z ROTATIONS\nThe program prepares the qubit in |e⟩ via a ge π-pulse, then performs a Ramsey sequence\non the e→f transition: x90_ef – idle_time – x90_ef (with virtual detuning applied via\nframe rotation).  A final ge π-pulse is applied before readout to maximise readout contrast.\n\nThe EF Ramsey oscillation frequency is used to precisely determine the EF transition\nfrequency (i.e., correct the anharmonicity stored in the QUAM state), and the decay\nenvelope gives the EF coherence time T2*_ef.\n\nThe virtual detuning is applied symmetrically (± frequency_detuning_in_mhz) to\ndisambiguate the sign of the frequency correction.\n\nPrerequisites:\n    - Having calibrated the ge x180 and x90 pulses (nodes 03a, 04b/04c).\n    - Having run qubit EF spectroscopy to set q.anharmonicity (node 12).\n    - (optional) Having calibrated a dedicated x90_ef operation for better EF pi/2 pulses.\n\nState update:\n    - The 

#### 6e. T1 of |f level

Measures the decay time of the second excited state (|f  |e relaxation).
Prepares |f via ge x180 + EF_x180, waits a variable idle time, then applies a final ge x180 before readout.

**State update**: `qubit.T1_ef`

> Prerequisite: calibrated `EF_x180` pulse (node 6c/6d).

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

T1_f = library.nodes["05b_T1_ef"].copy(name="T1_f")
T1_f.parameters.qubits = ["q1"]
T1_f.parameters.ef_x180_operation = "EF_x180"
T1_f.parameters.num_shots = 500
T1_f.parameters.min_wait_time_in_ns = 16
T1_f.parameters.max_wait_time_in_ns = 1_000_000
T1_f.parameters.wait_time_num_points = 100
T1_f.parameters.log_or_linear_sweep = "linear"
T1_f.run()

2026-07-02 13:42:45,644 - qualibrate - INFO - Creating node 05b_T1_ef
2026-07-02 13:42:45,734 - qualibrate - INFO - Copying node with name 05b_T1_ef with parameters name = 'T1_f', node_parameters = {}
2026-07-02 13:42:45,744 - qualibrate - INFO - Creating node 05b_T1_ef
2026-07-02 13:42:45,814 - qualibrate - INFO - Run node T1_f with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 13:42:46,084 - qm - INFO     - Performing health check
2026-07-02 13:42:46,094 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 13:42:46,274 - qm - INFO     - Opened quantum machine with id: QM-285ec5a6-12be-43f8-a76e-627b3c880214
2026-07-02 13:42:46,274 - qm - INFO     - Opening QM
2026-07-02 13:42:46,284 - qm - INFO     - Clearing queue
2026-07-02 13:42:46,294 - qm - INFO     - Adding program to queue.
2026-07-02 13:42:46,756 - qm - INFO     - Program added to queue. Job id: 503cdc52-b3ef-47ea-b1c7-58f094258e62
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 99.35s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 99.39s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 99.44s
Progress: [##########

2026-07-02 13:44:26,941 - qualibrate - INFO - Node T1_f - Execution report for job 503cdc52-b3ef-47ea-b1c7-58f094258e62
No errors


Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 99.78s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 99.83s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 99.88s
2026-07-02 13:44:26,951 - qm - INFO     - Closing QM


2026-07-02 13:44:26,992 - qualibrate - INFO - Node T1_f - T1_ef for qubit q1: 178.28 ± 2.94 µs --> SUCCESS!


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\05b_T1_ef.py:200: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-02 13:44:27,081 - qualibrate - INFO - Saving node T1_f to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-07-02 13:44:27,235 - qualibrate - INFO - Saving machine state to db
2026-07-02 13:44:27,251 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-02 13:44:27,251 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-02 13:44:27,277 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-02\#303_T1_f_134427\quam_state


Action save_results finished


NodeRunSummary(name='T1_f', description='\n        T1_ef MEASUREMENT\nThe sequence prepares the qubit in |f⟩ via two consecutive pi pulses (ge x180 then EF_x180),\nwaits a variable idle time, and then applies a final ge x180 before readout to improve\nreadout fidelity.  The exponential decay of the measured quadrature gives the |f⟩ lifetime T1_ef.\n\nThe signal decays from the f-state level (short t) to the e-state level (long t, |f⟩ → |e⟩\nrelaxation dominates).  The final ge x180 before readout maps |e⟩ → |g⟩ to exploit the\nbest-contrast readout state.\n\nPrerequisites:\n    - Having calibrated the ge x180 pulse (nodes 03a, 04b/04c).\n    - Having calibrated the EF_x180 pulse (node 13_power_rabi_ef).\n\nState update:\n    - The |f⟩ relaxation time: qubit.T1_ef\n', created_at=datetime.datetime(2026, 7, 2, 13, 42, 45, 814334, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 7, 2, 13, 44, 27, 287828, tz

#### Readout optimization

##### 6f. GEF readout frequency optimization

Sweeps the readout IF around the current point while preparing the qubit in |g, |e,
and |f. Finds the frequency that maximises the minimum centroid separation between all
three state pairs. Updates `qubit.resonator.GEF_frequency_shift`.

> Prerequisite: `EF_x180` operation calibrated (nodes 6b/6c).

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_freq_opt = library.nodes["14_gef_frequency_optimization"].copy(name="gef_freq_opt")
gef_freq_opt.parameters.qubits = ["q1"]
gef_freq_opt.parameters.num_shots = 200
gef_freq_opt.parameters.frequency_span_in_mhz = 4.0
gef_freq_opt.parameters.frequency_step_in_mhz = 0.1
gef_freq_opt.run()

Getting calibration path from config
c:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-04 19:54:59,808 - qm - INFO     - Performing health check
2026-07-04 19:54:59,808 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-04 19:55:00,008 - qm - INFO     - Opened quantum machine with id: QM-eaa704c9-013e-4fce-9a29-fe4cf5be4e4c
2026-07-04 19:55:00,008 - qm - INFO     - Opening QM
2026-07-04 19:55:00,008 - qm - INFO     - Clearing queue
2026-07-04 19:55:00,025 - qm - INFO     - Adding program to queue.
2026-07-04 19:55:00,753 - qm - INFO     - Program added to queue. Job id: 7733cffe-c1e1-4f0f-ab33-1d569e2d3124
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 0.08s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 0.16s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 0.24s
Progress: [#############

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\14_gef_readout_frequency_optimization.py:283: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='gef_freq_opt', description="\n        G-E-F READOUT FREQUENCY OPTIMIZATION\nThis sequence sweeps the readout resonator intermediate frequency around the current operating point while preparing\nthe qubit successively in |g>, |e>, and |f> states. For every tested detuning, three IQ blobs (g, e, f) are acquired.\nThe distances between the three centroids are computed and fitted to identify the optimal frequency shift that\nmaximizes simultaneous separation (e.g. maximizes the minimum of {d_ge, d_ef, d_gf}). The resulting optimal detuning\nis then added to the stored `GEF_frequency_shift` parameter.\n\nPurpose:\n    - Optimize a single readout frequency for high-fidelity three-level (g/e/f) state discrimination\n        (including leakage monitoring).\n    - Improve discrimination robustness against slow frequency drifts or residual mis-calibration.\n\nMeasurement flow:\n    1. For each qubit, loop over the readout frequency detuning values.\n    2. For every detuning

##### 6f-ii. GEF readout power optimisation

Sweeps the readout pulse amplitude for all three qubit states (|g>, |e>, |f>) at the
GEF-optimised readout frequency (set by node 14). Computes
 vs amplitude and picks the maximum.

**State update**:  = optimal amplitude

> Prerequisites: GEF readout frequency calibrated (node 14); ge + EF -pulses calibrated.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_power_opt = library.nodes["14b_readout_gef_power_optimization"].copy(name="gef_power_opt")
gef_power_opt.parameters.qubits = ["q1"]
gef_power_opt.parameters.num_shots = 1000
gef_power_opt.parameters.min_amp_factor = 0.1
gef_power_opt.parameters.max_amp_factor = 1.9
gef_power_opt.parameters.num_amps = 11
gef_power_opt.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 14:25:08,763 - qm - INFO     - Performing health check
2026-07-02 14:25:08,763 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 14:25:08,993 - qm - INFO     - Opened quantum machine with id: QM-d3d1c1ab-e2e7-467e-9731-c89fd3c7473d
2026-07-02 14:25:08,993 - qm - INFO     - Opening QM
2026-07-02 14:25:09,003 - qm - INFO     - Clearing queue
2026-07-02 14:25:09,003 - qm - INFO     - Adding program to queue.
2026-07-02 14:25:09,718 - qm - INFO     - Program added to queue. Job id: 4509a32e-5b04-4615-b5af-43111b36c154
Progress: [##################################################] 100.0% (n=1000/1000) --> elapsed time: 49.25s
Progress: [##################################################] 100.0% (n=1000/1000) --> elapsed time: 49.34s
Progress: [##################################################] 100.0% (n=1000/1000) --> elapsed time: 49.43s
Progress: [####

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\14b_readout_gef_power_optimization.py:217: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='gef_power_opt', description='\n        GEF READOUT POWER OPTIMISATION\nSweeps the readout pulse amplitude while preparing the qubit successively in |g⟩, |e⟩,\nand |f⟩ at the current GEF readout frequency.  For each amplitude the three IQ centroids\nare measured and the pairwise distances D_ge, D_ef, D_gf are computed.  The discrimination\nmetric is min(D_ge, D_ef, D_gf) — the worst-case separation.  The amplitude that maximises\nthis metric is selected as the new readout amplitude.\n\nMeasurement flow:\n    1. Sweep readout amplitude from min_amp_factor × A_0 to max_amp_factor × A_0.\n    2. For each amplitude acquire averaged IQ for |g⟩, |e⟩, and |f⟩.\n       - |g⟩: idle thermalization then readout\n       - |e⟩: ge x180 then readout\n       - |f⟩: ge x180 + EF_x180 then readout\n    3. Compute Distance = min(D_ge, D_ef, D_gf) vs amplitude.\n    4. Find the optimum and update the machine state.\n\nPrerequisites:\n    - GEF readout frequency calibrated (node 14).\n

##### 6f-iii. GEF readout length optimisation

Sweeps the cumulative readout integration time (via accumulated demodulation) for all
three qubit states (|g>, |e>, |f>) at the GEF-optimised frequency and power.
Computes  at each cumulative length and finds the optimum.

**State update**:  = optimal length [ns]

> Prerequisites: GEF readout frequency (node 14) and power (node 14b) calibrated;
> ge + EF -pulses calibrated; integration weight names match parameters (default: iw1/iw2/iw3).

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_length_opt = library.nodes["14c_readout_gef_length_optimization"].copy(name="gef_length_opt")
gef_length_opt.parameters.qubits = ["q1"]
gef_length_opt.parameters.num_shots = 2000
gef_length_opt.parameters.max_readout_length_in_ns = 80_000
gef_length_opt.parameters.division_length_in_ns = 160
# gef_length_opt.parameters.cos_weight_name = "iw1"   # adjust if your integration weights differ
# gef_length_opt.parameters.sin_weight_name = "iw2"
# gef_length_opt.parameters.minus_sin_weight_name = "iw3"
gef_length_opt.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-02 14:35:17,305 - qm - INFO     - Performing health check
2026-07-02 14:35:17,315 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-02 14:35:17,495 - qm - INFO     - Opened quantum machine with id: QM-4be26e75-67ad-4665-a63d-04fdddccbd91
2026-07-02 14:35:17,495 - qm - INFO     - Opening QM
2026-07-02 14:35:17,495 - qm - INFO     - Clearing queue
2026-07-02 14:35:17,505 - qm - INFO     - Adding program to queue.
2026-07-02 14:35:29,588 - qm - INFO     - Program added to queue. Job id: e6d3f549-c71f-499c-9091-1242d2dee787
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 9.65s
2026-07-02 14:35:39,911 - qm - INFO     - Closing QM
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action up

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\14c_readout_gef_length_optimization.py:296: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='gef_length_opt', description='\n        GEF READOUT LENGTH OPTIMISATION\nFinds the optimal readout pulse duration for three-level (g/e/f) state discrimination\nby maximising min(D_ge, D_ef, D_gf) as a function of cumulative integration time.\n\nUses accumulated demodulation: within a single readout pulse the IQ signal is averaged\nin chunks of `division_length_in_ns` nanoseconds. For each shot all three states are\nmeasured (idle → |g⟩, x180 → |e⟩, x180+EF_x180 → |f⟩). The averaged IQ centroids\nat each cumulative length are used to compute the three pairwise distances; the\nworst-case metric min(D_ge, D_ef, D_gf) is maximised to find the optimum.\n\nThe readout operates at the GEF-optimised frequency (qubit.resonator.GEF_frequency_shift).\n\nPrerequisites:\n    - GEF readout frequency calibrated (node 14).\n    - GEF readout power calibrated (node 14b).\n    - ge and EF π-pulses calibrated (nodes 04b, 13).\n    - Integration weight names match parameters (default:

##### 6g. GEF IQ blobs

Captures single-shot IQ blobs for all three states (|g, |e, |f) at the optimised
GEF readout frequency. Plots IQ distributions and confusion matrix.
Updates `qubit.resonator.gef_centers` and `qubit.resonator.gef_confusion_matrix`.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_blobs = library.nodes["15_iq_blobs_gef"].copy(name="gef_blobs")
gef_blobs.parameters.qubits = ["q1"]
gef_blobs.parameters.num_shots = 4000
gef_blobs.parameters.ge_pi_pulse = "selective_x180"
gef_blobs.parameters.ef_pi_pulse = "selective_EF_x180"
gef_blobs.parameters.operation = "readout"  # or "readout_QND"
gef_blobs.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-06 12:21:38,653 - qm - INFO     - Performing health check
2026-07-06 12:21:38,663 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-06 12:21:38,813 - qm - INFO     - Opened quantum machine with id: QM-0c8636f7-5f3f-435e-bff0-7c904d65db8e
2026-07-06 12:21:38,813 - qm - INFO     - Opening QM
2026-07-06 12:21:38,813 - qm - INFO     - Clearing queue
2026-07-06 12:21:38,813 - qm - INFO     - Adding program to queue.
2026-07-06 12:21:39,603 - qm - INFO     - Program added to queue. Job id: 6c931cae-528c-4457-9715-48943d3a7a44
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.09s
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.17s
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.26s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\15_iq_blobs_gef.py:258: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='gef_blobs', description="\n        IQ BLOBS GEF\nThis sequence involves measuring the state of the resonator 'N' times, first after thermalization (with the qubit in\nthe |g> state), then after applying a x180 (pi) pulse to the qubit (bringing the qubit to the |e> state) and finally\nafter applying a x180 (pi) pulse plus an EF_180 pulse (bringing the qubit to the |f> state).\nThe resulting IQ blobs are displayed, and the data is processed to determine:\n    - The centers of the |g>, |e> and |f> state IQ blobs.\n    - The readout confusion matrix, which is also influenced by the x180 and EF_180 pulses fidelities.\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters.\n    - Having calibrated the qubit EF_180 pulse parameters.\n\nState update:\n    - qubit.resonator.gef_centers (3×2 blob centres in raw ADC units, for readout_state_gef())\n    - qubit.gef_rotation_angle, 

#### 6h. Qubit thermal population (RPM)

Measures the qubit thermal population P_th by comparing two EF Rabi sweeps:
- **'g' sweep** (from |g>): ge_  ef(a)  ef_  ge_  readout    A_g
- **'e' sweep** (from thermal): ef(a)  ge_  readout    A_e

P_th = A_e / (A_e + A_g).  Reports effective qubit temperature.  No state update.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

rpm = library.nodes["20_qubit_rpm"].copy(name="qubit_rpm")
rpm.parameters.qubits = ["q1"]
rpm.parameters.num_shots = 500
rpm.parameters.min_amp_factor = -2.0
rpm.parameters.max_amp_factor = 1.99  # 2 full periods (np.arange stops before 4.0)
rpm.parameters.amp_factor_step = 0.02
rpm.run()


#### 6i. Qubit Coherence Monitor (T1, T2\*, T2 echo, Omega_f, P_th)

Repeatedly runs five experiments per iteration -> T1 decay, Ramsey (T2\* + qubit frequency offset), Hahn echo (T2), and two RPM sweeps (from |g> and from thermal) to monitor long-timescale drift in qubit coherence and thermal population.

Each iteration records T1, T2\*, T2 echo [ms], qubit frequency offset [kHz], and thermal population P_th [%]. All raw and fit data are saved as xr.Datasets (.h5). No QUAM state is updated.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

monitor = library.nodes["32_qubit_coherence_monitor"].copy(name="qubit_coherence_monitor")
monitor.parameters.qubits = ["q1"]
monitor.parameters.n_iter = 120
monitor.parameters.num_shots = 200
# T1 sweep
monitor.parameters.min_wait_time_in_ns = 16
monitor.parameters.max_wait_time_in_ns = 300_000
monitor.parameters.wait_time_num_points = 71
monitor.parameters.log_or_linear_sweep = "linear"
# Ramsey (T2* + qubit frequency offset) sweep
# monitor.parameters.ramsey_min_wait_time_in_ns = 16
monitor.parameters.ramsey_max_wait_time_in_ns = 15_000
monitor.parameters.ramsey_wait_time_num_points = 300
monitor.parameters.ramsey_log_or_linear_sweep = "linear"
monitor.parameters.ramsey_frequency_detuning_in_mhz = 1.0
# T2 echo (Hahn echo) sweep
monitor.parameters.echo_min_wait_time_in_ns = 100
monitor.parameters.echo_max_wait_time_in_ns = 50_000
monitor.parameters.echo_wait_time_num_points = 100
monitor.parameters.echo_log_or_linear_sweep = "linear"
# RPM sweep (thermal population)
monitor.parameters.min_amp_factor = -2.0
monitor.parameters.max_amp_factor = 1.99
monitor.parameters.amp_factor_step = 0.02
monitor.run()

#### 6i. EF Rabi RPM -- f-state preparation check

Calibrates the EF pi-pulse amplitude using a back-swap readout scheme (no GEF readout required).  The signal P(a) = cos(pi*a/2) has a minimum at a=1 (correct  pulse).

**Sequence**: ge_  ef(a)  ef_ (back-swap)  ge_  readout

**State update**: `q1.xy.operations["EF_x180"].amplitude`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ef_rpm = library.nodes["20b_ef_rabi_rpm"].copy(name="ef_rabi_rpm")
ef_rpm.parameters.qubits = ["q1"]
ef_rpm.parameters.num_shots = 50
ef_rpm.parameters.min_amp_factor = 0.0
ef_rpm.parameters.max_amp_factor = 1.99
ef_rpm.parameters.amp_factor_step = 0.02
ef_rpm.parameters.use_state_discrimination = True
ef_rpm.run()


#### GEF Dispersive shift (chi_ge, chi_ef)

Sweeps the readout resonator frequency for all three qubit states |g, |e, |f.
Fits a Lorentzian dip to each spectrum and extracts:
- chi_ge = f_resonator(|e) - f_resonator(|g)
- chi_ef = f_resonator(|f) - f_resonator(|e)

Sets the readout frequency to f_resonator(|e) and updates , .

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

disp_shift_gef = library.nodes["20b_dispersive_shift_gef"].copy(name="dispersive_shift_gef")
disp_shift_gef.parameters.qubits = ["q1"]
disp_shift_gef.parameters.num_shots = 100
disp_shift_gef.parameters.frequency_span_in_mhz = 40.
disp_shift_gef.parameters.frequency_step_in_mhz = 0.05
disp_shift_gef.parameters.fit_on_phase = True
disp_shift_gef.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-04 01:32:48,452 - qm - INFO     - Performing health check
2026-07-04 01:32:48,462 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-04 01:32:48,672 - qm - INFO     - Opened quantum machine with id: QM-a4fad65a-297f-47fe-9426-f11bf3c38d5a
2026-07-04 01:32:48,672 - qm - INFO     - Opening QM
2026-07-04 01:32:48,672 - qm - INFO     - Clearing queue
2026-07-04 01:32:48,672 - qm - INFO     - Adding program to queue.
2026-07-04 01:32:49,473 - qm - INFO     - Program added to queue. Job id: a79cbf18-859f-48ed-8bcc-a48fea37eb22
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 125.81s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 125.86s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 125.90s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\20b_dispersive_shift_gef.py:217: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='dispersive_shift_gef', description='\n        GEF DISPERSIVE SHIFT MEASUREMENT\nThis node measures all three resonator frequencies by sweeping the readout\nresonator frequency in three conditions:\n  1. Qubit in |g⟩ (thermal / after reset)\n  2. Qubit in |e⟩ (after x180 pulse)\n  3. Qubit in |f⟩ (after x180 + EF_x180 pulses)\n\nEach spectrum is fitted with a Lorentzian dip. The extracted quantities are:\n  chi_ge = f_resonator(|e⟩) - f_resonator(|g⟩)\n  chi_ef = f_resonator(|f⟩) - f_resonator(|e⟩)\n\nThe optimal readout frequency is set to f_resonator(|e⟩), which gives maximum\ndiscrimination contrast between |g⟩ and |e⟩.\n\nPrerequisites:\n    - Calibrated resonator (nodes 02a/02b).\n    - Calibrated x180 pulse (node 04b).\n    - Calibrated EF_x180 pulse (node 13).\n\nState updates:\n    - qubit.resonator.RF_frequency → f_resonator(|e⟩).\n    - qubit.chi    (if attribute exists) → chi_ge [Hz].\n    - qubit.chi_ef (if attribute exists) → chi_ef [Hz].\n', created_at

#### Selective EF Power Rabi

Calibrates the amplitude of a narrow-bandwidth `selective_EF_x180` **DragGaussianPulse**.
The pulse is long (default 4 μs) so its bandwidth is narrow (~100 kHz), enabling
photon-number-resolved operations on the e→f transition.

Prerequisites:
    - Calibrated EF_x180 pulse (nodes 12, 13).

State update: `qubit.xy.operations["selective_EF_x180"].amplitude`

In [ ]:
# Add selective_EF_x180 (DragGaussianPulse) to q1.xy if not already present.
# length and sigma are QUAM references to selective_x180, so they track it automatically.
# Only the amplitude needs to be computed from EF_x180.
#
# Only inserts missing keys - all other state values are left unchanged.
from quam_config import Quam
from quam.components.pulses import DragGaussianPulse

machine = Quam.load()
xy = machine.qubits["q1"].xy

# Read reference values
ef_x180    = xy.operations["EF_x180"]
sel_x180   = xy.operations["selective_x180"]
sel_length = sel_x180.length      # ns (used only for amplitude scaling)

ef_x180_amplitude   = ef_x180.amplitude   # V
selective_ef_amplitude = ef_x180_amplitude * (ef_x180.length / sel_length)

print(f"EF_x180 reference  : length={ef_x180.length} ns, amplitude={ef_x180_amplitude:.6f} V")
print(f"selective_EF_x180  : length=selective_x180/length (ref), amplitude={selective_ef_amplitude:.6f} V, sigma=selective_x180/sigma (ref)")

if "selective_EF_x180" not in xy.operations:
    xy.operations["selective_EF_x180"] = DragGaussianPulse(
        length="#../selective_x180/length",
        amplitude=selective_ef_amplitude,
        sigma="#../selective_x180/sigma",
        alpha=0.0,
        anharmonicity=ef_x180.anharmonicity,
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    print("selective_EF_x180 added")
else:
    print("selective_EF_x180 already exists - skipped (delete it first to re-add)")
    sel = xy.operations["selective_EF_x180"]
    print(f"  current: amplitude={sel.amplitude:.6f} V")

machine.save()
print("State saved.")

EF_x180 reference  : length=40 ns, amplitude=0.119993 V
selective_EF_x180  : length=selective_x180/length (ref), amplitude=0.001200 V, sigma=selective_x180/sigma (ref)
selective_EF_x180 already exists - skipped (delete it first to re-add)
  current: amplitude=0.002305 V
State saved.


In [2]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

selective_ef_rabi = library.nodes["13_power_rabi_ef"].copy(name="selective_EF_power_rabi")
selective_ef_rabi.parameters.qubits = ["q1"]
selective_ef_rabi.parameters.operation = "selective_ef_x180"
# selective_ef_rabi.parameters.operation_length_in_ns = None
selective_ef_rabi.parameters.min_amp_factor = 0.001
selective_ef_rabi.parameters.max_amp_factor = 1.99
selective_ef_rabi.parameters.amp_factor_step = 0.01
selective_ef_rabi.parameters.num_shots = 200
selective_ef_rabi.run()

2026-07-14 10:49:20,640 - qualibrate - INFO - Creating node 13_power_rabi_ef
2026-07-14 10:49:20,708 - qualibrate - INFO - Copying node with name 13_power_rabi_ef with parameters name = 'selective_EF_power_rabi', node_parameters = {}
2026-07-14 10:49:20,708 - qualibrate - INFO - Creating node 13_power_rabi_ef
2026-07-14 10:49:20,778 - qualibrate - INFO - Run node selective_EF_power_rabi with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 10:49:20,918 - qm - INFO     - Performing health check
2026-07-14 10:49:20,918 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 10:49:21,108 - qm - INFO     - Opened quantum machine with id: QM-5c8fa13c-692c-49f3-bdc4-6060eb8a6e8b
2026-07-14 10:49:21,118 - qm - INFO     - Opening QM
2026-07-14 10:49:21,118 - qm - INFO     - Clearing queue
2026-07-14 10:49:21,118 - qm - INFO     - Adding program to queue.
2026-07-14 10:49:21,339 - qm - INFO     - Program added to queue. Job id: 0be0e611-4714-4b4a-a7fa-c2c0619a0427
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.42s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.47s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.51s
Progress: [##########

2026-07-14 10:50:07,525 - qualibrate - INFO - Node selective_EF_powe... - Execution report for job 0be0e611-4714-4b4a-a7fa-c2c0619a0427
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.72s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.76s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.80s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.84s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.88s
2026-07-14 10:50:07,535 - qm - INFO     - Closing QM


2026-07-14 10:50:07,565 - qualibrate - INFO - Node selective_EF_powe... - Results for qubit q1:  SUCCESS!
The calibrated selective_ef_x180 amplitude: 2.63 mV (x1.16)
 Rabi periods in sweep: 0.95
 Residual chi2: 0.015
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\13_power_rabi_ef.py:248: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-14 10:50:07,665 - qualibrate - INFO - Saving node selective_EF_power_rabi to local storage


Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-07-14 10:50:07,902 - qualibrate - INFO - Saving machine state to db
2026-07-14 10:50:07,909 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-14 10:50:07,911 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-14 10:50:07,920 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-14\#864_selective_EF_power_rabi_105007\quam_state


Action save_results finished


NodeRunSummary(name='selective_EF_power_rabi', description='\n        EF POWER RABI CALIBRATION\nThis node calibrates the pi pulse operation between the |e> and |f> states of a superconducting\nqubit by populating the |e> state with a previously calibrated pi pulse and applying a varying\namplitude detuned pulse at the |e> -> |f> transition frequency.\nPrerequisites:\n        - Having calibrated a pi pulse operation between the |g> and |e> states of the qubit (x180).\n            (04_power_rabi.py)\n        - Having calibrated the readout resonator dispersive shift (chi).\n            (08a_readout_frequency_optimization.py)\n        - Having calibrated the qubit anharmonicity.\n\nState update:\n        - The qubit pulse amplitude corresponding to the EF_x180 operation\n            (qubit.xy.operations["EF_x180"].amplitude).\n', created_at=datetime.datetime(2026, 7, 14, 10, 49, 20, 778982, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), co

#### Selective EF Ramsey (T2* with selective EF pi-pulse)

Runs a Ramsey experiment using the `selective_EF_x180` pulse (amplitude_scale=0.5 for the x90).
Because the selective pulse probes a narrow EF frequency window, this gives a cleaner
frequency calibration when cavity photons are present.

State update: updates `selective_EF_x180.detuning` with the frequency correction instead
of the global `q.anharmonicity`, preserving the fast-pulse calibration.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

sel_ef_ramsey = library.nodes["06b_ramsey_ef"].copy(name="selective_EF_ramsey")
sel_ef_ramsey.parameters.qubits = ["q1"]
sel_ef_ramsey.parameters.num_shots = 200
sel_ef_ramsey.parameters.ef_x180_operation = "selective_EF_x180"
sel_ef_ramsey.parameters.frequency_detuning_in_mhz = 1.0
sel_ef_ramsey.parameters.max_wait_time_in_ns = 5_000
sel_ef_ramsey.parameters.wait_time_num_points = 100
sel_ef_ramsey.parameters.log_or_linear_sweep = "linear"
sel_ef_ramsey.parameters.selective_state_update = True
sel_ef_ramsey.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-06 11:37:54,181 - qm - INFO     - Performing health check
2026-07-06 11:37:54,181 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-06 11:37:54,411 - qm - INFO     - Opened quantum machine with id: QM-4b5df5e6-dbef-457d-b4db-34e74bf350b8
2026-07-06 11:37:54,411 - qm - INFO     - Opening QM
2026-07-06 11:37:54,411 - qm - INFO     - Clearing queue
2026-07-06 11:37:54,421 - qm - INFO     - Adding program to queue.
2026-07-06 11:37:55,094 - qm - INFO     - Program added to queue. Job id: 20d8b955-d062-4732-8450-19b5992b00af
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 63.99s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 64.04s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 64.09s
Progress: [##########

C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06b_ramsey_ef.py:231: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='selective_EF_ramsey', description='\n        EF RAMSEY WITH VIRTUAL Z ROTATIONS\nThe program prepares the qubit in |e⟩ via a ge π-pulse, then performs a Ramsey sequence\non the e→f transition: x90_ef – idle_time – x90_ef (with virtual detuning applied via\nframe rotation).  A final ge π-pulse is applied before readout to maximise readout contrast.\n\nThe EF Ramsey oscillation frequency is used to precisely determine the EF transition\nfrequency (i.e., correct the anharmonicity stored in the QUAM state), and the decay\nenvelope gives the EF coherence time T2*_ef.\n\nThe virtual detuning is applied symmetrically (± frequency_detuning_in_mhz) to\ndisambiguate the sign of the frequency correction.\n\nPrerequisites:\n    - Having calibrated the ge x180 and x90 pulses (nodes 03a, 04b/04c).\n    - Having run qubit EF spectroscopy to set q.anharmonicity (node 12).\n    - (optional) Having calibrated a dedicated x90_ef operation for better EF pi/2 pulses.\n\nState update:\n

### 7. Sideband calibration for active cavity cooling

Calibrates the `|f, k⟩ ↔ |g, k+1⟩` red sideband transitions for *k* = 0 … *max_fock_level* − 1, enabling active photon-number cooling of the Alice cavity.

**Calibration order is sequential**: each level *k* uses the previously calibrated pulses for levels 0 … *k* − 1 to prepare the Fock state `|k⟩` before sweeping the *k* → *k* + 1 sideband transition.

**Nodes used**:
- `26_fNgN1_spectroscopy` – finds the resonance frequency of the `|f, k⟩ ↔ |g, k+1⟩` sideband transition.
- `28b_fNgN1_time_rabi` – calibrates the π-pulse duration for the `|f, k⟩ ↔ |g, k+1⟩` sideband transition.
- `28c_fNgN1_ramsey` – refines the sideband transition frequency with high precision.

**State updates** are stored in:
- `cavity_transmon_pairs["q1_alice"].extras["f{k}g{k+1}_RF_frequency"]`
- `sideband_drive.operations["f{k}g{k+1}_pi"].length`


#### f0g1 - |0> -> |1> sideband

##### Spectroscopy

Sweeps the sideband drive frequency while the qubit is in |f> and the cavity is in
Fock |0> (prepared using already-calibrated sideband pulses).

**State update**: `pair.extras["f0g1_RF_frequency"]`

In [8]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26_fNgN1_spectroscopy"].copy(name="f0g1_spectroscopy")
node.parameters.qubits                        = ["q1"]
node.parameters.mode_name                     = "alice"
node.parameters.fock_level                    = 0
node.parameters.frequency_span_in_mhz         = 5.0
node.parameters.frequency_step_in_mhz         = .05
node.parameters.operation                     = "sideband_flat_top"
node.parameters.operation_len_in_ns           = None #10000
node.parameters.operation_amplitude_factor    = 1
node.parameters.num_shots                     = 100
node.parameters.cavity_reset_type             = "thermal"
node.parameters.use_state_discrimination      = True
node.parameters.use_confusion_matrix_correction = False
node.parameters.use_theoretical_frequency_estimate = False
node.parameters.use_gaussian_fit              = False
node.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-13 17:27:41,052 - qm - INFO     - Performing health check
2026-07-13 17:27:41,052 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-13 17:27:41,222 - qm - INFO     - Opened quantum machine with id: QM-263096b3-658f-4afa-874e-db50beb80ead
2026-07-13 17:27:41,232 - qm - INFO     - Opening QM
2026-07-13 17:27:41,232 - qm - INFO     - Clearing queue
2026-07-13 17:27:41,232 - qm - INFO     - Adding program to queue.
2026-07-13 17:27:41,534 - qm - INFO     - Program added to queue. Job id: 58ee3cdd-504d-41fd-801b-9f9b433a5615
2026-07-13 17:30:08,739 - qm - INFO     - Closing QM         ] 74.0% (n=74/100) --> elapsed time: 144.97s
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state_node
Action update_state_node finished
Running action save_resul

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26_fNgN1_spectroscopy.py:327: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='f0g1_spectroscopy', description='\n        SIDEBAND SPECTROSCOPY - generalised to any |n⟩ → |n+1⟩ transition\n\nSweeps the sideband drive frequency while the qubit is prepared in |f⟩ and the\ncavity is prepared in Fock |fock_level⟩.\n\nWhen the sideband drive is resonant, the |f, n⟩ â†" |g, n+1⟩ transition is driven,\nthe qubit is left in |g⟩, and the back-swap π_ef leaves it in |g⟩ → DIP in state\nmeasurement.\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity |j+1⟩, qubit back to |g⟩.\n  2. π_ge  →  |e⟩\n  3. π_ef  →  |f⟩\n  4. Sweep f{k}g{k+1} sideband IF;  play saturation/long pulse.\n  5. π_ef  (back-swap)\n  6. Measure qubit state.\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - For fock_level > 0: calibrated sideband pulses for transitions 0…fock_level-1\n      stored in pair.transitions["f{j}g{j+1}"].pi_flat_top_length_ns.\n\nSta

##### Time Rabi

Sweeps the |0>->|1> sideband drive duration to calibrate the  π-pulse length.

**State update**: `sideband_drive.operations["f0g1_pi"].length`

In [7]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26b_fNgN1_time_rabi"].copy(name="f0g1_time_rabi")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 0
node.parameters.min_duration_ns            = 16
node.parameters.max_duration_ns            = 5_000
node.parameters.duration_step_ns           = 80
node.parameters.ramp_length_ns             = 200  # ns, multiple of 4
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.parameters.use_confusion_matrix_correction = False
node.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-13 17:25:24,036 - qm - INFO     - Performing health check
2026-07-13 17:25:24,046 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-13 17:25:24,228 - qm - INFO     - Opened quantum machine with id: QM-2166a797-ac0d-44eb-892d-d1e5b770a5fb
2026-07-13 17:25:24,228 - qm - INFO     - Opening QM
2026-07-13 17:25:24,228 - qm - INFO     - Clearing queue
2026-07-13 17:25:24,240 - qm - INFO     - Adding program to queue.
2026-07-13 17:25:27,584 - qm - INFO     - Program added to queue. Job id: 0d6d6191-91f8-49a9-9d0b-30d6e20ca522
2026-07-13 17:26:59,723 - qm - INFO     - Closing QM         ] 74.0% (n=74/100) --> elapsed time: 90.69s
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26b_fNgN1_time_rabi.py:272: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='f0g1_time_rabi', description='\n        SIDEBAND TIME RABI - generalised to any |n⟩ → |n+1⟩ transition\n\nSweeps the sideband drive duration while the qubit is in |f⟩ and the cavity in\nFock |fock_level⟩.  A Rabi-like oscillation is observed; the π-pulse duration is\nextracted from the first minimum of the fitted sinusoid.\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity in |j+1⟩, qubit in |g⟩.\n  2. π_ge  →  |e⟩\n  3. π_ef  →  |f⟩\n  4. Play f{k}g{k+1} sideband pulse with swept duration.\n  5. π_ef  (back-swap)\n  6. Measure qubit state.\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - Calibrated sideband frequency for this transition (node 26, fock_level=k).\n    - For fock_level > 0: calibrated sideband pulses for transitions 0…fock_level-1.\n\nState update:\n    - cavity_transmon_pairs["{qubit}_{mode}"].sideband_drive.operations["f{

##### Ramsey

Performs a Ramsey fringe experiment on the f0g1 sideband to refine its
resonance frequency. The artificial detuning (1 MHz default) ensures visible
fringes even when close to resonance.

**State update**: `pair.extras["f0g1_RF_frequency"]` (fine-tuned)

In [6]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26c_fNgN1_ramsey"].copy(name="f0g1_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                = 0
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.artificial_detuning_hz     = 0.5e6
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-13 15:11:46,335 - qm - INFO     - Performing health check
2026-07-13 15:11:46,345 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-13 15:11:46,505 - qm - INFO     - Opened quantum machine with id: QM-a85fbf35-30ea-46dc-b249-1892bcb44061
2026-07-13 15:11:46,505 - qm - INFO     - Opening QM
2026-07-13 15:11:46,505 - qm - INFO     - Clearing queue
2026-07-13 15:11:46,515 - qm - INFO     - Adding program to queue.
2026-07-13 15:11:49,961 - qm - INFO     - Program added to queue. Job id: 95bbf570-bd3f-4794-9806-c11bb2bbcf19
2026-07-13 15:16:57,531 - qm - INFO     - Closing QM         ] 77.0% (n=77/100) --> elapsed time: 303.37s
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26c_fNgN1_ramsey.py:437: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='f0g1_ramsey', description='\n        SIDEBAND RAMSEY - precise frequency calibration for any |n⟩ → |n+1⟩ transition\n\nPerforms a Ramsey fringe experiment on the sideband drive to extract the resonance\nfrequency of the |f, n⟩ â†" |g, n+1⟩ transition with high precision.\n\nAn artificial detuning δ is added so that fringes are visible even at zero drive\ndetuning.  The fitted oscillation frequency f_obs satisfies:\n    f_obs = |f_drive - f_sideband + δ|\nfrom which the corrected sideband frequency is:\n    f_sideband = f_drive + δ - f_obs  (using the sign that minimises |correction|)\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity in |j+1⟩, qubit in |g⟩.\n  2. π_ge  →  |e⟩\n  3. π_ef  →  |f⟩\n  4. π/2 sideband pulse  →  superposition (|f,n⟩ + |g,n+1⟩).\n  5. Wait τ  +  virtual frame rotation (artificial detuning δ).\n  6. π/2 sideband pulse  →  interference.\n  7. π_e

##### Qubit ge and ef frequency shift calibration

Measures the dispersive qubit ge shift (chi_focka) and ef shift (ef_chi_focka)
when the cavity is in Fock |k⟩. These are used to drive the correct qubit frequencies
in all subsequent Fock state preparations.

**State update**:  and 

In [3]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26e_fNgN1_qubit_ge_spectroscopy"].copy(name="f0g1_ge_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 0
node.parameters.frequency_span_in_mhz      = 5.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "selective_x180"
node.parameters.operation_len_in_ns        = None
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.cavity_active_cooling_fock_n = 1   # start cooling from |1⟩ (thermal state)
node.parameters.sideband_pulse_duration_ns = None #1_000_000
node.parameters.use_state_discrimination   = True
node.parameters.use_confusion_matrix_correction   = False

node.run()


2026-07-14 10:52:17,178 - qualibrate - INFO - Creating node 26e_fNgN1_qubit_ge_spectroscopy
2026-07-14 10:52:17,238 - qualibrate - INFO - Copying node with name 26e_fNgN1_qubit_ge_spectroscopy with parameters name = 'f0g1_ge_spectroscopy', node_parameters = {}
2026-07-14 10:52:17,248 - qualibrate - INFO - Creating node 26e_fNgN1_qubit_ge_spectroscopy
2026-07-14 10:52:17,318 - qualibrate - INFO - Run node f0g1_ge_spectroscopy with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 10:52:17,518 - qm - INFO     - Performing health check
2026-07-14 10:52:17,518 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 10:52:17,688 - qm - INFO     - Opened quantum machine with id: QM-8ec0bfa7-12f4-4c3c-ac2e-2b2caf79ff9a
2026-07-14 10:52:17,688 - qm - INFO     - Opening QM
2026-07-14 10:52:17,688 - qm - INFO     - Clearing queue
2026-07-14 10:52:17,698 - qm - INFO     - Adding program to queue.
2026-07-14 10:52:18,019 - qm - INFO     - Program added to queue. Job id: 386603df-9ade-4e4c-b988-c013881674dc
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 195.96s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.01s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.06s
Progress: [#######

2026-07-14 10:55:38,251 - qualibrate - INFO - Node f0g1_ge_spectroscopy - Execution report for job 386603df-9ade-4e4c-b988-c013881674dc
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.06s
2026-07-14 10:55:38,261 - qm - INFO     - Closing QM


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26e_fNgN1_qubit_ge_spectroscopy.py:270: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-14 10:55:38,361 - qualibrate - INFO - Saving node f0g1_ge_spectroscopy to local storage


Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-07-14 10:55:38,543 - qualibrate - INFO - Saving machine state to db
2026-07-14 10:55:38,551 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-14 10:55:38,553 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-14 10:55:38,562 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-14\#865_f0g1_ge_spectroscopy_105538\quam_state


Action save_results finished


NodeRunSummary(name='f0g1_ge_spectroscopy', description='\n        QUBIT ge SPECTROSCOPY AT FOCK |n⟩\n\nSweeps the qubit ge drive frequency while the cavity is prepared in Fock state |fock_level⟩.\nThe dispersive shift causes the qubit ge resonance to shift by (k+1)×chi + delta_f_focka\nrelative to the vacuum (|0⟩) frequency.  At fock_level=0 this directly calibrates chi\n(stored on the pair); for higher levels it calibrates the nonlinear correction delta_f_focka.\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity |j+1⟩, qubit |g⟩.\n  2. Sweep qubit ge frequency around f_ge + fock_level × chi_estimate.\n  3. Play saturation pulse on qubit ge.\n  4. Measure qubit state.\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - For fock_level > 0: calibrated sideband transitions 0…fock_level-1 (nodes 26/26b).\n\nState update:\n    - cavity_transmon_pairs["{qubit}_{mode}

In [4]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26f_fNgN1_ge_ramsey"].copy(name="f0g1_ge_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 0
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 51
node.parameters.frequency_detuning_in_mhz = 1.0
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


Getting calibration path from config
C:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-13 21:25:16,207 - qm - INFO     - Performing health check
2026-07-13 21:25:16,217 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-13 21:25:16,387 - qm - INFO     - Opened quantum machine with id: QM-298f1793-c1f0-4d08-bf95-36eea47ebe8b
2026-07-13 21:25:16,387 - qm - INFO     - Opening QM
2026-07-13 21:25:16,387 - qm - INFO     - Clearing queue
2026-07-13 21:25:16,397 - qm - INFO     - Adding program to queue.
2026-07-13 21:25:20,058 - qm - INFO     - Program added to queue. Job id: 6db8157b-8c4b-48e0-912a-6e2a71d06109
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 199.78s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 199.83s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 199.89s
Progress: [#######

C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26f_fNgN1_ge_ramsey.py:268: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='f0g1_ge_ramsey', description='\n        ge RAMSEY AT FOCK |n⟩ — precise calibration of chi_focka\n\nPerforms a Ramsey experiment on the qubit ge transition while the cavity is in Fock\nstate |fock_level⟩.  The fitted oscillation frequency reveals the residual detuning of\nthe ge drive from the true ge resonance at this Fock level.  This refines the coarse\nchi_focka value measured by node 26d (ge spectroscopy) with much higher precision.\n\nBoth signs of the artificial detuning ±δ are swept so the frequency correction is\nunambiguous:\n    f_obs±  =  |f_drive - f_ge_focka ± δ|\n    correction  =  (f_obs+ - f_obs-) / 2   →   chi_focka -= correction\n\nSequence:\n  0. Cavity + qubit reset (thermal or active sideband).\n  1. [Fock prep] For j = 0 … fock_level-1:\n       Ï€_ge → Ï€_ef → sideband_pi(f{j}g{j+1}) → cavity |j+1⟩, qubit |g⟩.\n  2. Set ge drive to chi_focka-shifted frequency (_ge_if_at_fock).\n  3. Ï€/2 ge pulse (x180 at half amplitude).\n  4. Wait Ï" + virt

In [4]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26g_fNgN1_qubit_ef_spectroscopy"].copy(name="f0g1_ef_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 0
node.parameters.frequency_span_in_mhz      = 5.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "selective_ef_x180"
node.parameters.operation_len_in_ns        = None
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


2026-07-14 10:57:11,860 - qualibrate - INFO - Creating node 26g_fNgN1_qubit_ef_spectroscopy
2026-07-14 10:57:11,930 - qualibrate - INFO - Copying node with name 26g_fNgN1_qubit_ef_spectroscopy with parameters name = 'f0g1_ef_spectroscopy', node_parameters = {}
2026-07-14 10:57:11,930 - qualibrate - INFO - Creating node 26g_fNgN1_qubit_ef_spectroscopy
2026-07-14 10:57:12,000 - qualibrate - INFO - Run node f0g1_ef_spectroscopy with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 10:57:12,210 - qm - INFO     - Performing health check
2026-07-14 10:57:12,210 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 10:57:12,411 - qm - INFO     - Opened quantum machine with id: QM-c078aee6-0012-4f02-8737-9286a0877886
2026-07-14 10:57:12,411 - qm - INFO     - Opening QM
2026-07-14 10:57:12,411 - qm - INFO     - Clearing queue
2026-07-14 10:57:12,421 - qm - INFO     - Adding program to queue.
2026-07-14 10:57:12,761 - qm - INFO     - Program added to queue. Job id: ad0171e6-2569-4590-bc42-db82bdae1c6c
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 195.95s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 195.99s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.04s
Progress: [#######

2026-07-14 11:00:33,025 - qualibrate - INFO - Node f0g1_ef_spectroscopy - Execution report for job ad0171e6-2569-4590-bc42-db82bdae1c6c
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.05s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.10s
2026-07-14 11:00:33,025 - qm - INFO     - Closing QM


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26g_fNgN1_qubit_ef_spectroscopy.py:280: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-14 11:00:33,135 - qualibrate - INFO - Saving node f0g1_ef_spectroscopy to local storage


Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-07-14 11:00:33,329 - qualibrate - INFO - Saving machine state to db
2026-07-14 11:00:33,340 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-14 11:00:33,341 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-14 11:00:33,362 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-14\#866_f0g1_ef_spectroscopy_110033\quam_state


Action save_results finished


NodeRunSummary(name='f0g1_ef_spectroscopy', description='\n        QUBIT ef SPECTROSCOPY AT FOCK |n⟩\n\nSweeps the qubit ef drive frequency while the cavity is prepared in Fock state |fock_level⟩.\nThe Kerr nonlinearity shifts the ef resonance (ef_delta_f_focka) relative to vacuum.\nThis node calibrates the per-Fock ef frequency shift.\n\nSequence:\n  0. Cavity + qubit reset (thermal or active sideband).\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity |j+1⟩, qubit |g⟩.\n  2. π_ge at chi_focka-shifted frequency → qubit |e⟩.\n  3. Sweep ef frequency around ef_chi_focka estimate.\n  4. Play saturation pulse on qubit ef.\n  5. Refocusing π_ge at chi_focka-shifted frequency: maps |e⟩→|g⟩ off-resonance, leaves |f⟩ untouched on-resonance.\n  6. Measure qubit state.\n\nPrerequisites:\n    - Calibrated chi_focka (nodes 26d/26e, fock_level=k).\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - For fock_level > 0: calibrated sideband

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26h_fNgN1_ef_ramsey"].copy(name="f0g1_ef_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 0
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.frequency_detuning_in_mhz = 1.0
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


#### f1g2 - |1> -> |2> sideband

> **Prerequisite**: f0g1 fully calibrated (all three cells above must have been run).

##### Spectroscopy

Sweeps the sideband drive frequency while the qubit is in |f> and the cavity is in
Fock |1> (prepared using already-calibrated sideband pulses).

**State update**: `pair.extras["f1g2_RF_frequency"]`

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26_fNgN1_spectroscopy"].copy(name="f1g2_spectroscopy")
node.parameters.qubits                        = ["q1"]
node.parameters.mode_name                     = "alice"
node.parameters.fock_level                    = 1
node.parameters.frequency_span_in_mhz         = 5.0
node.parameters.frequency_step_in_mhz         = .05
node.parameters.operation                     = "sideband_flat_top"
node.parameters.operation_len_in_ns           = None #10000
node.parameters.operation_amplitude_factor    = 1
node.parameters.num_shots                     = 100
node.parameters.cavity_reset_type             = "thermal"
node.parameters.use_state_discrimination      = True
node.parameters.use_confusion_matrix_correction = False
node.parameters.use_theoretical_frequency_estimate = False
node.parameters.use_gaussian_fit              = False
node.run()


2026-07-14 11:02:39,762 - qualibrate - INFO - Creating node 26_fNgN1_spectroscopy
2026-07-14 11:02:39,830 - qualibrate - INFO - Copying node with name 26_fNgN1_spectroscopy with parameters name = 'f1g2_spectroscopy', node_parameters = {}
2026-07-14 11:02:39,840 - qualibrate - INFO - Creating node 26_fNgN1_spectroscopy
2026-07-14 11:02:39,900 - qualibrate - INFO - Run node f1g2_spectroscopy with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 11:02:40,140 - qm - INFO     - Performing health check
2026-07-14 11:02:40,150 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 11:02:40,341 - qm - INFO     - Opened quantum machine with id: QM-455eb1ce-71a5-4205-a693-92edeb0ab6f1
2026-07-14 11:02:40,341 - qm - INFO     - Opening QM
2026-07-14 11:02:40,341 - qm - INFO     - Clearing queue
2026-07-14 11:02:40,355 - qm - INFO     - Adding program to queue.
2026-07-14 11:02:40,692 - qm - INFO     - Program added to queue. Job id: 08c7eac9-9c2c-4af5-98af-104f8e060d61
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 195.97s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.02s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.07s
Progress: [#######

2026-07-14 11:06:00,915 - qualibrate - INFO - Node f1g2_spectroscopy - Execution report for job 08c7eac9-9c2c-4af5-98af-104f8e060d61
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 197.91s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 197.96s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.01s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.06s
2026-07-14 11:06:00,925 - qm - INFO     - Closing QM
Action execute_qua_program finished

2026-07-14 11:06:00,975 - qualibrate - INFO - Node f1g2_spectroscopy - Results for qubit q1: SUCCESS
	f0g1 frequency: 3.3007 GHz | FWHM: 898.8 kHz
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26_fNgN1_spectroscopy.py:327: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-14 11:06:01,026 - qualibrate - INFO - Saving node f1g2_spectroscopy to local storage



Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state_node
Action update_state_node finished
Running action save_results


2026-07-14 11:06:01,208 - qualibrate - INFO - Saving machine state to db
2026-07-14 11:06:01,224 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-14 11:06:01,226 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-14 11:06:01,242 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-14\#867_f1g2_spectroscopy_110601\quam_state


Action save_results finished


NodeRunSummary(name='f1g2_spectroscopy', description='\n        SIDEBAND SPECTROSCOPY - generalised to any |n⟩ → |n+1⟩ transition\n\nSweeps the sideband drive frequency while the qubit is prepared in |f⟩ and the\ncavity is prepared in Fock |fock_level⟩.\n\nWhen the sideband drive is resonant, the |f, n⟩ â†" |g, n+1⟩ transition is driven,\nthe qubit is left in |g⟩, and the back-swap π_ef leaves it in |g⟩ → DIP in state\nmeasurement.\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity |j+1⟩, qubit back to |g⟩.\n  2. π_ge  →  |e⟩\n  3. π_ef  →  |f⟩\n  4. Sweep f{k}g{k+1} sideband IF;  play saturation/long pulse.\n  5. π_ef  (back-swap)\n  6. Measure qubit state.\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - For fock_level > 0: calibrated sideband pulses for transitions 0…fock_level-1\n      stored in pair.transitions["f{j}g{j+1}"].pi_flat_top_length_ns.\n\nSta

##### Time Rabi

Sweeps the |1>->|2> sideband drive duration to calibrate the  π-pulse length.

**State update**: `sideband_drive.operations["f1g2_pi"].length`

In [6]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26b_fNgN1_time_rabi"].copy(name="f1g2_time_rabi")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 1
node.parameters.min_duration_ns            = 16
node.parameters.max_duration_ns            = 5_000
node.parameters.duration_step_ns           = 80
node.parameters.ramp_length_ns             = 200  # ns, multiple of 4
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.parameters.use_confusion_matrix_correction = False
node.run()

2026-07-14 11:06:31,422 - qualibrate - INFO - Creating node 26b_fNgN1_time_rabi
2026-07-14 11:06:31,492 - qualibrate - INFO - Copying node with name 26b_fNgN1_time_rabi with parameters name = 'f1g2_time_rabi', node_parameters = {}
2026-07-14 11:06:31,502 - qualibrate - INFO - Creating node 26b_fNgN1_time_rabi
2026-07-14 11:06:31,592 - qualibrate - INFO - Run node f1g2_time_rabi with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 11:06:31,822 - qm - INFO     - Performing health check
2026-07-14 11:06:31,832 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 11:06:32,022 - qm - INFO     - Opened quantum machine with id: QM-0c51e55b-7263-406d-bf30-1ebb1cd80628
2026-07-14 11:06:32,032 - qm - INFO     - Opening QM
2026-07-14 11:06:32,032 - qm - INFO     - Clearing queue
2026-07-14 11:06:32,032 - qm - INFO     - Adding program to queue.
2026-07-14 11:06:35,373 - qm - INFO     - Program added to queue. Job id: 195e5d00-41d2-41c8-8654-f1f52d45c955
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.45s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.50s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.55s
Progress: [#######

2026-07-14 11:08:41,593 - qualibrate - INFO - Node f1g2_time_rabi - Execution report for job 195e5d00-41d2-41c8-8654-f1f52d45c955
No errors


2026-07-14 11:08:41,603 - qm - INFO     - Closing QM


2026-07-14 11:08:41,623 - qualibrate - INFO - Node f1g2_time_rabi - Results for qubit q1: SUCCESS
	f0g1 pi-duration: 820 ns | chi2: 0.049 | periods: 2.50
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26b_fNgN1_time_rabi.py:272: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-14 11:08:41,684 - qualibrate - INFO - Saving node f1g2_time_rabi to local storage


Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-07-14 11:08:41,835 - qualibrate - INFO - Saving machine state to db
2026-07-14 11:08:41,843 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-14 11:08:41,843 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-14 11:08:41,854 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-14\#868_f1g2_time_rabi_110841\quam_state


Action save_results finished


NodeRunSummary(name='f1g2_time_rabi', description='\n        SIDEBAND TIME RABI - generalised to any |n⟩ → |n+1⟩ transition\n\nSweeps the sideband drive duration while the qubit is in |f⟩ and the cavity in\nFock |fock_level⟩.  A Rabi-like oscillation is observed; the π-pulse duration is\nextracted from the first minimum of the fitted sinusoid.\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity in |j+1⟩, qubit in |g⟩.\n  2. π_ge  →  |e⟩\n  3. π_ef  →  |f⟩\n  4. Play f{k}g{k+1} sideband pulse with swept duration.\n  5. π_ef  (back-swap)\n  6. Measure qubit state.\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - Calibrated sideband frequency for this transition (node 26, fock_level=k).\n    - For fock_level > 0: calibrated sideband pulses for transitions 0…fock_level-1.\n\nState update:\n    - cavity_transmon_pairs["{qubit}_{mode}"].sideband_drive.operations["f{

##### Ramsey

Performs a Ramsey fringe experiment on the f1g2 sideband to refine its
resonance frequency. The artificial detuning (1 MHz default) ensures visible
fringes even when close to resonance.

**State update**: `pair.extras["f1g2_RF_frequency"]` (fine-tuned)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26c_fNgN1_ramsey"].copy(name="f1g2_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                = 1
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.artificial_detuning_hz     = 1e6
node.parameters.num_shots                  = 200
node.parameters.cavity_thermalization_time_ns = 200_000
node.parameters.use_state_discrimination   = True
node.run()

##### Qubit ge and ef frequency shift calibration

Measures the dispersive qubit ge shift (chi_focka) and ef shift (ef_chi_focka)
when the cavity is in Fock |k⟩. These are used to drive the correct qubit frequencies
in all subsequent Fock state preparations.

**State update**:  and 

In [16]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26e_fNgN1_qubit_ge_spectroscopy"].copy(name="f1g2_ge_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 1
node.parameters.frequency_span_in_mhz      = 5.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "selective_x180"
node.parameters.operation_len_in_ns        = None
node.parameters.num_shots                  = 100
node.parameters.cavity_reset_type          = "thermal"
node.parameters.cavity_active_cooling_fock_n = 1   # start cooling from |1⟩ (thermal state)
node.parameters.sideband_pulse_duration_ns = None #1_000_000
node.parameters.use_state_discrimination   = True
node.parameters.use_confusion_matrix_correction   = False

node.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 02:02:42,969 - qm - INFO     - Performing health check
2026-07-14 02:02:42,979 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 02:02:43,149 - qm - INFO     - Opened quantum machine with id: QM-65dda8b2-1f20-4208-a151-8c5704b330e4
2026-07-14 02:02:43,159 - qm - INFO     - Opening QM
2026-07-14 02:02:43,159 - qm - INFO     - Clearing queue
2026-07-14 02:02:43,159 - qm - INFO     - Adding program to queue.
2026-07-14 02:02:43,531 - qm - INFO     - Program added to queue. Job id: 0f9249c9-0d85-4dea-9cc4-9b9bc437d4cc
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 195.98s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.02s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.06s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\26e_fNgN1_qubit_ge_spectroscopy.py:265: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='f1g2_ge_spectroscopy', description='\n        QUBIT ge SPECTROSCOPY AT FOCK |n⟩\n\nSweeps the qubit ge drive frequency while the cavity is prepared in Fock state |fock_level⟩.\nThe dispersive shift causes the qubit ge resonance to shift by chi_focka relative to the\nvacuum (|0⟩) frequency.  This node calibrates that per-Fock shift.\n\nSequence:\n  0. Thermalize cavity and qubit.\n  1. [Fock prep] For j = 0 … fock_level-1:\n       π_ge → π_ef → sideband_pi(f{j}g{j+1}) → cavity |j+1⟩, qubit |g⟩.\n  2. Sweep qubit ge frequency around f_ge + fock_level × chi_estimate.\n  3. Play saturation pulse on qubit ge.\n  4. Measure qubit state.\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n    - For fock_level > 0: calibrated sideband transitions 0…fock_level-1 (nodes 26/26b).\n\nState update:\n    - cavity_transmon_pairs["{qubit}_{mode}"].transitions["f{k}g{k+1}"].chi_focka\n', created_at=datetime.datetime(2026, 7, 14, 2, 2, 42, 769135, tzinfo=date

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26f_fNgN1_ge_ramsey"].copy(name="f1g2_ge_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 1
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.frequency_detuning_in_mhz = 1.0
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26g_fNgN1_qubit_ef_spectroscopy"].copy(name="f1g2_ef_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 1
node.parameters.frequency_span_in_mhz      = 2.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "saturation"
node.parameters.operation_len_in_ns        = 20_000
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26h_fNgN1_ef_ramsey"].copy(name="f1g2_ef_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 1
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.frequency_detuning_in_mhz = 1.0
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


#### f2g3 - |2> -> |3> sideband

> **Prerequisite**: f1g2 fully calibrated (all three cells above must have been run).

##### Spectroscopy

Sweeps the sideband drive frequency while the qubit is in |f> and the cavity is in
Fock |2> (prepared using already-calibrated sideband pulses).

**State update**: `pair.extras["f2g3_RF_frequency"]`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26_fNgN1_spectroscopy"].copy(name="f2g3_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                = 2
node.parameters.frequency_span_in_mhz      = 10.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "saturation"
node.parameters.operation_len_in_ns        = 20_000
node.parameters.operation_amplitude_factor = 1.0
node.parameters.num_shots                  = 500
node.parameters.cavity_thermalization_time_ns = 200_000
node.parameters.use_state_discrimination   = True
node.parameters.use_gaussian_fit              = False
node.run()

##### Time Rabi

Sweeps the |2>->|3> sideband drive duration to calibrate the  π-pulse length.

**State update**: `sideband_drive.operations["f2g3_pi"].length`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26b_fNgN1_time_rabi"].copy(name="f2g3_time_rabi")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                = 2
node.parameters.min_duration_ns            = 16
node.parameters.max_duration_ns            = 20_000
node.parameters.duration_step_ns           = 4
node.parameters.num_shots                  = 100
node.parameters.cavity_thermalization_time_ns = 200_000
node.parameters.use_state_discrimination   = True
node.run()

##### Ramsey

Performs a Ramsey fringe experiment on the f2g3 sideband to refine its
resonance frequency. The artificial detuning (1 MHz default) ensures visible
fringes even when close to resonance.

**State update**: `pair.extras["f2g3_RF_frequency"]` (fine-tuned)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26c_fNgN1_ramsey"].copy(name="f2g3_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                = 2
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.artificial_detuning_hz     = 1e6
node.parameters.num_shots                  = 200
node.parameters.cavity_thermalization_time_ns = 200_000
node.parameters.use_state_discrimination   = True
node.run()

##### Qubit ge and ef frequency shift calibration

Measures the dispersive qubit ge shift (chi_focka) and ef shift (ef_chi_focka)
when the cavity is in Fock |k⟩. These are used to drive the correct qubit frequencies
in all subsequent Fock state preparations.

**State update**:  and 

##### ge IQ blobs at Fock |3⟩

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26d_fNgN1_ge_iq_blobs"].copy(name="f2g3_ge_iq_blobs")
node.parameters.qubits                      = ["q1"]
node.parameters.mode_name                   = "alice"
node.parameters.fock_level                  = 2  # prepares Fock |3⟩
node.parameters.num_shots                   = 2000
node.parameters.cavity_reset_type           = "thermal"
node.parameters.sideband_pulse_duration_ns  = 1_000_000
node.run()


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26e_fNgN1_qubit_ge_spectroscopy"].copy(name="f2g3_ge_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 2
node.parameters.frequency_span_in_mhz      = 2.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "saturation"
node.parameters.operation_len_in_ns        = 20_000
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26f_fNgN1_ge_ramsey"].copy(name="f2g3_ge_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 2
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.frequency_detuning_in_mhz = 1.0
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26g_fNgN1_qubit_ef_spectroscopy"].copy(name="f2g3_ef_spectroscopy")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 2
node.parameters.frequency_span_in_mhz      = 2.0
node.parameters.frequency_step_in_mhz      = 0.05
node.parameters.operation                  = "saturation"
node.parameters.operation_len_in_ns        = 20_000
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

node = library.nodes["26h_fNgN1_ef_ramsey"].copy(name="f2g3_ef_ramsey")
node.parameters.qubits                     = ["q1"]
node.parameters.mode_name                  = "alice"
node.parameters.fock_level                 = 2
node.parameters.min_wait_ns                = 16
node.parameters.max_wait_ns                = 5_000
node.parameters.num_wait_points            = 101
node.parameters.frequency_detuning_in_mhz = 1.0
node.parameters.num_shots                  = 300
node.parameters.cavity_reset_type          = "thermal"
node.parameters.use_state_discrimination   = True
node.run()


### 8. Alice cavity calibration

#### Cavity spectroscopy

Sweeps the Alice cavity drive frequency using the `selective_x180` qubit probe to locate the cavity resonance.

**State update**: `cavity_mode.cavity_mode_drive.RF_frequency`

In [10]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["21_cavity_mode_spectroscopy"].copy(name="cavity_mode_spectroscopy")
parameters = node.parameters
parameters.num_shots = 200
parameters.mode_name = "alice"
parameters.frequency_span_in_mhz = 10
parameters.frequency_step_in_mhz = 0.05
parameters.operation = "displacement"
parameters.operation_amplitude_factor = 1.0
parameters.operation_len_in_ns = None
parameters.cavity_thermalization_time_ns = 10_000  # 20 ms = 1x T1; increase if spectrum is noisy
parameters.use_state_discrimination = True
parameters.qubit_probe_operation = "selective_x180"
parameters.use_gaussian_fit                   = False
node.run()

2026-07-14 15:44:35,715 - qualibrate - INFO - Creating node 21_cavity_mode_spectroscopy
2026-07-14 15:44:35,778 - qualibrate - INFO - Copying node with name 21_cavity_mode_spectroscopy with parameters name = 'cavity_mode_spectroscopy', node_parameters = {}
2026-07-14 15:44:35,788 - qualibrate - INFO - Creating node 21_cavity_mode_spectroscopy
2026-07-14 15:44:35,858 - qualibrate - INFO - Run node cavity_mode_spectroscopy with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 15:44:36,108 - qm - INFO     - Performing health check
2026-07-14 15:44:36,108 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 15:44:36,390 - qm - INFO     - Opened quantum machine with id: QM-5104ed6c-5435-43f4-b5f3-422763622eba
2026-07-14 15:44:36,390 - qm - INFO     - Opening QM
2026-07-14 15:44:36,400 - qm - INFO     - Clearing queue
2026-07-14 15:44:36,410 - qm - INFO     - Adding program to queue.
2026-07-14 15:44:36,853 - qm - INFO     - Program added to queue. Job id: ca09a44f-9218-41aa-a436-3511a85d71b5
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 90.79s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 90.86s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 90.93s
Progress: [##########

2026-07-14 15:46:09,193 - qualibrate - INFO - Node cavity_mode_spect... - Execution report for job ca09a44f-9218-41aa-a436-3511a85d71b5
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 91.65s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 91.74s
2026-07-14 15:46:09,203 - qm - INFO     - Closing QM


2026-07-14 15:46:09,253 - qualibrate - INFO - Node cavity_mode_spect... - Results for qubit q1: SUCCESS
	Cavity resonance: 5.993877 GHz | FWHM: 0.920 MHz | Detuning offset: 0.025 MHz


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\21_cavity_mode_spectroscopy.py:291: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-07-14 15:46:09,303 - qualibrate - INFO - Saving node cavity_mode_spectroscopy to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-07-14 15:46:09,486 - qualibrate - INFO - Saving machine state to db
2026-07-14 15:46:09,496 - qualibrate - WARNING - save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
2026-07-14 15:46:09,504 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run011\srf_qubit_2_qualibrate\state
2026-07-14 15:46:09,527 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run011\srf_qubit_2_qualibrate\calibration_storage\2026-07-14\#897_cavity_mode_spectroscopy_154609\quam_state


Action save_results finished


NodeRunSummary(name='cavity_mode_spectroscopy', description="\n        CAVITY MODE SPECTROSCOPY\nFinds the bare resonance frequency of a storage cavity mode (e.g. alice or bob)\nby sweeping the cavity drive frequency and using dispersive coupling to the qubit\nas the photon detector.\n\nSequence (per cavity detuning df):\n  1. Wait 2× thermalization time (qubit and cavity thermalise to |g,0⟩).\n  2. Set qubit drive to bare ge frequency (no sweep on qubit).\n  3. Sweep cavity drive to (IF_cavity + df) and play saturation / probe pulse.\n  4. Apply selective_x180 on qubit at bare ge frequency.\n     - Off resonance (no photons): selective pulse succeeds → qubit in |e⟩.\n     - On resonance (photons present): dispersive shift detunes qubit → pulse\n       fails → qubit stays in |g⟩.\n  5. Measure qubit state.\n\nThe result is a DIP in the qubit excitation probability at the cavity resonance.\nA Lorentzian dip fit extracts the cavity frequency.\n\nPrerequisites:\n    - Calibrated ge and ef

#### Displacement calibration

Sweeps the displacement amplitude and measures vacuum-state population using a
selective pi-pulse.  Fits P_e(a) = A * exp(-|alpha/A_disp|^2)exp(-|a/A_disp|^2) to extract the unit
displacement amplitude Aph (amplitude_scale=1  1 photon).

**State update**: `cavity_mode.cavity_mode_drive.operations["displacement"].amplitude` (Alice)


In [9]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["22_displacement_calibration_vacuum"].copy(name="alice_disp_calib")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.amp_min = -3
parameters.amp_max = 3
parameters.amp_points = 21
parameters.num_shots = 100
parameters.cavity_reset_type = "thermal"
parameters.use_state_discrimination = True
parameters.subtract_baseline = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1] #TODO: remove
node.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 18:13:42,330 - qm - INFO     - Performing health check
2026-07-14 18:13:42,330 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 18:13:42,570 - qm - INFO     - Opened quantum machine with id: QM-b0dfc4f2-5c9b-4dcd-8e1f-9a7c5cbc820d
2026-07-14 18:13:42,570 - qm - INFO     - Opening QM
2026-07-14 18:13:42,570 - qm - INFO     - Clearing queue
2026-07-14 18:13:42,580 - qm - INFO     - Adding program to queue.
2026-07-14 18:13:42,970 - qm - INFO     - Program added to queue. Job id: a0a06a20-03c7-4267-8ffc-bc2a766ed138
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 82.34s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 82.39s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 82.44s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\22_displacement_calibration_vacuum.py:397: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_disp_calib', description='\n        DISPLACEMENT VACUUM-POPULATION CALIBRATION (35) — dual-sequence with baseline\n\nCalibrates the unit displacement amplitude by sweeping the cavity displacement\namplitude and measuring the vacuum-state population with a selective qubit π-pulse.\n\nBecause of cross-Kerr coupling between the cavity mode and the readout resonator, the\nresonator IQ response shifts as a function of displacement amplitude even when the qubit\nis untouched.  To remove this spurious baseline, each averaging iteration now runs TWO\nsub-sequences for every amplitude point:\n\n  PART 1 — Baseline (no qubit π-pulse):\n    1a. Reset cavity + qubit.\n    1b. Displace cavity to |α = a · A_unit⟩.\n    1c. Measure readout IQ  →  I_base, Q_base  (cross-Kerr offset only).\n  PART 2 — Signal (full protocol):\n    2a. Reset cavity + qubit (independent reset, same conditions as Part 1).\n    2b. Displace cavity identically.\n    2c. Apply selective_x180 (or x18

#### Coherent T1

Prepares | by displacement, waits variable time t, then probes vacuum population with `selective_x180`. Fits a Gumbel decay to extract T1.

**State update**: `cavity_mode.T1`

In [7]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["23_cavity_coherent_T1"].copy(name="alice_coherent_T1")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.displacement_alpha = 4.0   # scale=1  1 photon (after node 35); 1.9  ~3.6 photons
# min/max_wait_time_in_ns are the *per-repeat* range.
# Total sweep spans [min, delay_repeats  max] ns.
parameters.min_wait_time_in_ns = 100
parameters.max_wait_time_in_ns = 40_000_000
parameters.wait_time_num_points = 11
parameters.log_or_linear_sweep = "linear"      # "log" or "linear"
parameters.delay_repeats = 1
parameters.num_shots = 200
parameters.cavity_reset_type = "thermal"  # "thermal" or "active_sideband"
# parameters.cavity_active_cooling_fock_n = 1
parameters.use_state_discrimination = True
parameters.subtract_baseline = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-06 21:27:30,149 - qm - INFO     - Performing health check
2026-07-06 21:27:30,149 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-06 21:27:30,340 - qm - INFO     - Opened quantum machine with id: QM-cded8562-3c3e-49e4-95b7-3ddd75b652c0
2026-07-06 21:27:30,340 - qm - INFO     - Opening QM
2026-07-06 21:27:30,340 - qm - INFO     - Clearing queue
2026-07-06 21:27:30,349 - qm - INFO     - Adding program to queue.
2026-07-06 21:27:49,728 - qm - INFO     - Program added to queue. Job id: c770634f-2570-48b4-8ae1-e38d6bfe3e6f
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 152.52s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 152.60s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 152.67s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\23_cavity_coherent_T1.py:367: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_coherent_T1', description="\n        CAVITY COHERENT T1 (23)\n\nMeasures the energy relaxation time T1 of a selected cavity mode by preparing\na coherent state |α⟩ and probing the vacuum-state population with a selective\nqubit π-pulse.\n\nSequence (per wait time t):\n  1. Thermalize cavity (wait ≥ 5×T1) and reset qubit.\n  2. Apply displacement pulse: amplitude_scale = displacement_alpha / displacement_alpha_max.\n  3. Wait for total time t = delay_repeats × t_per_rep.\n  4. Apply selective_x180 on qubit — flips qubit only when cavity is in |0⟩.\n  5. Measure qubit state.\n\nThe measured signal is:\n    P_e(t) = A · exp(-|αâ'€|² · exp(-t / T1)) + offset\n\nwhere |αâ'€|² = displacement_alpha² and T1 is the cavity photon lifetime.\n\nFitting extracts T1 and |αâ'€|².  The dataset is augmented with the inferred\nphoton-number decay |α(t)|² = -ln((P_e - offset) / A), plotted as a simple\nexponential: |αâ'€|² · exp(-t / T1).\n\nParameters:\n  - mode_name:         

#### Photon number resolved spectroscopy

Displaces Alice to |alpha> and sweeps qubit ge spectroscopy. Photon-number-resolved peaks separated by chi reveal P(n). Use `displacement_alpha` to scale the coherent state amplitude.

**State update**: `cavity_mode.chi`

In [8]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["24_photon_number_resolved_spectroscopy"].copy(name="alice_pns")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.displacement_alpha = 1.0
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.right_offset_mhz = 0.25
parameters.left_span_mhz = 2.0
parameters.frequency_step_in_mhz = 0.05
parameters.max_peaks = 3
parameters.chi2_threshold = 2.0
parameters.num_shots = 200
parameters.cavity_reset_type = "thermal"
# parameters.cavity_active_cooling_fock_n = 4
# parameters.f0g1_pulse_duration_ns = 1e6 # 1 ms
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1] #TODO: remove
node.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-06 21:44:10,269 - qm - INFO     - Performing health check
2026-07-06 21:44:10,279 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-06 21:44:10,516 - qm - INFO     - Opened quantum machine with id: QM-07e84f62-797d-4bfa-88eb-b8a5016e095d
2026-07-06 21:44:10,517 - qm - INFO     - Opening QM
2026-07-06 21:44:10,518 - qm - INFO     - Clearing queue
2026-07-06 21:44:10,519 - qm - INFO     - Adding program to queue.
2026-07-06 21:44:10,911 - qm - INFO     - Program added to queue. Job id: e9c397c7-bcfe-480c-adf5-1301b02860f9
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.23s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.29s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.35s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\24_photon_number_resolved_spectroscopy.py:265: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_pns', description="\n        PHOTON NUMBER RESOLVED SPECTROSCOPY — CHI MEASUREMENT (29)\n\nDisplaces the selected cavity mode to a coherent state |α⟩ and sweeps the\nqubit ge spectroscopy frequency.  The resulting spectrum shows photon-number-\nsplit peaks:\n\n    f_n = f_q + chi*n   (n=0, 1, 2, ...)\n\nseparated by |chi|.  The node auto-detects the number of peaks (1 → max_peaks)\nby fitting successive multi-Gaussian models until the reduced chi² drops below\nthe threshold.  chi = -(mean PNRS peak spacing) is saved to the machine state.\n\nConvention: chi [Hz] is the full per-photon qubit frequency shift (negative for\ntypical transmon-cavity systems where more photons lower the qubit frequency).\nchi = -(PNRS peak spacing) and |chi| = PNRS peak spacing.\n\nAfter measurement an optional active reset applies D(-α) to return the cavity\nto vacuum immediately, replacing passive thermalization.\n\nPrerequisites:\n    - Calibrated qubit_pulse operation on qubit.x

#### Parity time calibration  (Alice)

Calibrates the dispersive Ramsey wait time tau_parity required for Wigner tomography:

    tau = pi / (2 * chi)  # tau_parity_eff = tau_parity = pi/(2*chi) = 1 / (2 * chi) = pi / (2 * delta_f)

A short displacement (~1 photon) is applied, followed by a Ramsey sequence
`y90  wait(…)  y90`. P(e) oscillates at f_chi; the fit extracts tau_parity.

**State update**: `cavity_transmon_pairs["q1_alice"].parity_time`


In [7]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_parity = library.nodes["28_parity_time_measurement"]
alice_parity.parameters.qubits              = ["q1"]
alice_parity.parameters.mode_name           = "alice"
alice_parity.parameters.num_shots           = 100
alice_parity.parameters.min_delay_ns        = 16
alice_parity.parameters.max_delay_ns        = 5_000    # cover >1 period for chi ~ 70 kHz (T ~ 14 us)
alice_parity.parameters.delay_step_ns       = 100
alice_parity.parameters.cavity_reset_type   = "thermal"   # or "active_sideband"
alice_parity.parameters.use_state_discrimination = True
alice_parity.run()


Getting calibration path from config
C:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-15 00:31:39,771 - qm - INFO     - Performing health check
2026-07-15 00:31:39,779 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-15 00:31:40,029 - qm - INFO     - Opened quantum machine with id: QM-13711c78-eada-44ca-9267-d13ac8139c26
2026-07-15 00:31:40,029 - qm - INFO     - Opening QM
2026-07-15 00:31:40,029 - qm - INFO     - Clearing queue
2026-07-15 00:31:40,029 - qm - INFO     - Adding program to queue.
2026-07-15 00:31:49,820 - qm - INFO     - Program added to queue. Job id: 2eb74394-bf52-4c06-b942-ab1cfae40a57
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 99.81s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 99.86s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 99.90s
Progress: [##########

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\28_parity_time_measurement.py:290: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='28_parity_time_measurement', description='\n        PARITY-TIME CALIBRATION — WIGNER TOMOGRAPHY (30)\n\nExperimentally calibrates the dispersive Ramsey wait time t_parity required\nfor Wigner tomography.  t_parity is the duration for which the qubit\naccumulates phase n*pi when the cavity contains n photons:\n\n    chi_eff * t_parity = pi   ->   t_parity = 1 / (2 * f_chi)\n\nExperiment sequence\n-------------------\nFor each delay tau:\n\n  1. Reset cavity and qubit.\n  2. Prepare Fock |1> via the f0g1 sideband ladder\n     (ge pi -> ef pi -> f0g1 pi, identical to the Fock-state T1/T2 nodes).\n  3. Reset qubit frequency to the bare GE IF (n=0 photons).\n  4. Standard Ramsey:  x90 -> wait(tau) -> x90\n  5. Measure qubit state.\n\nWith 1 photon in the cavity the qubit is detuned from the bare GE frequency\nby chi_eff, so P(e) oscillates at f_chi = chi_eff / (2*pi).\nA damped-cosine fit extracts f_chi and t_parity = 1 / (2 * f_chi).\n\nPrerequisites: calibrated f0g1 s

#### Coherent T2 Ramsey

Prepares a coherent state |α> by displacement, waits variable time τ with an artificial
detuning frame rotation, reverses the displacement, then applies a qubit  π pulse and measures.
P(|e>) vs Ï" follows a decaying sinusoid -> T2ramsey.

**State update**: `cavity_mode.T2ramsey` (Alice, coherent method)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_T2 = library.nodes["27_cavity_coherent_T2"].copy(name="alice_cavity_coherent_T2")

# --- Qubit / target ---
alice_T2.parameters.qubits = ["q1"]
alice_T2.parameters.multiplexed = False
alice_T2.parameters.reset_type = "thermal"          # "thermal" | "active" | "active_gef"

# --- Cavity mode ---
alice_T2.parameters.mode_name = "alice"
alice_T2.parameters.cavity_reset_type = "thermal"   # "thermal" | "active_sideband"
alice_T2.parameters.cavity_active_cooling_fock_n = 1
alice_T2.parameters.sideband_pulse_duration_ns = None   # None = use calibrated pulse length

# --- Experiment ---
alice_T2.parameters.num_shots = 100
alice_T2.parameters.displacement_alpha = 2.0
alice_T2.parameters.ramsey_detuning_hz = 2.5e3
alice_T2.parameters.qubit_probe_operation = "selective_x180"  # "x180" | "selective_x180"
alice_T2.parameters.use_state_discrimination = True
alice_T2.parameters.use_confusion_matrix_correction = False

# --- Time sweep ---
alice_T2.parameters.min_wait_time_in_ns = 16
alice_T2.parameters.max_wait_time_in_ns = 400_000
alice_T2.parameters.wait_time_num_points = 100
alice_T2.parameters.log_or_linear_sweep = "linear"  # "log" | "linear"

# --- Runtime ---
alice_T2.parameters.simulate = False
alice_T2.parameters.timeout = 120
alice_T2.parameters.load_data_id = None

alice_T2.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-14 23:33:32,594 - qm - INFO     - Performing health check
2026-07-14 23:33:32,594 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-14 23:33:32,874 - qm - INFO     - Opened quantum machine with id: QM-181e3ced-ce94-4c7d-867d-f2d1e4d0b20d
2026-07-14 23:33:32,874 - qm - INFO     - Opening QM
2026-07-14 23:33:32,874 - qm - INFO     - Clearing queue
2026-07-14 23:33:32,884 - qm - INFO     - Adding program to queue.
2026-07-14 23:33:39,476 - qm - INFO     - Program added to queue. Job id: 2993207e-4ce2-4e75-9faa-6e98c3853890
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 215.88s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 215.94s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 215.99s
Progress: [#######

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\27_cavity_coherent_T2.py:292: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_cavity_coherent_T2', description="\n        CAVITY MODE T2 RAMSEY (COHERENT STATE)\nMeasures the T2 Ramsey coherence time of a cavity mode (e.g. alice or bob) by\ncreating a coherent state and performing a displacement-based Ramsey experiment.\n\nThe artificial detuning is encoded directly in the cavity drive frequency: both\nthe forward and reverse displacements are played at (cavity_IF + ramsey_detuning_hz).\nDuring the wait τ the cavity evolves at its natural frequency, so the reverse\ndisplacement sees a phase offset of 2π × detuning × τ.  This produces Ramsey fringes\nwithout an explicit frame rotation.\n\nSequence:\n  1. Reset cavity and qubit\n  2. Shift cavity drive to (cavity_IF + ramsey_detuning_hz)\n  3. Displace cavity → |α⟩\n  4. Wait variable time tau\n  5. Reverse displace (same amplitude, opposite sign)\n  6. Reset cavity drive frequency\n  7. π pulse on qubit  (non-selective x180)\n  8. Measure qubit state\n\nP(|e⟩) vs tau follows a decaying 

#### Fock |1> T1 (displacement + SNAP)

Prepares the cavity Fock |1> state via the D-SNAP₀-D protocol:
`D(α₁) -> selective_x180 × 2 (SNAP₀) -> D(α₂)`, waits variable time τ,
then reads out with PNRS: a `selective_x180` at the n=1 dressed qubit
frequency (`qubit_IF − 2χ`) maps P(n=1) -> P(|e>).

**Prerequisites**: displacement calibration (node 22/32), χ calibrated (node 25), `selective_x180` tuned (node 33).

**State update**: `cavity_mode.T1` (Alice, Fock1 method)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_fock1_T1 = library.nodes["33_cavity_fock1_T1"].copy(name="alice_fock1_T1")
parameters = alice_fock1_T1.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "alice"
parameters.fock1_alpha1 = 1.0        # first displacement amplitude [photons]
parameters.fock1_alpha2 = -0.59      # correction displacement amplitude [photons]
parameters.use_state_discrimination = True
parameters.num_shots = 500
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 500_000   # set to ~3–5× expected T1
parameters.wait_time_num_points = 51
parameters.log_or_linear = "logarithmic"
alice_fock1_T1.run()

In [4]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_fock1_T1 = library.nodes["33_cavity_fock1_T1"].copy(name="alice_fock1_T1")
parameters = alice_fock1_T1.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "alice"
parameters.fock1_prep_method = "sideband"
parameters.use_state_discrimination = True
parameters.num_shots = 100
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 50_000_000   # set to ~3–5× expected T1
parameters.wait_time_num_points = 51
parameters.log_or_linear_sweep = "linear"
alice_fock1_T1.run()


Getting calibration path from config
C:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-15 02:28:29,920 - qm - INFO     - Performing health check
2026-07-15 02:28:29,930 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-15 02:28:30,170 - qm - INFO     - Opened quantum machine with id: QM-8781f6ce-43a1-435c-a063-44162d297685
2026-07-15 02:28:30,170 - qm - INFO     - Opening QM
2026-07-15 02:28:30,170 - qm - INFO     - Clearing queue
2026-07-15 02:28:30,180 - qm - INFO     - Adding program to queue.
2026-07-15 02:28:34,601 - qm - INFO     - Program added to queue. Job id: e24cda7f-bf88-48be-8181-f211b3cce62d
2026-07-15 02:31:03,881 - qm - INFO     - Closing QM####     ] 90.0% (n=90/100) --> elapsed time: 147.23s
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\33_cavity_fock1_T1.py:336: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_fock1_T1', description="\n        CAVITY FOCK |1> T1\nMeasures the photon lifetime T1 of the cavity Fock |1> state.\n\nTwo preparation / readout methods are available via fock1_prep_method:\n\n  'sideband' (default):\n    Prep:    ge pi -> ef pi -> f0g1 pi  ->  |g,1>\n    Readout: inverse sideband (f0g1 pi -> ef pi -> measure ge)\n    Requires: calibrated f0g1 sideband (nodes 26, 26b) and EF_x180.\n\n  'snap_displacement':\n    Prep:    D(alpha1) -> selective_x180 × 2 (SNAP₀) -> D(alpha2)  ->  ~|1>\n    Readout: selective_x180 at n=1 dressed qubit frequency (PNRS)\n             maps P(n=1) -> P(|e>).\n    Requires: displacement calibration (node 22/30), chi calibrated (node 25),\n              selective_x180 tuned (node 33).\n\nSequence (per wait time tau):\n  1. Reset cavity and qubit.\n  2. Prepare Fock |1> (method selected by fock1_prep_method).\n  3. Wait variable time tau.\n  4. Readout (method-matched).\n  5. Measure qubit state.\n\nP(|e>) vs tau follow

#### Fock |1> T2 Ramsey (chi-dispersive)

After Fock |1> preparation (D-SNAP-D), performs a chi-dispersive Ramsey on the qubit
at the n=1 dressed frequency (`qubit_IF -> 2χ`):
`X90 -> wait τ -> frame_rotation(detuning x τ) -> X90 -> measure`.

The Ramsey decay envelope gives T2 of the qubit coupled to the Fock |1> state
(limited by cavity T1 and pure cavity dephasing).

**Prerequisites**: same as Fock1 T1, plus `x90` calibrated on the qubit.

**State update**: `cavity_mode.T2ramsey` (Alice, Fock1 method)

In [7]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_fock1_T2 = library.nodes["34_cavity_fock1_T2"].copy(name="alice_fock1_T2")
parameters = alice_fock1_T2.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "alice"
parameters.fock1_prep_method = "sideband"      # 'sideband' or 'snap_displacement' (scaffold)
parameters.ramsey_detuning_hz = 10_000.0          # artificial detuning for Ramsey fringes [Hz]
parameters.use_state_discrimination = True
parameters.num_shots = 100
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 20_000        # set to ~3-5x expected T2
parameters.wait_time_num_points = 101
parameters.log_or_linear_sweep = "linear"
alice_fock1_T2.run()


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-15 12:23:08,907 - qm - INFO     - Performing health check
2026-07-15 12:23:08,909 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-15 12:23:09,119 - qm - INFO     - Opened quantum machine with id: QM-ce96e3ea-7709-4337-ab00-0f74ce84121b
2026-07-15 12:23:09,119 - qm - INFO     - Opening QM
2026-07-15 12:23:09,119 - qm - INFO     - Clearing queue
2026-07-15 12:23:09,128 - qm - INFO     - Adding program to queue.
2026-07-15 12:23:13,949 - qm - INFO     - Program added to queue. Job id: bb960fa8-8f6f-4264-a80c-3d21b5766165
2026-07-15 12:25:14,209 - qm - INFO     - Closing QM         ] 48.0% (n=48/100) --> elapsed time: 117.58s
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\34_cavity_fock1_T2.py:341: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_fock1_T2', description="\n        CAVITY FOCK |1> T2 RAMSEY\nMeasures the T2 Ramsey coherence time using one of two protocols selected by\nfock1_prep_method.\n\n  'sideband' (default) — cavity Fock superposition T2:\n    Sequence:\n      1. Reset cavity and qubit.\n      2. Create (|0>+|1>)/sqrt(2) superposition:\n           ge pi/2  ->  ef pi  ->  f0g1 pi\n      3. Sideband Ramsey:\n           wait tau, frame rotation (ramsey_detuning_hz), f0g1 pi (close)\n      4. Back-conversion: ef pi  ->  ge pi/2\n      5. Measure qubit.\n    Requires: calibrated f0g1 sideband (nodes 26, 26b), EF_x180, x90.\n    Extracts: T2ramsey of the cavity (|0>+|1>)/sqrt(2) state.\n\n  'snap_displacement' — cavity Fock superposition T2 via SNAP+displacement:\n    Sequence:\n      1. Reset cavity and qubit.\n      2. Create (|0>+|1>)/sqrt(2) in the cavity using a SNAP+displacement\n         sequence (exact pulses TBD — scaffold in place).\n      3. Cavity Ramsey:\n           wait tau

#### Active Reset Test (Alice)

Prepares Fock |1⟩ via the sideband ladder (ge π → ef π → f0g1 π), drives the
sideband reset for a variable flat-top duration, then reads out the remaining
cavity population via the inverse sideband (f0g1 π → ef π → measure).

P(0) = 1 - P(|e⟩) vs reset duration should rise from ≈0 to ≈1.
The duration at which P(0) first exceeds 0.95 (t95) is reported.

**Prerequisites**: calibrated f0g1 sideband (node 26/26b) and EF_x180.

**No state update** (characterisation node).

In [10]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_reset_test = library.nodes["35_cavity_reset_test"].copy(name="alice_reset_test")
parameters = alice_reset_test.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "alice"
parameters.num_shots = 100
parameters.reset_duration_start_in_ns = 1_000          # start of flat-top sweep [ns]
parameters.reset_duration_end_in_ns   = 1_000_000    # 5 ms; set to ~5x expected reset time
parameters.reset_duration_step_in_ns  = 10_000       # 10 us step
parameters.use_state_discrimination = True
parameters.use_confusion_matrix_correction = False
parameters.cavity_pre_reset_type = "thermal"         # pre-shot reset: 'thermal' or 'active_sideband'
parameters.cavity_active_cooling_fock_n = 1          # starting Fock level for active pre-reset
parameters.sideband_pulse_duration_ns = None         # None -> use calibrated pi_flat_top_length_ns
alice_reset_test.run()

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-15 12:34:00,799 - qm - INFO     - Performing health check
2026-07-15 12:34:00,809 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-15 12:34:01,020 - qm - INFO     - Opened quantum machine with id: QM-ab12359b-d10b-418b-8d05-4a5423da321d
2026-07-15 12:34:01,020 - qm - INFO     - Opening QM
2026-07-15 12:34:01,020 - qm - INFO     - Clearing queue
2026-07-15 12:34:01,030 - qm - INFO     - Adding program to queue.
2026-07-15 12:34:05,532 - qm - INFO     - Program added to queue. Job id: 36e813e6-945e-4b55-971a-0f4516a795e9
2026-07-15 12:35:54,640 - qm - INFO     - Closing QM         ] 44.0% (n=44/100) --> elapsed time: 106.43s
Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action save_results


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\35_cavity_reset_test.py:309: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_reset_test', description="\n        CAVITY ACTIVE RESET TEST\nCharacterises the effectiveness of the sideband active reset by measuring\nP(0) = 1 − P(|e⟩) as a function of the reset drive flat-top duration.\n\nProtocol (per reset-duration point):\n  1. Pre-reset cavity and wait qubit thermalization.\n  2. Prepare Fock |1⟩ via sideband ladder:\n       ge pi → ef pi → f0g1 sideband pi\n  3. Drive the sideband reset for a variable flat-top duration t.\n  4. Inverse sideband readout:\n       f0g1 sideband pi → ef pi → measure qubit ge\n       P(|e⟩) = photon survived; P(|g⟩) = cavity reset to |0⟩\n  5. Measure and save qubit state.\n\nP(0) = 1 − P(|e⟩) vs t should rise from ≈0 to ≈1.  The duration at which\nP(0) first exceeds 0.95 (t95) is reported.\n\nPrerequisites:\n  - Calibrated f0g1 sideband operations on the CavityTransmonPair\n    sideband_drive (nodes 26, 26b).\n  - Calibrated EF_x180 pulse.\n\nParameters:\n  - mode_name:                    'alice' or 'bo

#### 2D Wigner Tomography (36)

Measures W(β) of a prepared cavity Fock state |n⟩ using optimised probe displacements
(minimise condition number κ of the tomography matrix; ~10–30× fewer points than a grid).

Two polarity programs (±1) are run and subtracted to cancel qubit-relaxation background:
`parity[k] = P_e(pol=-1)[k] - P_e(pol=+1)[k]` → reconstruct ρ via lstsq + PSD projection.

**Prep methods:** `sideband` (nodes 26–26h) or `snap_displacement` (set `snap_displacement_photons`).
**Cache:** set `probe_displacements_path` from the path in `node.results` after first run.

In [4]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_wigner = library.nodes["36_wigner_tomography_2d"].copy(name="alice_wigner_2d")
p = alice_wigner.parameters

# ── Qubit / mode ──────────────────────────────────────────────────────────
p.qubits    = ["q1"]
p.mode_name = "alice"

# ── Averaging ─────────────────────────────────────────────────────────────
p.num_shots = 100

# ── Fock-state target ─────────────────────────────────────────────────────
p.target_fock_level = 1      # 0 = vacuum, 1 = |1>, 2 = |2>, ..

# ── State preparation ─────────────────────────────────────────────────────
p.fock_prep_method = "sideband"   # "sideband" or "snap_displacement"

# snap_displacement only: list of (target_fock_level + 1) amplitudes [photons]
# for the D-SNAP sequence: D(a0) -> SNAP|0> -> D(a1) -> ... -> D(an)
# Fock |1>: [1.0, -0.59]   Fock |2>: [0.49686, -1.13306, 0.43235]
# p.snap_displacement_photons = [1.0, -0.59]

# ── Reconstruction ────────────────────────────────────────────────────────
p.n_fock_cutoff  = 4     # density matrix size (N_ph x N_ph)
p.n_probe_points = None  # None -> N_ph^2 + 30

# ── Probe displacement cache ──────────────────────────────────────────────
# First run: leave None.  Then copy path from node.results['probe_displacements_path']
p.probe_displacements_path = None
# p.probe_displacements_path = r"D:\MData\DR3-Run011\...\probe_displacements_N4.npz"

# ── Parity timing ─────────────────────────────────────────────────────────
p.parity_time_ns = None  # None -> derived from chi in QuAM (node 28)

# ── Plot settings ─────────────────────────────────────────────────────────
p.wigner_range = 0.0   # 0 = auto; or set e.g. 3.0
p.n_grid       = 101

# ── Readout ───────────────────────────────────────────────────────────────
p.use_state_discrimination        = True
p.use_confusion_matrix_correction = False
p.cavity_reset_type               = "thermal"   # or "active_sideband"

alice_wigner.run()

Getting calibration path from config
C:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action compute_probe_displacements
Action compute_probe_displacements finished
Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_programs
2026-07-15 14:50:25,582 - qm - INFO     - Performing health check
2026-07-15 14:50:25,587 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-15 14:50:25,807 - qm - INFO     - Opened quantum machine with id: QM-c01a1002-4f02-448d-852a-8b2a2da3cb29
2026-07-15 14:50:25,808 - qm - INFO     - Opening QM
2026-07-15 14:50:25,810 - qm - INFO     - Clearing queue
2026-07-15 14:50:25,819 - qm - INFO     - Adding program to queue.
2026-07-15 14:50:30,206 - qm - INFO     - Program added to queue. Job id: dbd8aec1-391c-47c7-a61a-41fee3795bbe
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 113.08s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 113.12s
Progress: [###########################

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\36_wigner_tomography_2d.py:457: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


NodeRunSummary(name='alice_wigner_2d', description="\n        2D WIGNER TOMOGRAPHY (36)\n\nMeasures the Wigner function W(β) of a prepared cavity Fock state using\noptimised probe displacements that minimise the tomography matrix condition\nnumber κ, reducing the number of measurements by ~10–30× vs. a square grid\nwhile improving reconstruction accuracy.\n\nSequence (repeated for polarity = +1 and −1):\n  For each probe point β_k (N_probe points):\n    1. Thermalize cavity (T1 × factor) and reset qubit.\n    2. Prepare Fock |n⟩ in the cavity:\n         'sideband'         — ladder of f{j}g{j+1} sideband π-pulses\n         'snap_displacement'— sequential Displacement–SNAP protocol\n    3. Probe displacement D(−β_k) applied to cavity.\n    4. Parity Ramsey (strict timing):\n         x90 → wait(t_parity) → ±x90   (polarity determines sign)\n    5. Qubit readout (state discrimination).\n\nAnalysis:\n  parity[k] = P_e(polarity=−1)[k] − P_e(polarity=+1)[k]\n  ρ = reconstruct_state(probe_disp

### 9. Bob cavity calibration

#### Cavity spectroscopy

**State update**: `cavity_mode.cavity_mode_drive.RF_frequency` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["21_cavity_mode_spectroscopy"].copy(name="cavity_mode_spectroscopy_bob")
node.parameters.mode_name = "bob"
node.parameters.frequency_span_in_mhz = 400.0
node.parameters.frequency_step_in_mhz = 1
node.parameters.cavity_thermalization_time_ns = 20_000_000  # 20 ms = 1x T1; increase if spectrum is noisy
node.parameters.use_gaussian_fit              = False
node.run()

#### Displacement calibration

Sweeps the displacement amplitude and measures vacuum-state population using a
selective pi-pulse.  Fits P_e(a) = A * exp(-|alpha/A_disp|^2)exp(-|a/A_disp|^2) to extract the unit
displacement amplitude Aph (amplitude_scale=1  1 photon).

**State update**: `cavity_mode.cavity_mode_drive.operations["displacement"].amplitude` (Bob)


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["22_displacement_calibration_vacuum"].copy(name="bob_disp_calib")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.amp_min = 0.0
parameters.amp_max = 2.0
parameters.amp_points = 51
parameters.active_reset = True
parameters.num_shots = 1000
parameters.use_state_discrimination = False
parameters.subtract_baseline = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


#### Coherent T1

**State update**: `cavity_mode.T1` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["23_cavity_coherent_T1"].copy(name="bob_coherent_T1")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.displacement_scale = 1.9
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 5_000_000
parameters.wait_time_num_points = 51
parameters.log_or_linear_sweep = "log"      # "log" or "linear"
parameters.delay_repeats = 1
parameters.num_shots = 1000
parameters.cavity_reset_type = "thermal"  # "thermal" or "active_sideband"
parameters.cavity_active_cooling_fock_n = 1
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


#### Photon number resolved spectroscopy

Use `displacement_alpha` to scale the coherent state amplitude.

**State update**: `cavity_mode.chi` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["24_photon_number_resolved_spectroscopy"].copy(name="bob_pns")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.displacement_scale = 1.5
parameters.displacement_alpha = 1.0
parameters.active_reset = True
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.right_offset_mhz = 2.0
parameters.left_span_mhz = 6.0
parameters.frequency_step_in_mhz = 0.04
parameters.max_peaks = 8
parameters.chi2_threshold = 2.0
parameters.num_shots = 100
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


#### Parity time calibration  (Bob)

Calibrates the dispersive Ramsey wait time tau_parity required for Wigner tomography:

    tau = pi / (2 * chi)  # tau_parity_eff = tau_parity = pi/(2*chi) = 1 / (2 * chi) = pi / (2 * delta_f)

A short displacement (~1 photon) is applied, followed by a Ramsey sequence
`y90  wait(…)  y90`. P(e) oscillates at f_chi; the fit extracts tau_parity.

**State update**: `cavity_transmon_pairs["q1_bob"].parity_time`


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_parity = library.nodes["28_parity_time_measurement"]
bob_parity.parameters.qubits = ["q1"]
bob_parity.parameters.mode_name = "bob"
bob_parity.parameters.num_shots = 1000
bob_parity.parameters.min_delay_ns = 16
bob_parity.parameters.max_delay_ns = 4000
bob_parity.parameters.delay_step_ns = 16
bob_parity.parameters.displacement_scale = 0.5
bob_parity.parameters.cavity_reset_type = "thermal"   # or "active_sideband"
bob_parity.parameters.use_state_discrimination = True
bob_parity.run()


#### Coherent T2 Ramsey

Prepares a coherent state |α> by displacement, waits variable time τ with an artificial
detuning frame rotation, reverses the displacement, then applies a qubit  π pulse and measures.
P(|e>) vs Ï" follows a decaying sinusoid -> T2ramsey.

**State update**: `cavity_mode.T2ramsey` (Bob, coherent method)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_T2 = library.nodes["27_cavity_coherent_T2"].copy(name="bob_cavity_coherent_T2")
bob_T2.parameters.qubits = ["q1"]
bob_T2.parameters.mode_name = "bob"
bob_T2.parameters.num_shots = 200
bob_T2.parameters.displacement_alpha = 1.0
bob_T2.parameters.ramsey_detuning_hz = 1000.0
# bob_T2.parameters.max_wait_time_in_ns = 100_000
bob_T2.run()

#### Fock |1> T1 (displacement + SNAP)

Prepares the cavity Fock |1> state via the D-SNAPâ'€-D protocol:
`D(α₁) -> selective_x180 × 2 (SNAP₀) -> D(α₂)`, waits variable time Ï",
then reads out with PNRS: a `selective_x180` at the n=1 dressed qubit
frequency (`qubit_IF - 2χ) maps P(n=1) -> P(|e>).

**Prerequisites**: displacement calibration (node 22/32), χ calibrated (node 25), `selective_x180` tuned (node 33).

**State update**: `cavity_mode.T1` (Bob, Fock1 method)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_fock1_T1 = library.nodes["33_cavity_fock1_T1"].copy(name="bob_fock1_T1")
parameters = bob_fock1_T1.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "bob"
parameters.fock1_alpha1 = 1.0        # first displacement amplitude [photons]
parameters.fock1_alpha2 = -0.59      # correction displacement amplitude [photons]
parameters.use_state_discrimination = True
parameters.num_shots = 500
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 500_000   # set to ~3–5× expected T1
parameters.wait_time_num_points = 51
parameters.log_or_linear = "logarithmic"
bob_fock1_T1.run()

#### Fock |1> T2 Ramsey (chi-dispersive)

After Fock |1> preparation (D-SNAPâ'€-D), performs a chi-dispersive Ramsey on the qubit
at the n=1 dressed frequency (`qubit_IF - 2χ`):
`X90 -> wait τ -> frame_rotation(detuning × τ) -> X90 -> measure`.

The Ramsey decay envelope gives T2 of the qubit coupled to the Fock |1> state
(limited by cavity T1 and pure cavity dephasing).

**Prerequisites**: same as Fock1 T1, plus `x90` calibrated on the qubit.

**State update**: `cavity_mode.T2ramsey` (Bob, Fock1 method)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_fock1_T2 = library.nodes["34_cavity_fock1_T2"].copy(name="bob_fock1_T2")
parameters = bob_fock1_T2.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "bob"
parameters.fock1_prep_method = "sideband"      # 'sideband' or 'snap_displacement' (scaffold)
parameters.ramsey_detuning_hz = 1000.0          # artificial detuning for Ramsey fringes [Hz]
parameters.use_state_discrimination = True
parameters.num_shots = 500
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 100_000        # set to ~3-5x expected T2
parameters.wait_time_num_points = 51
parameters.log_or_linear_sweep = "linear"
bob_fock1_T2.run()


#### Active Reset Test (Bob)

Prepares Fock |1⟩ via the sideband ladder (ge π → ef π → f0g1 π), drives the
sideband reset for a variable flat-top duration, then reads out the remaining
cavity population via the inverse sideband (f0g1 π → ef π → measure).

P(0) = 1 - P(|e⟩) vs reset duration should rise from ≈0 to ≈1.
The duration at which P(0) first exceeds 0.95 (t95) is reported.

**Prerequisites**: calibrated f0g1 sideband (node 26/26b) and EF_x180.

**No state update** (characterisation node).

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_reset_test = library.nodes["35_cavity_reset_test"].copy(name="bob_reset_test")
parameters = bob_reset_test.parameters
parameters.qubits = ["q1"]
parameters.mode_name = "bob"
parameters.num_shots = 500
parameters.reset_duration_start_in_ns = 100          # start of flat-top sweep [ns]
parameters.reset_duration_end_in_ns   = 5_000_000    # 5 ms; set to ~5x expected reset time
parameters.reset_duration_step_in_ns  = 10_000       # 10 us step
parameters.use_state_discrimination = True
parameters.use_confusion_matrix_correction = False
parameters.cavity_pre_reset_type = "thermal"         # pre-shot reset: 'thermal' or 'active_sideband'
parameters.cavity_active_cooling_fock_n = 1          # starting Fock level for active pre-reset
parameters.sideband_pulse_duration_ns = None         # None -> use calibrated pi_flat_top_length_ns
bob_reset_test.run()

#### 2D Wigner Tomography (36)

Measures W(β) of a prepared cavity Fock state |n⟩ using optimised probe displacements
(minimise condition number κ of the tomography matrix; ~10–30× fewer points than a grid).

Two polarity programs (±1) are run and subtracted to cancel qubit-relaxation background:
`parity[k] = P_e(pol=-1)[k] - P_e(pol=+1)[k]` → reconstruct ρ via lstsq + PSD projection.

**Prep methods:** `sideband` (nodes 26–26h) or `snap_displacement` (set `snap_displacement_photons`).
**Cache:** set `probe_displacements_path` from the path in `node.results` after first run.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_wigner = library.nodes["36_wigner_tomography_2d"].copy(name="bob_wigner_2d")
p = bob_wigner.parameters

# ── Qubit / mode ──────────────────────────────────────────────────────────
p.qubits    = ["q1"]
p.mode_name = "bob"

# ── Averaging ─────────────────────────────────────────────────────────────
p.num_shots = 400

# ── Fock-state target ─────────────────────────────────────────────────────
p.target_fock_level = 0      # 0 = vacuum, 1 = |1>, 2 = |2>, ..

# ── State preparation ─────────────────────────────────────────────────────
p.fock_prep_method = "sideband"   # "sideband" or "snap_displacement"

# snap_displacement only: list of (target_fock_level + 1) amplitudes [photons]
# for the D-SNAP sequence: D(a0) -> SNAP|0> -> D(a1) -> ... -> D(an)
# Fock |1>: [1.0, -0.59]   Fock |2>: [0.49686, -1.13306, 0.43235]
# p.snap_displacement_photons = [1.0, -0.59]

# ── Reconstruction ────────────────────────────────────────────────────────
p.n_fock_cutoff  = 4     # density matrix size (N_ph x N_ph)
p.n_probe_points = None  # None -> N_ph^2 + 30

# ── Probe displacement cache ──────────────────────────────────────────────
# First run: leave None.  Then copy path from node.results['probe_displacements_path']
p.probe_displacements_path = None
# p.probe_displacements_path = r"D:\MData\DR3-Run011\...\probe_displacements_N4.npz"

# ── Parity timing ─────────────────────────────────────────────────────────
p.parity_time_ns = None  # None -> derived from chi in QuAM (node 28)

# ── Plot settings ─────────────────────────────────────────────────────────
p.wigner_range = 0.0   # 0 = auto; or set e.g. 3.0
p.n_grid       = 101

# ── Readout ───────────────────────────────────────────────────────────────
p.use_state_discrimination        = True
p.use_confusion_matrix_correction = False
p.cavity_reset_type               = "thermal"   # or "active_sideband"

bob_wigner.run()

## 10. Automated calibration graphs

### 10a. TWPA bring-up graph

Automated TWPA calibration:
```
twpa_pump_power_sweep
   twpa_pump_frequency_sweep
      twpa_signal_saturation_power_sweep
```

Graph name in library: **`twpa_bringup_graph`**

> Run this **before** the GE bring-up graph (10b) -- TWPA gain must be set
> before optimizing readout power.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_twpa = library.graphs["twpa_bringup_graph"]
p = g_twpa.parameters

p.qubits = ["q1"]
p.twpa_id = "twpa1"

# (parameters) -- uncomment to override defaults
# p.signal_power_dbm = None  # None -> keep currently configured readout amplitude
# p.power_sweep_pump_power_min_dbm = -25.0
# p.power_sweep_pump_power_max_dbm = -10.0
# p.freq_sweep_pump_frequency_span_mhz = 200.0
# p.saturation_signal_power_min_dbm = -20.0
# p.saturation_signal_power_max_dbm = 0.0

g_twpa.run()

### 10a2. Resonator bring-up graph (02f)

Standalone resonator bring-up (also runs as a subgraph inside the GE bring-up graph):
```
resonator_discovery [loop: retry on no dip]:
  broad_resonator_spectroscopy
  -> resonator_spectroscopy_high_power
-> resonator_punch_out  [loop: retry on failure]
-> resonator_spectroscopy_low_power
```

Graph name in library: **`resonator_bringup_graph`**

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_res = library.graphs["resonator_bringup_graph"]
p = g_res.parameters

p.qubits = ["q1"]

# (parameters) -- uncomment to override defaults
# p.multiplexed                        = False
# p.use_adaptive_span                  = True
# p.max_resonator_discovery_iterations = 5
# p.max_punch_out_iterations           = 5

g_res.run(qubits=p.qubits)

### 10b. GE bring-up graph (92)

Full automated GE (ground-excited) bring-up using the trial-and-error FSM:
```
mixer_calibration
   resonator_bringup    (broad spec  high-power  punch-out  low-power)
   qubit_calibration    (spec_vs_power  qubit_spec  power_rabi)  [outer loop]
   x180_fine_calibration (Ramsey  power_rabi loop)
   T1
   readout_frequency_optimization
   readout_length_optimization
   readout_power_optimization
```

Graph name in library: **`transmon_bringup_adaptive`**

> EF-transition and cavity-mode bring-up have been split into their own graphs:
> **`ef_bringup_graph`** (93) and **`cavity_bringup_graph`** (94). Run them after this graph finishes.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g92 = library.graphs["transmon_bringup_adaptive"]
p = g92.parameters

# GE (ground-excited) bring-up only. Run 93_ef_bringup_graph / 94_cavity_bringup_graph afterwards.
p.qubits = ["q1"]
p.multiplexed = False

# (parameters) -- uncomment to override defaults
# --- Iteration limits ---
# p.max_resonator_discovery_iterations = 5
# p.max_punch_out_iterations           = 5
# p.max_spec_vs_power_iterations       = 5
# p.max_qubit_calibration_iterations   = 3
# p.x180_max_iterations                = 10
# p.x180_rabi_max_amplitude_iterations = 5
# --- Mixer calibration ---
# p.mixer_calibrate_resonator       = True
# p.mixer_calibrate_drive           = True
# p.mixer_calibrate_cavity_drive    = True
# p.mixer_calibrate_sideband_drive  = True
# --- Adaptive behaviour ---
# p.use_adaptive_span               = True
# p.spec_vs_power_use_adaptive_span = True
# p.x180_rabi_use_adaptive          = True
# --- Convergence threshold ---
# p.x180_freq_threshold_hz          = 50_000.0   # [Hz] stop x180 loop when |detuning| < this

# --- Where to (re)start the graph ---
# "full"            -> run every node from the start (default)
# "<node_name>"    -> start from a specific node, e.g. "T1"
# "last_successful" -> resume right after the last node that finished
#                       successfully on the previous g92.run() in this kernel
#                       session (falls back to "full" if there is none)
restart_mode = "resonator_bringup"

start_from = None
if restart_mode == "last_successful":
    from qualibrate.core.models.node_status import ElementRunStatus

    history = g92._orchestrator.get_execution_history().items
    finished = [item.metadata.name for item in history if item.metadata.status == ElementRunStatus.finished]
    if finished:
        last_node = g92._elements[finished[-1]]
        successors = list(g92._graph.successors(last_node))
        start_from = successors[0].name if successors else None
    if start_from is None:
        print("No previous successful run found in this session -- running the full graph.")
elif restart_mode != "full":
    start_from = restart_mode

g92.run(qubits=p.qubits, start_from=start_from)


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 12:03:17,047 - qm - INFO     - Performing health check
2026-06-30 12:03:17,047 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 12:03:17,337 - qm - INFO     - Opened quantum machine with id: QM-8f8a4178-8e83-426d-826c-13a3548dfac9
2026-06-30 12:03:17,337 - qm - INFO     - Opening QM
2026-06-30 12:03:17,337 - qm - INFO     - Clearing queue
2026-06-30 12:03:17,349 - qm - INFO     - Adding program to queue.
2026-06-30 12:03:17,568 - qm - INFO     - Program added to queue. Job id: b5054751-dc35-4c8e-9043-25b54cfe8548
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 0.88s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 0.93s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 0.97s
Progress: [###################

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02d_broad_resonator_spectroscopy.py:215: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished
Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-30 12:03:19,479 - qm - INFO     - Performing health check
2026-06-30 12:03:19,490 - qm - INFO     - Cluster healthcheck completed successfully.
2026-06-30 12:03:19,790 - qm - INFO     - Opened quantum machine with id: QM-463a47c0-f27f-4eb2-8382-5ba315ff59ee
2026-06-30 12:03:19,800 - qm - INFO     - Opening QM
2026-06-30 12:03:19,800 - qm - INFO     - Clearing queue
2026-06-30 12:03:19,811 - qm - INFO     - Adding program to queue.
2026-06-30 12:03:20,039 - qm - INFO     - Program added to queue. Job id: 4f3a1196-815f-4da7-ab11-4b6144f0819e
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 0.41s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 0.46s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 0.52s
P

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:219: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:219: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:233: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'
Failed to run QualibrationGraph: qubit_calibration (mode: inspection=False interactive=False external=False; parameters: parameters=_QubitCalibrationSubgraphParameters(qubits=['q0'], spec_vs_power_use_adaptive_span=True, spec_vs_power_frequency_span_mhz=300.0, spec_vs_power_frequency_step_mhz=1.0, spec_vs_power_num_power_points=10, spec_vs_power_num_shots=100, spec_vs_power_min_power_dbm=-60.0, spec_vs_power_max_power_dbm=10.0, spec_vs_power_operation_len_ns=20000, spec_vs_power_linewidth_threshold_hz=1000000.0, spec_vs_power_max_amplitude_opx=0.24, spec_vs_power_min_amplitude_opx=0.01, spec_vs_power_rabi_target_periods=1, spec_vs_power_rabi_sweep_max_duration_ns=300.0, max_spec_vs_power_iterations=5, time_rabi_min_duration_ns=16, time_rabi_max_duration_ns=300, time_rabi_duration_step_ns=4, time_rabi_num_shots=200, time_rabi_max_amplitude_opx=0.1, time_rabi_drive_power_dbm=None) nodes=GraphElementsParame

Action save_results finished


ValidationError: 1 validation error for _QubitCalibrationSubgraphParameters
time_rabi_drive_power_dbm
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type

### 10c. EF bring-up graph (93)

Calibrates the EF (|e> -> |f>) transition:
```
ef_spectroscopy  [loop: retry on no peak]
-> ef_tentative_rabi   (oscillation check; blacklists freq on failure)
   [outer loop: retry on NO_OSCILLATION]
-> ef_rabi_ramsey  [loop: until |EF detuning| converges]:
    ef_power_rabi  [inner loop: amplitude convergence]
    -> ef_ramsey
-> ef_T1
-> gef_readout_frequency_optimization
-> gef_iq_blobs
```

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_ef = library.graphs["ef_bringup_graph"]
p = g_ef.parameters

# EF Calibration Macronode
# Sequence:
#   ef_spectroscopy [loop: retry on no peak, max_ef_spec_iterations]
#   -> ef_rabi_ramsey [loop: until |EF detuning| < ef_freq_threshold_hz, ef_max_iterations]:
#       ef_power_rabi [inner loop: period convergence, ef_rabi_max_amplitude_iterations]
#       -> ef_ramsey
#   -> ef_T1
#   -> gef_readout_frequency_optimization
#   -> gef_iq_blobs
p.qubits = ["q1"]

# (parameters) -- uncomment to override defaults
# Iteration limits
# p.max_ef_discovery_iterations      = 3   # retries on NO_OSCILLATION in tentative Rabi
# p.max_ef_spec_iterations           = 3
# p.ef_max_iterations                = 5
# p.ef_rabi_max_amplitude_iterations = 5
# Convergence threshold
# p.ef_freq_threshold_hz = 50_000.0  # stop when |EF detuning| < this [Hz]

g_ef.run(qubits=p.qubits)

2026-06-23 11:52:20,704 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_Qubit_Spectroscopy_E_to_F.py
2026-06-23 11:52:20,731 - qualibrate - INFO - Creating node 12_qubit_spectroscopy_EF
2026-06-23 11:52:20,857 - qualibrate - INFO - Loaded node 12_qubit_spectroscopy_EF from C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_Qubit_Spectroscopy_E_to_F.py
2026-06-23 11:52:20,858 - qualibrate - INFO - Creating node 12_qubit_spectroscopy_EF
2026-06-23 11:52:20,946 - qualibrate - INFO - Copying node with name 12_qubit_spectroscopy_EF with parameters name = 'ef_spectroscopy', node_parameters = {'frequency_span_in_mhz': 300.0, 'frequency_step_in_mhz': 1.0, 'operation': 'saturation', 'operation_len_in_ns': 20000, 'operation_amplitude_factor': 1.0, 'num_shots': 100, 'target_peak_width': 3000000.0, 'update_pulses_amplitude': False, 'find_dip': False,

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-06-23 11:52:24,284 - qm - INFO     - Performing health check
2026-06-23 11:52:24,796 - qm - INFO     - Health check passed
2026-06-23 11:52:28,452 - qm - INFO     - Opening QM
2026-06-23 11:52:28,452 - qm - INFO     - Sending program to QOP for compilation
2026-06-23 11:52:28,656 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 76.34s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 76.38s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 76.43s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 76.48s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 76.52s
Progress: [############################

2026-06-23 11:53:46,889 - qualibrate - INFO - Node ef_spectroscopy - Execution report for job 1780001966209
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 77.06s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 77.10s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 77.17s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 77.20s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 77.23s
2026-06-23 11:53:46,889 - qm - INFO     - Closing QM
Action execute_qua_program finished
Running action analyse_data


c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibration_libs\analysis\feature_detection.py:109: FutureWarning: Reductions are applied along the rolling dimension(s) '['detuning']'. Passing the 'dim' kwarg to reduction operations has no effect.
  rolling = da.rolling({dim: 10}, center=True).mean(dim=dim)
2026-06-23 11:53:46,983 - qualibrate - INFO - Node ef_spectroscopy - Results for qubit q1:  SUCCESS!
	EF frequency: 3.998 GHz | FWHM: 2283.8 kHz | The integration weight angle: 3.937 rad
 To get the desired FWHM, the saturation amplitude is updated to: 21.9 mV | To get the desired EF_x180 gate, the EF_x180 amplitude is updated to: 159.3 mV
 Residual chi2: 0.075
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_Qubit_Spectroscopy_E_to_F.py:225: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-06-23 11:53:47,079 - qualibrate - INFO - Saving node ef_spectroscopy to local

...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\15_iq_blobs_gef.py:256: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-06-23 12:04:27,277 - qualibrate - INFO - Saving node gef_iq_blobs to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-06-23 12:04:27,751 - qualibrate - INFO - Saving machine state to db
2026-06-23 12:04:27,767 - qualibrate - WARNING - save failed: No database connection configured for project 'automatic_calibration'
2026-06-23 12:04:27,769 - qualibrate - INFO - Saving machine to active path D:\MData\DR6-Run007\qm_cal\automatic_calibration\state
2026-06-23 12:04:27,793 - qualibrate - INFO - Saving machine to data folder D:\MData\DR6-Run007\qm_cal\automatic_calibration\calibration_storage\2026-06-23\#502_gef_iq_blobs_120427\quam_state
2026-06-23 12:04:27,825 - qualibrate - INFO - Updating workflow snapshot 493 for ef_bringup
2026-06-23 12:04:27,834 - qualibrate - INFO - Saved target_paths to snapshot 493
2026-06-23 12:04:27,836 - qualibrate - INFO - Finalized workflow snapshot 493 for graph ef_bringup with status finished


Action save_results finished


GraphRunSummary(name='ef_bringup', description=None, created_at=datetime.datetime(2026, 6, 23, 11, 52, 23, 220799, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 6, 23, 12, 4, 27, 837363, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), parameters=ExecutionParameters(parameters=_EFCalibrationSubgraphParameters(qubits=['q1']), nodes=GraphElementsParameters(ef_discovery=ExecutionParameters(parameters=_EFDiscoverySubgraphParameters(qubits=['q1']), nodes=GraphElementsParameters(ef_spectroscopy=Parameters(multiplexed=False, use_state_discrimination=False, reset_type='thermal', qubits=None, num_shots=100, frequency_span_in_mhz=300.0, frequency_step_in_mhz=1.0, operation='saturation', operation_amplitude_factor=1.0, operation_len_in_ns=20000, target_peak_width=3000000.0, update_pulses_amplitude=False, find_dip=False, signal_source='I_rot', update_integration_wei

### 10d. Sideband bring-up graph (95)

Calibrates a single f|k>g|k+1> sideband transition selected by `sideband_level`.
Setting `sideband_level = N` calibrates the f|N-1>g|N> transition (1-based):

```
sideband_level = 1  ->  f0g1
sideband_level = 2  ->  f1g2
sideband_level = 3  ->  f2g3
```

The calibration chain (nodes named fNgN1_* in the GUI):
```
fNgN1_spectroscopy
-> fNgN1_time_rabi
-> fNgN1_ramsey            (sideband frequency fine-tuning)
-> fNgN1_ge_spectroscopy   (qubit GE shift in Fock |k>)
-> fNgN1_ge_ramsey         (precise qubit GE frequency at Fock |k>)
-> fNgN1_ef_spectroscopy   (qubit EF shift in Fock |k>)
-> fNgN1_ef_ramsey         (Kerr-corrected EF frequency)
```

For levels > 1, lower-level transitions must already be calibrated so that
the Fock-state preparation succeeds.

> **Note**: `sideband_level` is a build-time parameter — changing it requires
> reloading the library (re-running `95_sideband_bringup_graph.py`).

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_sb = library.graphs["sideband_bringup_graph"]
p = g_sb.parameters

p.qubits        = ["q1"]
p.mode_name     = "alice"
# p.sideband_level = 1   # 1 -> f0g1, 2 -> f1g2, 3 -> f2g3, ... (requires library reload)

g_sb.run(qubits=p.qubits)

### 10e. Cavity bring-up graph (94)

Calibrates a single cavity mode (alice or bob):
```
cavity_mode_spectroscopy
-> displacement_calibration   (vacuum state; 1-photon amplitude)  [loop]
-> cavity_T1                  (coherent T1)
-> cavity_T2                  (coherent T2 Ramsey)
-> parity_time_measurement    (optimal parity mapping time)
-> fock1_T1                   (Fock |1> photon lifetime)
-> fock1_T2                   (Fock |1> T2 Ramsey)
```

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_cav = library.graphs["cavity_bringup_graph"]
p = g_cav.parameters

p.qubits           = ["q1"]
p.cavity_mode_name = "alice"

# (parameters) -- uncomment to override defaults
# p.max_displacement_vacuum_iterations = 5

g_cav.run(qubits=p.qubits)

### 10f. GE retuning graph (96)

Refines GE transition frequency and x180 amplitude, then measures GE IQ blobs:


Ramsey  and  are computed
from  in the QUAM state when the library loads.

Graph name in library: ****

In [4]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_ge_retune = library.graphs["ge_retuning_graph"]
p = g_ge_retune.parameters

p.qubits = ["q1"]

# (parameters) -- uncomment to override defaults
# p.freq_threshold_hz = 50_000.0  # stop when |detuning| < this [Hz]
# p.max_iterations    = 5

g_ge_retune.run(qubits=p.qubits)

Getting calibration path from config
C:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-16 12:43:17,669 - qm - INFO     - Performing health check
2026-07-16 12:43:17,678 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-16 12:43:18,039 - qm - INFO     - Opened quantum machine with id: QM-8554ebff-a044-4dea-a8d5-f0758b8dbd37
2026-07-16 12:43:18,039 - qm - INFO     - Opening QM
2026-07-16 12:43:18,039 - qm - INFO     - Clearing queue
2026-07-16 12:43:18,049 - qm - INFO     - Adding program to queue.
2026-07-16 12:43:19,744 - qm - INFO     - Program added to queue. Job id: 3e190cb7-c0cc-45bd-9bc0-5a8b020bfebd
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 23.29s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 23.34s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 23.40s
Progress: [##########

C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06a_ramsey.py:261: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\07_iq_blobs.py:236: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


GraphRunSummary(name='ge_retuning_graph', description=None, created_at=datetime.datetime(2026, 7, 16, 12, 43, 16, 818263, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 7, 16, 12, 46, 20, 417103, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), parameters=ExecutionParameters(parameters=GERetuningParameters(qubits=['q1'], freq_threshold_hz=50000.0, max_iterations=5), nodes=GraphElementsParameters(x180_refinement=ExecutionParameters(parameters=_SubgraphParameters(qubits=['q1']), nodes=GraphElementsParameters(ramsey=Parameters(multiplexed=False, use_state_discrimination=True, reset_type='thermal', qubits=None, num_shots=200, frequency_detuning_in_mhz=0.36357, x180_operation='x180', selective_state_update=False, correct_with_pulse_detuning=False, min_wait_time_in_ns=16, max_wait_time_in_ns=27505, wait_time_num_points=100, log_or_linear_sweep='linear', simulat

### 10g. EF retuning graph (97)

Refines EF transition frequency (anharmonicity) and EF_x180 amplitude, then
measures GEF IQ blobs for three-state discrimination:


Ramsey  and  are
computed from  in the QUAM state when the library loads.

Graph name in library: ****

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

g_ef_retune = library.graphs["ef_retuning_graph"]
p = g_ef_retune.parameters

p.qubits = ["q1"]

# (parameters) -- uncomment to override defaults
# p.freq_threshold_hz = 50_000.0  # stop when |EF detuning| < this [Hz]
# p.max_iterations    = 5

g_ef_retune.run(qubits=p.qubits)

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-16 13:21:30,431 - qm - INFO     - Performing health check
2026-07-16 13:21:30,431 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-16 13:21:30,672 - qm - INFO     - Opened quantum machine with id: QM-eac335e9-fe0f-4c6b-8a60-24e1d6bfebcc
2026-07-16 13:21:30,672 - qm - INFO     - Opening QM
2026-07-16 13:21:30,682 - qm - INFO     - Clearing queue
2026-07-16 13:21:30,682 - qm - INFO     - Adding program to queue.
2026-07-16 13:21:31,211 - qm - INFO     - Program added to queue. Job id: eda285fb-89ac-47a7-a51f-f89d1c799385
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 46.47s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 46.52s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 46.57s
Progress: [##########

C:\Users\td-srv-quantum\.conda\envs\qm_cal\Lib\site-packages\qualibration_libs\analysis\fitting.py:147: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06b_ramsey_ef.py:231: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished
Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-16 13:22:19,142 - qm - INFO     - Performing health check
2026-07-16 13:22:19,147 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-16 13:22:19,382 - qm - INFO     - Opened quantum machine with id: QM-3ac6bd04-f49e-4ffd-9a52-e1a3f01bf004
2026-07-16 13:22:19,384 - qm - INFO     - Opening QM
2026-07-16 13:22:19,385 - qm - INFO     - Clearing queue
2026-07-16 13:22:19,395 - qm - INFO     - Adding program to queue.
2026-07-16 13:22:19,810 - qm - INFO     - Program added to queue. Job id: 5a6f31c2-3a93-449b-8559-e814b35928bc
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 43.21s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 43.25s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time:

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\13_power_rabi_ef.py:248: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished
Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-07-16 13:23:04,672 - qm - INFO     - Performing health check
2026-07-16 13:23:04,676 - qm - INFO     - Cluster healthcheck completed successfully.
2026-07-16 13:23:04,923 - qm - INFO     - Opened quantum machine with id: QM-f37d3ff8-ca3d-49ed-893e-b43ea7ba5120
2026-07-16 13:23:04,926 - qm - INFO     - Opening QM
2026-07-16 13:23:04,927 - qm - INFO     - Clearing queue
2026-07-16 13:23:04,935 - qm - INFO     - Adding program to queue.
2026-07-16 13:23:05,196 - qm - INFO     - Program added to queue. Job id: 51099b9d-039f-4795-960a-08dea9ef5aed
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.43s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 45.48s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time:

...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\15_iq_blobs_gef.py:258: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


save failed: No database connection configured for project 'dr3_run11_srf_qubit_2'


Action save_results finished


GraphRunSummary(name='ef_retuning_graph', description=None, created_at=datetime.datetime(2026, 7, 16, 13, 21, 29, 892806, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 7, 16, 13, 24, 1, 796645, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), parameters=ExecutionParameters(parameters=EFRetuningParameters(qubits=['q1'], freq_threshold_hz=50000.0, max_iterations=5), nodes=GraphElementsParameters(ef_x180_refinement=ExecutionParameters(parameters=_SubgraphParameters(qubits=['q1']), nodes=GraphElementsParameters(ef_ramsey=Parameters(multiplexed=False, use_state_discrimination=False, reset_type='thermal', qubits=None, num_shots=200, frequency_detuning_in_mhz=0.2, ef_x180_operation='EF_x180', selective_state_update=False, min_wait_time_in_ns=16, max_wait_time_in_ns=50000, wait_time_num_points=100, log_or_linear_sweep='linear', simulate=False, simulation_duratio